In [ ]:
# Cell 1 — Environment Setup (no cloud mutations)

import importlib
import shutil
import sys

print("=== Python Package Check ===")
required_py = ["oci", "pandas", "numpy", "ipywidgets", "matplotlib"]
for pkg in required_py:
    try:
        importlib.import_module(pkg)
        print(f"{pkg}: present")
    except Exception:
        print(f"{pkg}: missing")

print("\n=== CLI Tool Check ===")
required_cli = ["terraform", "kubectl", "oci", "helm"]
for cli in required_cli:
    path = shutil.which(cli)
    if path:
        print(f"{cli}: found at {path}")
    else:
        print(f"{cli}: missing")

print("\nPython:", sys.version)
print("Cell 1 complete: Environment setup check finished.")
print("Next: Run Cell 2.")

In [ ]:
# Cell 2 — SSOT Defaults + Local Credential Context (no network calls)
import os
import json
from pathlib import Path
from datetime import datetime, timezone

import oci


def _as_bool(value, default=False):
    if value is None:
        return default
    return str(value).strip().lower() in {"1", "true", "yes", "y", "on"}


def _as_int(value, default):
    try:
        return int(value)
    except Exception:
        return default


def _as_float(value, default):
    try:
        return float(value)
    except Exception:
        return default


def _setenv(k, v):
    os.environ[k] = "" if v is None else str(v)


OCI_CONFIG_FILE = os.path.expanduser(os.environ.get("OCI_CONFIG_FILE", "~/.oci/config"))
OCI_PROFILE = os.environ.get("OCI_PROFILE", "DEFAULT")

_cfg = oci.config.from_file(file_location=OCI_CONFIG_FILE, profile_name=OCI_PROFILE)

TENANCY_OCID = os.environ.get("TENANCY_OCID", _cfg.get("tenancy", ""))
USER_OCID = os.environ.get("USER_OCID", _cfg.get("user", ""))
FINGERPRINT = os.environ.get("FINGERPRINT", _cfg.get("fingerprint", ""))
REGION = os.environ.get("REGION", _cfg.get("region", "us-ashburn-1"))

TARGET_CLUSTER = os.environ.get("TARGET_CLUSTER", "both").strip().lower()
TARGET_SCENARIOS = os.environ.get("TARGET_SCENARIOS", "all").strip().lower()
REPEATS = _as_int(os.environ.get("REPEATS"), 3)
IMAGE_MODE = os.environ.get("IMAGE_MODE", "pin").strip().lower()
ENABLE_METRICS_SERVER = _as_bool(os.environ.get("ENABLE_METRICS_SERVER"), False)

KUBERNETES_VERSION = os.environ.get("KUBERNETES_VERSION", "v1.34.2").strip()

RUN_ID = os.environ.get(
    "RUN_ID", f"ca-vs-kpo-{datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')}"
)

OUTPUT_DIR = os.path.expanduser(
    os.environ.get("OUTPUT_DIR", str((Path.cwd() / "artifacts").resolve()))
)
Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

HOME_REGION = os.environ.get("HOME_REGION", "")
PARENT_COMPARTMENT_OCID = os.environ.get("PARENT_COMPARTMENT_OCID", "")
IAM_POLICY_COMPARTMENT_OCID = os.environ.get("IAM_POLICY_COMPARTMENT_OCID", "")

CA_NAMESPACE = os.environ.get("CA_NAMESPACE", "kube-system")
CA_SERVICE_ACCOUNT = os.environ.get("CA_SERVICE_ACCOUNT", "cluster-autoscaler")
KPO_NAMESPACE = os.environ.get("KPO_NAMESPACE", "karpenter")
KPO_SERVICE_ACCOUNT = os.environ.get("KPO_SERVICE_ACCOUNT", "karpenter")

TELEMETRY_PRINCIPAL_GROUP_NAME = os.environ.get("TELEMETRY_PRINCIPAL_GROUP_NAME", "")

CA_WORKLOAD_POLICY_NAME = os.environ.get("CA_WORKLOAD_POLICY_NAME", "")
KPO_WORKLOAD_POLICY_NAME = os.environ.get("KPO_WORKLOAD_POLICY_NAME", "")
KPO_NODES_POLICY_NAME = os.environ.get("KPO_NODES_POLICY_NAME", "")
KPO_NODES_DYNAMIC_GROUP_NAME = os.environ.get("KPO_NODES_DYNAMIC_GROUP_NAME", "")
KPO_NODES_DYNAMIC_GROUP_RULE = os.environ.get("KPO_NODES_DYNAMIC_GROUP_RULE", "")

BENCHMARK_REGION = os.environ.get("BENCHMARK_REGION", "")
BENCHMARK_COMPARTMENT_OCID = os.environ.get("BENCHMARK_COMPARTMENT_OCID", "")

CA_BASELINE_POOL_SIZE = _as_int(os.environ.get("CA_BASELINE_POOL_SIZE"), 2)
KPO_BASELINE_POOL_SIZE = _as_int(os.environ.get("KPO_BASELINE_POOL_SIZE"), 2)

CA_AUTOSCALED_POOL_MIN_SIZE = _as_int(os.environ.get("CA_AUTOSCALED_POOL_MIN_SIZE"), 0)
CA_AUTOSCALED_POOL_MAX_SIZE = _as_int(os.environ.get("CA_AUTOSCALED_POOL_MAX_SIZE"), 10)
KPO_AUTOSCALED_POOL_MIN_SIZE = _as_int(
    os.environ.get("KPO_AUTOSCALED_POOL_MIN_SIZE"), 0
)
KPO_AUTOSCALED_POOL_MAX_SIZE = _as_int(
    os.environ.get("KPO_AUTOSCALED_POOL_MAX_SIZE"), 10
)

CA_BASELINE_POOL_SHAPE = os.environ.get("CA_BASELINE_POOL_SHAPE", "").strip()
CA_BASELINE_POOL_IMAGE_ID = os.environ.get("CA_BASELINE_POOL_IMAGE_ID", "").strip()
CA_BASELINE_POOL_FLEX_OCPUS = os.environ.get("CA_BASELINE_POOL_FLEX_OCPUS", "").strip()
CA_BASELINE_POOL_FLEX_MEMORY_GB = os.environ.get(
    "CA_BASELINE_POOL_FLEX_MEMORY_GB", ""
).strip()

CA_AUTOSCALED_POOL_SHAPE = os.environ.get("CA_AUTOSCALED_POOL_SHAPE", "").strip()
CA_AUTOSCALED_POOL_IMAGE_ID = os.environ.get("CA_AUTOSCALED_POOL_IMAGE_ID", "").strip()
CA_AUTOSCALED_POOL_FLEX_OCPUS = os.environ.get(
    "CA_AUTOSCALED_POOL_FLEX_OCPUS", ""
).strip()
CA_AUTOSCALED_POOL_FLEX_MEMORY_GB = os.environ.get(
    "CA_AUTOSCALED_POOL_FLEX_MEMORY_GB", ""
).strip()

KPO_BASELINE_POOL_SHAPE = os.environ.get("KPO_BASELINE_POOL_SHAPE", "").strip()
KPO_BASELINE_POOL_IMAGE_ID = os.environ.get("KPO_BASELINE_POOL_IMAGE_ID", "").strip()
KPO_BASELINE_POOL_FLEX_OCPUS = os.environ.get(
    "KPO_BASELINE_POOL_FLEX_OCPUS", ""
).strip()
KPO_BASELINE_POOL_FLEX_MEMORY_GB = os.environ.get(
    "KPO_BASELINE_POOL_FLEX_MEMORY_GB", ""
).strip()

KPO_AUTOSCALED_POOL_SHAPE = os.environ.get("KPO_AUTOSCALED_POOL_SHAPE", "").strip()
KPO_AUTOSCALED_POOL_IMAGE_ID = os.environ.get(
    "KPO_AUTOSCALED_POOL_IMAGE_ID", ""
).strip()
KPO_AUTOSCALED_POOL_FLEX_OCPUS = os.environ.get(
    "KPO_AUTOSCALED_POOL_FLEX_OCPUS", ""
).strip()
KPO_AUTOSCALED_POOL_FLEX_MEMORY_GB = os.environ.get(
    "KPO_AUTOSCALED_POOL_FLEX_MEMORY_GB", ""
).strip()

CONTROL_PLANE_ENDPOINT_MODE = (
    os.environ.get("CONTROL_PLANE_ENDPOINT_MODE", "public").strip().lower()
)
CONTROL_PLANE_ALLOWED_CIDRS = os.environ.get("CONTROL_PLANE_ALLOWED_CIDRS", "").strip()
KUBECONFIG_ENDPOINT_MODE = (
    os.environ.get("KUBECONFIG_ENDPOINT_MODE", "auto").strip().lower()
)
K8S_API_CHECK_TIMEOUT_SECONDS = _as_int(
    os.environ.get("K8S_API_CHECK_TIMEOUT_SECONDS"), 15
)

HEALTHCHECK_ALLOW_API_UNREACHABLE = _as_bool(
    os.environ.get(
        "HEALTHCHECK_ALLOW_API_UNREACHABLE",
    ),
    False,
)

METRICS_SERVER_SKIP_ADDON_DEPENDENCIES_CHECK = _as_bool(
    os.environ.get("METRICS_SERVER_SKIP_ADDON_DEPENDENCIES_CHECK"),
    False,
)

KPO_HELM_RELEASE = os.environ.get("KPO_HELM_RELEASE", "karpenter").strip()
KPO_HELM_CHART = os.environ.get("KPO_HELM_CHART", "").strip()
KPO_OCINODECLASS_NAME = os.environ.get(
    "KPO_OCINODECLASS_NAME", "benchmark-ocinodeclass"
).strip()
KPO_WORKLOAD_NODEPOOL = os.environ.get(
    "KPO_WORKLOAD_NODEPOOL", "benchmark-nodepool"
).strip()
KPO_NPN_IP_COUNT = _as_int(os.environ.get("KPO_NPN_IP_COUNT"), 16)
KPO_CPU_PER_OCPU_FACTOR = _as_float(os.environ.get("KPO_CPU_PER_OCPU_FACTOR"), 2.0)

KPO_IMAGE_FILTER_COMPARTMENT_ID = os.environ.get(
    "KPO_IMAGE_FILTER_COMPARTMENT_ID", ""
).strip()
KPO_IMAGE_OS_VERSION_FILTER = os.environ.get("KPO_IMAGE_OS_VERSION_FILTER", "8").strip()

SCENARIO_POLL_SECONDS = _as_int(os.environ.get("SCENARIO_POLL_SECONDS"), 10)
SCENARIO_PENDING_TIMEOUT_SECONDS = _as_int(
    os.environ.get("SCENARIO_PENDING_TIMEOUT_SECONDS"), 300
)
SCENARIO_READY_TIMEOUT_SECONDS = _as_int(
    os.environ.get("SCENARIO_READY_TIMEOUT_SECONDS"), 1800
)
SCENARIO_SCALE_DOWN_TIMEOUT_SECONDS = _as_int(
    os.environ.get("SCENARIO_SCALE_DOWN_TIMEOUT_SECONDS"), 1800
)

SCENARIO_BURST_REPLICAS = _as_int(os.environ.get("SCENARIO_BURST_REPLICAS"), 20)
SCENARIO_BURST_HOLD_SECONDS = _as_int(
    os.environ.get("SCENARIO_BURST_HOLD_SECONDS"), 120
)
SCENARIO_STEADY_REPLICAS = _as_int(os.environ.get("SCENARIO_STEADY_REPLICAS"), 8)
SCENARIO_STEADY_HOLD_SECONDS = _as_int(
    os.environ.get("SCENARIO_STEADY_HOLD_SECONDS"), 300
)

CA_EXPECTED_BENCHMARK_NODE_FLOOR = _as_int(
    os.environ.get(
        "CA_EXPECTED_BENCHMARK_NODE_FLOOR", str(CA_AUTOSCALED_POOL_MIN_SIZE)
    ),
    CA_AUTOSCALED_POOL_MIN_SIZE,
)
KPO_EXPECTED_BENCHMARK_NODE_FLOOR = _as_int(
    os.environ.get(
        "KPO_EXPECTED_BENCHMARK_NODE_FLOOR", str(KPO_AUTOSCALED_POOL_MIN_SIZE)
    ),
    KPO_AUTOSCALED_POOL_MIN_SIZE,
)
REQUIRE_KPO_NODECLAIMS_AT_FLOOR = _as_bool(
    os.environ.get("REQUIRE_KPO_NODECLAIMS_AT_FLOOR"), True
)

WORKLOAD_NAME = os.environ.get("WORKLOAD_NAME", "benchmark-load").strip()
WORKLOAD_IMAGE = os.environ.get("WORKLOAD_IMAGE", "registry.k8s.io/pause:3.9").strip()
WORKLOAD_BASE_REPLICAS = _as_int(os.environ.get("WORKLOAD_BASE_REPLICAS"), 0)

WORKLOAD_CPU_REQUEST = os.environ.get("WORKLOAD_CPU_REQUEST", "500m").strip()
WORKLOAD_MEM_REQUEST = os.environ.get("WORKLOAD_MEM_REQUEST", "512Mi").strip()
WORKLOAD_CPU_LIMIT = os.environ.get("WORKLOAD_CPU_LIMIT", WORKLOAD_CPU_REQUEST).strip()
WORKLOAD_MEM_LIMIT = os.environ.get("WORKLOAD_MEM_LIMIT", WORKLOAD_MEM_REQUEST).strip()

WORKLOAD_NAMESPACE_PREFIX = os.environ.get(
    "WORKLOAD_NAMESPACE_PREFIX", "benchmark"
).strip()
WORKLOAD_NAMESPACE_KPO = os.environ.get(
    "WORKLOAD_NAMESPACE_KPO", "benchmark-kpo"
).strip()
WORKLOAD_DEPLOYMENT_KPO = os.environ.get(
    "WORKLOAD_DEPLOYMENT_KPO", WORKLOAD_NAME
).strip()

KPO_REMEDIATION_APPLY_NSG = _as_bool(
    os.environ.get("KPO_REMEDIATION_APPLY_NSG", "false"),
    False,
)
KPO_REMEDIATION_APPLY_LIMITS = _as_bool(
    os.environ.get("KPO_REMEDIATION_APPLY_LIMITS", "false"),
    False,
)
KPO_REMEDIATION_LIMITS_CPU = os.environ.get("KPO_REMEDIATION_LIMITS_CPU", "").strip()
KPO_REMEDIATION_LIMITS_MEMORY = os.environ.get(
    "KPO_REMEDIATION_LIMITS_MEMORY", ""
).strip()
KPO_REMEDIATION_LIMITS_CPU_HEADROOM = _as_float(
    os.environ.get("KPO_REMEDIATION_LIMITS_CPU_HEADROOM", "1.0"),
    1.0,
)
KPO_REMEDIATION_LIMITS_MEMORY_HEADROOM_GI = _as_float(
    os.environ.get("KPO_REMEDIATION_LIMITS_MEMORY_HEADROOM_GI", "8.0"),
    8.0,
)

for k, v in {
    "OCI_CONFIG_FILE": OCI_CONFIG_FILE,
    "OCI_PROFILE": OCI_PROFILE,
    "TENANCY_OCID": TENANCY_OCID,
    "USER_OCID": USER_OCID,
    "FINGERPRINT": FINGERPRINT,
    "REGION": REGION,
    "TARGET_CLUSTER": TARGET_CLUSTER,
    "TARGET_SCENARIOS": TARGET_SCENARIOS,
    "REPEATS": REPEATS,
    "IMAGE_MODE": IMAGE_MODE,
    "ENABLE_METRICS_SERVER": str(ENABLE_METRICS_SERVER).lower(),
    "KUBERNETES_VERSION": KUBERNETES_VERSION,
    "RUN_ID": RUN_ID,
    "OUTPUT_DIR": OUTPUT_DIR,
    "HOME_REGION": HOME_REGION,
    "PARENT_COMPARTMENT_OCID": PARENT_COMPARTMENT_OCID,
    "IAM_POLICY_COMPARTMENT_OCID": IAM_POLICY_COMPARTMENT_OCID,
    "CA_NAMESPACE": CA_NAMESPACE,
    "CA_SERVICE_ACCOUNT": CA_SERVICE_ACCOUNT,
    "KPO_NAMESPACE": KPO_NAMESPACE,
    "KPO_SERVICE_ACCOUNT": KPO_SERVICE_ACCOUNT,
    "TELEMETRY_PRINCIPAL_GROUP_NAME": TELEMETRY_PRINCIPAL_GROUP_NAME,
    "CA_WORKLOAD_POLICY_NAME": CA_WORKLOAD_POLICY_NAME,
    "KPO_WORKLOAD_POLICY_NAME": KPO_WORKLOAD_POLICY_NAME,
    "KPO_NODES_POLICY_NAME": KPO_NODES_POLICY_NAME,
    "KPO_NODES_DYNAMIC_GROUP_NAME": KPO_NODES_DYNAMIC_GROUP_NAME,
    "KPO_NODES_DYNAMIC_GROUP_RULE": KPO_NODES_DYNAMIC_GROUP_RULE,
    "BENCHMARK_REGION": BENCHMARK_REGION,
    "BENCHMARK_COMPARTMENT_OCID": BENCHMARK_COMPARTMENT_OCID,
    "CA_BASELINE_POOL_SIZE": CA_BASELINE_POOL_SIZE,
    "KPO_BASELINE_POOL_SIZE": KPO_BASELINE_POOL_SIZE,
    "CA_AUTOSCALED_POOL_MIN_SIZE": CA_AUTOSCALED_POOL_MIN_SIZE,
    "CA_AUTOSCALED_POOL_MAX_SIZE": CA_AUTOSCALED_POOL_MAX_SIZE,
    "KPO_AUTOSCALED_POOL_MIN_SIZE": KPO_AUTOSCALED_POOL_MIN_SIZE,
    "KPO_AUTOSCALED_POOL_MAX_SIZE": KPO_AUTOSCALED_POOL_MAX_SIZE,
    "CA_BASELINE_POOL_SHAPE": CA_BASELINE_POOL_SHAPE,
    "CA_BASELINE_POOL_IMAGE_ID": CA_BASELINE_POOL_IMAGE_ID,
    "CA_BASELINE_POOL_FLEX_OCPUS": CA_BASELINE_POOL_FLEX_OCPUS,
    "CA_BASELINE_POOL_FLEX_MEMORY_GB": CA_BASELINE_POOL_FLEX_MEMORY_GB,
    "CA_AUTOSCALED_POOL_SHAPE": CA_AUTOSCALED_POOL_SHAPE,
    "CA_AUTOSCALED_POOL_IMAGE_ID": CA_AUTOSCALED_POOL_IMAGE_ID,
    "CA_AUTOSCALED_POOL_FLEX_OCPUS": CA_AUTOSCALED_POOL_FLEX_OCPUS,
    "CA_AUTOSCALED_POOL_FLEX_MEMORY_GB": CA_AUTOSCALED_POOL_FLEX_MEMORY_GB,
    "KPO_BASELINE_POOL_SHAPE": KPO_BASELINE_POOL_SHAPE,
    "KPO_BASELINE_POOL_IMAGE_ID": KPO_BASELINE_POOL_IMAGE_ID,
    "KPO_BASELINE_POOL_FLEX_OCPUS": KPO_BASELINE_POOL_FLEX_OCPUS,
    "KPO_BASELINE_POOL_FLEX_MEMORY_GB": KPO_BASELINE_POOL_FLEX_MEMORY_GB,
    "KPO_AUTOSCALED_POOL_SHAPE": KPO_AUTOSCALED_POOL_SHAPE,
    "KPO_AUTOSCALED_POOL_IMAGE_ID": KPO_AUTOSCALED_POOL_IMAGE_ID,
    "KPO_AUTOSCALED_POOL_FLEX_OCPUS": KPO_AUTOSCALED_POOL_FLEX_OCPUS,
    "KPO_AUTOSCALED_POOL_FLEX_MEMORY_GB": KPO_AUTOSCALED_POOL_FLEX_MEMORY_GB,
    "CONTROL_PLANE_ENDPOINT_MODE": CONTROL_PLANE_ENDPOINT_MODE,
    "CONTROL_PLANE_ALLOWED_CIDRS": CONTROL_PLANE_ALLOWED_CIDRS,
    "KUBECONFIG_ENDPOINT_MODE": KUBECONFIG_ENDPOINT_MODE,
    "K8S_API_CHECK_TIMEOUT_SECONDS": K8S_API_CHECK_TIMEOUT_SECONDS,
    "HEALTHCHECK_ALLOW_API_UNREACHABLE": str(HEALTHCHECK_ALLOW_API_UNREACHABLE).lower(),
    "METRICS_SERVER_SKIP_ADDON_DEPENDENCIES_CHECK": str(
        METRICS_SERVER_SKIP_ADDON_DEPENDENCIES_CHECK
    ).lower(),
    "KPO_HELM_RELEASE": KPO_HELM_RELEASE,
    "KPO_HELM_CHART": KPO_HELM_CHART,
    "KPO_OCINODECLASS_NAME": KPO_OCINODECLASS_NAME,
    "KPO_WORKLOAD_NODEPOOL": KPO_WORKLOAD_NODEPOOL,
    "KPO_NPN_IP_COUNT": KPO_NPN_IP_COUNT,
    "KPO_CPU_PER_OCPU_FACTOR": KPO_CPU_PER_OCPU_FACTOR,
    "KPO_IMAGE_FILTER_COMPARTMENT_ID": KPO_IMAGE_FILTER_COMPARTMENT_ID,
    "KPO_IMAGE_OS_VERSION_FILTER": KPO_IMAGE_OS_VERSION_FILTER,
    "SCENARIO_POLL_SECONDS": SCENARIO_POLL_SECONDS,
    "SCENARIO_PENDING_TIMEOUT_SECONDS": SCENARIO_PENDING_TIMEOUT_SECONDS,
    "SCENARIO_READY_TIMEOUT_SECONDS": SCENARIO_READY_TIMEOUT_SECONDS,
    "SCENARIO_SCALE_DOWN_TIMEOUT_SECONDS": SCENARIO_SCALE_DOWN_TIMEOUT_SECONDS,
    "SCENARIO_BURST_REPLICAS": SCENARIO_BURST_REPLICAS,
    "SCENARIO_BURST_HOLD_SECONDS": SCENARIO_BURST_HOLD_SECONDS,
    "SCENARIO_STEADY_REPLICAS": SCENARIO_STEADY_REPLICAS,
    "SCENARIO_STEADY_HOLD_SECONDS": SCENARIO_STEADY_HOLD_SECONDS,
    "CA_EXPECTED_BENCHMARK_NODE_FLOOR": CA_EXPECTED_BENCHMARK_NODE_FLOOR,
    "KPO_EXPECTED_BENCHMARK_NODE_FLOOR": KPO_EXPECTED_BENCHMARK_NODE_FLOOR,
    "REQUIRE_KPO_NODECLAIMS_AT_FLOOR": str(REQUIRE_KPO_NODECLAIMS_AT_FLOOR).lower(),
    "WORKLOAD_NAME": WORKLOAD_NAME,
    "WORKLOAD_IMAGE": WORKLOAD_IMAGE,
    "WORKLOAD_BASE_REPLICAS": WORKLOAD_BASE_REPLICAS,
    "WORKLOAD_CPU_REQUEST": WORKLOAD_CPU_REQUEST,
    "WORKLOAD_MEM_REQUEST": WORKLOAD_MEM_REQUEST,
    "WORKLOAD_CPU_LIMIT": WORKLOAD_CPU_LIMIT,
    "WORKLOAD_MEM_LIMIT": WORKLOAD_MEM_LIMIT,
    "WORKLOAD_NAMESPACE_PREFIX": WORKLOAD_NAMESPACE_PREFIX,
    "WORKLOAD_NAMESPACE_KPO": WORKLOAD_NAMESPACE_KPO,
    "WORKLOAD_DEPLOYMENT_KPO": WORKLOAD_DEPLOYMENT_KPO,
    "KPO_REMEDIATION_APPLY_NSG": str(KPO_REMEDIATION_APPLY_NSG).lower(),
    "KPO_REMEDIATION_APPLY_LIMITS": str(KPO_REMEDIATION_APPLY_LIMITS).lower(),
    "KPO_REMEDIATION_LIMITS_CPU": KPO_REMEDIATION_LIMITS_CPU,
    "KPO_REMEDIATION_LIMITS_MEMORY": KPO_REMEDIATION_LIMITS_MEMORY,
    "KPO_REMEDIATION_LIMITS_CPU_HEADROOM": KPO_REMEDIATION_LIMITS_CPU_HEADROOM,
    "KPO_REMEDIATION_LIMITS_MEMORY_HEADROOM_GI": KPO_REMEDIATION_LIMITS_MEMORY_HEADROOM_GI,
}.items():
    _setenv(k, v)

summary = {
    "TARGET_CLUSTER": TARGET_CLUSTER,
    "TARGET_SCENARIOS": TARGET_SCENARIOS,
    "REPEATS": REPEATS,
    "IMAGE_MODE": IMAGE_MODE,
    "ENABLE_METRICS_SERVER": ENABLE_METRICS_SERVER,
    "OCI_PROFILE": OCI_PROFILE,
    "REGION": REGION,
    "KUBERNETES_VERSION": KUBERNETES_VERSION,
    "OUTPUT_DIR": OUTPUT_DIR,
    "RUN_ID": RUN_ID,
    "CA_NAMESPACE": CA_NAMESPACE,
    "KPO_NAMESPACE": KPO_NAMESPACE,
    "CA_AUTOSCALED_POOL_MIN_SIZE": CA_AUTOSCALED_POOL_MIN_SIZE,
    "KPO_AUTOSCALED_POOL_MIN_SIZE": KPO_AUTOSCALED_POOL_MIN_SIZE,
    "WORKLOAD_IMAGE": WORKLOAD_IMAGE,
    "SCENARIO_BURST_REPLICAS": SCENARIO_BURST_REPLICAS,
    "SCENARIO_STEADY_REPLICAS": SCENARIO_STEADY_REPLICAS,
}

print("Cell 2 complete: SSOT initialized (no network calls).")
print(json.dumps(summary, indent=2))

print("\nDeferred selections for Cell 3 UI:")
for name in [
    "BENCHMARK_REGION",
    "BENCHMARK_COMPARTMENT_OCID",
    "PARENT_COMPARTMENT_OCID",
    "IAM_POLICY_COMPARTMENT_OCID",
    "HOME_REGION",
]:
    print(f"- {name}")
print("\nNext: Run Cell 3 (interactive UI + validation).")

In [ ]:
# Cell 3 — Interactive UI + Validation (minimal user-facing controls)
import os
import re
import json
import ipywidgets as widgets
from IPython.display import display, HTML

import oci

display(
    HTML(
        """
<style>
  .nb3-container { max-width: 1400px; margin: 0 auto; }
  .nb3-section { border: 1px solid #e0e0e0; background: #f8f9fb; padding: 12px; margin: 8px 0; border-left: 4px solid #d2d7e5; }
  .nb3-title { font-weight: 600; margin: 0 0 6px 0; text-align: left; }
  .nb3-lbl {
    white-space: normal !important;
    overflow: visible !important;
    text-overflow: clip !important;
    line-height: 1.2;
    font-weight: 500;
  }
  .widget-label {
    white-space: normal !important;
    overflow: visible !important;
    text-overflow: clip !important;
    min-width: 320px !important;
  }
  .issues-strip { background: #fdecea; color: #ba1a1a; padding: 6px 10px; border: 1px solid #f5c2c7; border-radius: 6px; font-size: 13px; margin: 0 0 6px 0; }
  .ok-strip { background: #e8f4fd; color: #0b5394; padding: 6px 10px; border: 1px solid #a4c2f4; border-radius: 6px; font-size: 13px; margin: 0 0 6px 0; }
</style>
"""
    )
)


def _env(k, d=""):
    return os.environ.get(k, d)


def _setenv(k, v):
    os.environ[k] = "" if v is None else str(v)


def _bool(v, default=False):
    if v is None:
        return default
    return str(v).strip().lower() in {"1", "true", "yes", "y", "on"}


def _to_int(v, d=0):
    try:
        return int(v)
    except Exception:
        return d


def _to_float(v, d=0.0):
    try:
        return float(v)
    except Exception:
        return d


def _get_attr(obj, names, default=None):
    if obj is None:
        return default
    for n in names:
        if hasattr(obj, n):
            val = getattr(obj, n)
            if val is not None:
                return val
    return default


def _is_flex(shape_name):
    return bool(shape_name and str(shape_name).endswith(".Flex"))


def _k8_minor(version):
    m = re.search(r"v?(\d+)\.(\d+)", str(version))
    if not m:
        return ""
    return f"{m.group(1)}.{m.group(2)}"


def _k8_ge(version, major, minor):
    m = re.search(r"v?(\d+)\.(\d+)", str(version))
    if not m:
        return False
    vmaj, vmin = int(m.group(1)), int(m.group(2))
    return (vmaj, vmin) >= (major, minor)


def _safe_set_options(dd, options, preferred=None):
    if not options:
        options = [("No options available", "")]
    values = [v for _, v in options]
    current = dd.value if dd.value in values else None
    if preferred in values:
        value = preferred
    elif current in values:
        value = current
    else:
        value = values[0]
    dd.options = options
    dd.value = value


def _list_all(fn, *args, **kwargs):
    return oci.pagination.list_call_get_all_results(fn, *args, **kwargs).data


def _make_cfg(base_cfg, region):
    c = dict(base_cfg)
    c["region"] = region
    return c


def _shape_bounds(shape_obj):
    ocpu_opts = _get_attr(shape_obj, ["ocpu_options"], None)
    mem_opts = _get_attr(shape_obj, ["memory_options"], None)

    min_ocpu = _to_float(
        _get_attr(ocpu_opts, ["min", "min_in_ocpus", "min_ocpu"], 1.0), 1.0
    )
    max_ocpu = _to_float(
        _get_attr(ocpu_opts, ["max", "max_in_ocpus", "max_ocpu"], 64.0), 64.0
    )

    min_mem = _to_float(
        _get_attr(mem_opts, ["min_in_g_bs", "min_in_gbs", "min_memory_in_g_bs"], 1.0),
        1.0,
    )
    max_mem = _to_float(
        _get_attr(
            mem_opts, ["max_in_g_bs", "max_in_gbs", "max_memory_in_g_bs"], 1024.0
        ),
        1024.0,
    )

    default_per_ocpu = _to_float(
        _get_attr(
            mem_opts, ["default_per_ocpu_in_g_bs", "default_per_ocpu_in_gbs"], 16.0
        ),
        16.0,
    )
    return min_ocpu, max_ocpu, min_mem, max_mem, default_per_ocpu


def _row(label_text: str, control: widgets.Widget, label_width="360px") -> widgets.HBox:
    lbl = widgets.HTML(
        f"<span class='nb3-lbl'>{label_text}</span>",
        layout=widgets.Layout(width=label_width, min_width=label_width),
    )
    if hasattr(control, "layout"):
        control.layout.flex = "1 1 auto"
        control.layout.width = "auto"
    return widgets.HBox(
        [lbl, control],
        layout=widgets.Layout(
            width="100%", justify_content="flex-start", align_items="center", gap="12px"
        ),
    )


def _section(title: str, *children: widgets.Widget) -> widgets.VBox:
    return widgets.VBox(
        [widgets.HTML(f"<div class='nb3-title'>{title}</div>"), *children],
        layout=widgets.Layout(width="100%", align_items="stretch"),
        _dom_classes=["nb3-section"],
    )


OCI_CONFIG_FILE = os.path.expanduser(_env("OCI_CONFIG_FILE", "~/.oci/config"))
OCI_PROFILE = _env("OCI_PROFILE", "DEFAULT")
base_cfg = oci.config.from_file(file_location=OCI_CONFIG_FILE, profile_name=OCI_PROFILE)
TENANCY_OCID = _env("TENANCY_OCID", base_cfg.get("tenancy", ""))

if not TENANCY_OCID:
    raise RuntimeError(
        "TENANCY_OCID is empty. Set it in env or OCI config before running Cell 3."
    )

idc_home = oci.identity.IdentityClient(base_cfg)

subs = _list_all(idc_home.list_region_subscriptions, TENANCY_OCID)
region_names = sorted(
    {
        s.region_name
        for s in subs
        if getattr(s, "status", "READY") in ("READY", "ACTIVE", None)
    }
)
if not region_names:
    raise RuntimeError("No subscribed regions returned for tenancy.")

home_regions = [
    s.region_name for s in subs if bool(getattr(s, "is_home_region", False))
]
home_region_default = _env(
    "HOME_REGION", home_regions[0] if home_regions else region_names[0]
)
region_default = _env(
    "BENCHMARK_REGION", _env("REGION", base_cfg.get("region", region_names[0]))
)

if region_default not in region_names:
    region_default = region_names[0]
if home_region_default not in region_names:
    home_region_default = region_names[0]

comp_data = _list_all(
    idc_home.list_compartments,
    TENANCY_OCID,
    compartment_id_in_subtree=True,
    access_level="ACCESSIBLE",
)
comp_active = [c for c in comp_data if getattr(c, "lifecycle_state", "") == "ACTIVE"]

root_entry = ("TENANCY_ROOT", TENANCY_OCID)
comp_options = [root_entry] + sorted(
    [(f"{c.name}", c.id) for c in comp_active],
    key=lambda x: x[0].lower(),
)

parent_comp_default = _env(
    "PARENT_COMPARTMENT_OCID", _env("BENCHMARK_COMPARTMENT_OCID", "")
)
if parent_comp_default not in [v for _, v in comp_options]:
    parent_comp_default = TENANCY_OCID

iam_policy_comp_default = _env("IAM_POLICY_COMPARTMENT_OCID", parent_comp_default)
if iam_policy_comp_default not in [v for _, v in comp_options]:
    iam_policy_comp_default = parent_comp_default

status_html = widgets.HTML(
    "<div class='ok-strip'><b>Cell 3</b>: loading regional options...</div>"
)
issues_html = widgets.HTML("")
apply_out = widgets.Output(
    layout=widgets.Layout(border="1px solid #ddd", padding="8px")
)

region_dd = widgets.Dropdown(
    options=[(r, r) for r in region_names], value=region_default
)
home_region_dd = widgets.Dropdown(
    options=[(r, r) for r in region_names], value=home_region_default
)
bench_comp_dd = widgets.Dropdown(options=comp_options, value=parent_comp_default)
parent_comp_dd = widgets.Dropdown(options=comp_options, value=parent_comp_default)
iam_comp_dd = widgets.Dropdown(options=comp_options, value=iam_policy_comp_default)

target_dd = widgets.Dropdown(
    options=[("both", "both"), ("ca", "ca"), ("kpo", "kpo")],
    value=_env("TARGET_CLUSTER", "both"),
)
scenario_dd = widgets.Dropdown(
    options=[
        ("all", "all"),
        ("burst", "burst"),
        ("steady", "steady"),
        ("scale_down", "scale_down"),
    ],
    value=_env("TARGET_SCENARIOS", "all"),
)
repeats_in = widgets.BoundedIntText(
    value=_to_int(_env("REPEATS", "3"), 3), min=1, max=20
)
image_mode_dd = widgets.Dropdown(
    options=[("pin", "pin"), ("filter", "filter")], value=_env("IMAGE_MODE", "pin")
)
metrics_server_cb = widgets.Checkbox(
    value=_bool(_env("ENABLE_METRICS_SERVER", "false")), indent=False
)

k8_dd = widgets.Dropdown(
    options=[
        (_env("KUBERNETES_VERSION", "v1.34.2"), _env("KUBERNETES_VERSION", "v1.34.2"))
    ]
)

burst_replicas_in = widgets.BoundedIntText(
    value=_to_int(_env("SCENARIO_BURST_REPLICAS", "20"), 20), min=1, max=500
)
steady_replicas_in = widgets.BoundedIntText(
    value=_to_int(_env("SCENARIO_STEADY_REPLICAS", "8"), 8), min=1, max=500
)

workload_image_in = widgets.Text(
    value=_env("WORKLOAD_IMAGE", "registry.k8s.io/pause:3.9").strip()
)
workload_base_replicas_in = widgets.BoundedIntText(
    value=_to_int(_env("WORKLOAD_BASE_REPLICAS", "0"), 0), min=0, max=500
)
workload_cpu_request_in = widgets.Text(
    value=_env("WORKLOAD_CPU_REQUEST", "500m").strip()
)
workload_mem_request_in = widgets.Text(
    value=_env("WORKLOAD_MEM_REQUEST", "512Mi").strip()
)
workload_cpu_limit_in = widgets.Text(
    value=_env("WORKLOAD_CPU_LIMIT", _env("WORKLOAD_CPU_REQUEST", "500m")).strip()
)
workload_mem_limit_in = widgets.Text(
    value=_env("WORKLOAD_MEM_LIMIT", _env("WORKLOAD_MEM_REQUEST", "512Mi")).strip()
)

shape_options = [("Loading shapes...", "")]
image_options = [("Loading images...", "")]
shape_map = {}


def _pool_ui(title, prefix, autoscaled=False):
    shape_dd = widgets.Dropdown(options=shape_options)
    image_dd = widgets.Dropdown(options=image_options)

    if autoscaled:
        min_in = widgets.BoundedIntText(
            value=_to_int(_env(f"{prefix}_MIN_SIZE", "0"), 0),
            min=0,
            max=200,
        )
        max_in = widgets.BoundedIntText(
            value=_to_int(_env(f"{prefix}_MAX_SIZE", "5"), 5),
            min=0,
            max=300,
        )
        minmax_row = widgets.HBox(
            [
                _row("Minimum node count", min_in, label_width="240px"),
                _row("Maximum node count", max_in, label_width="240px"),
            ],
            layout=widgets.Layout(width="100%", gap="12px"),
        )
        size_in = None
    else:
        default_size = "2"
        size_in = widgets.BoundedIntText(
            value=_to_int(
                _env(f"{prefix}_SIZE", default_size), _to_int(default_size, 2)
            ),
            min=1,
            max=200,
        )
        min_in = None
        max_in = None
        minmax_row = _row("Node count", size_in)

    flex_ocpu = widgets.BoundedFloatText(
        value=_to_float(_env(f"{prefix}_FLEX_OCPUS", "2"), 2.0),
        min=1,
        max=128,
        step=1,
    )
    flex_mem = widgets.BoundedFloatText(
        value=_to_float(_env(f"{prefix}_FLEX_MEMORY_GB", "16"), 16.0),
        min=1,
        max=1024,
        step=1,
    )

    flex_box = widgets.VBox(
        [
            _row("Flex OCPUs", flex_ocpu),
            _row("Flex memory (GB)", flex_mem),
        ]
    )

    state = {
        "title": title,
        "prefix": prefix,
        "autoscaled": autoscaled,
        "shape_dd": shape_dd,
        "image_dd": image_dd,
        "size_in": size_in,
        "min_in": min_in,
        "max_in": max_in,
        "flex_ocpu": flex_ocpu,
        "flex_mem": flex_mem,
        "flex_box": flex_box,
    }

    box = _section(
        title,
        _row("Shape", shape_dd),
        _row("Image", image_dd),
        minmax_row,
        flex_box,
    )
    return state, box


ca_base_state, ca_base_box = _pool_ui(
    "CA Baseline Pool (non-autoscaled)", "CA_BASELINE_POOL", autoscaled=False
)
ca_auto_state, ca_auto_box = _pool_ui(
    "CA Autoscaled Benchmark Pool", "CA_AUTOSCALED_POOL", autoscaled=True
)
kpo_base_state, kpo_base_box = _pool_ui(
    "KPO Baseline Pool (non-autoscaled)", "KPO_BASELINE_POOL", autoscaled=False
)
kpo_auto_state, kpo_auto_box = _pool_ui(
    "KPO Autoscaled Profile", "KPO_AUTOSCALED_POOL", autoscaled=True
)
all_pools = [ca_base_state, ca_auto_state, kpo_base_state, kpo_auto_state]


def _refresh_flex_for_pool(pool_state):
    shape_name = pool_state["shape_dd"].value
    is_flex = _is_flex(shape_name)
    pool_state["flex_box"].layout.display = "" if is_flex else "none"
    if not is_flex:
        return

    shp = shape_map.get(shape_name)
    if shp is None:
        return

    min_ocpu, max_ocpu, min_mem, max_mem, default_per_ocpu = _shape_bounds(shp)

    pool_state["flex_ocpu"].min = min_ocpu
    pool_state["flex_ocpu"].max = max_ocpu
    if (
        pool_state["flex_ocpu"].value < min_ocpu
        or pool_state["flex_ocpu"].value > max_ocpu
    ):
        pool_state["flex_ocpu"].value = min_ocpu

    pool_state["flex_mem"].min = min_mem
    pool_state["flex_mem"].max = max_mem
    suggested_mem = max(
        min_mem, min(max_mem, pool_state["flex_ocpu"].value * default_per_ocpu)
    )
    if pool_state["flex_mem"].value < min_mem or pool_state["flex_mem"].value > max_mem:
        pool_state["flex_mem"].value = suggested_mem


def _pool_values(pool_state):
    d = {
        "shape": pool_state["shape_dd"].value,
        "image_id": pool_state["image_dd"].value,
        "flex_ocpus": (
            pool_state["flex_ocpu"].value
            if _is_flex(pool_state["shape_dd"].value)
            else None
        ),
        "flex_memory_gb": (
            pool_state["flex_mem"].value
            if _is_flex(pool_state["shape_dd"].value)
            else None
        ),
    }
    if pool_state["autoscaled"]:
        d["min_size"] = pool_state["min_in"].value
        d["max_size"] = pool_state["max_in"].value
    else:
        d["size"] = pool_state["size_in"].value
    return d


def _make_region_clients(region):
    cfg = _make_cfg(base_cfg, region)
    return (
        oci.container_engine.ContainerEngineClient(cfg),
        oci.core.ComputeClient(cfg),
    )


def _load_k8_versions(ce_client):
    try:
        resp = ce_client.get_cluster_options(cluster_option_id="all").data
        versions = list(getattr(resp, "kubernetes_versions", []) or [])
    except Exception:
        versions = []

    if not versions:
        versions = [_env("KUBERNETES_VERSION", "v1.34.2")]

    versions = [v for v in versions if _k8_ge(v, 1, 31)]
    if not versions:
        versions = [_env("KUBERNETES_VERSION", "v1.34.2")]

    return sorted(set(versions), reverse=True)


def _load_shapes(comp_client, compartment_id):
    data = _list_all(comp_client.list_shapes, compartment_id)
    active = [
        s for s in data if getattr(s, "lifecycle_state", "ACTIVE") in ("ACTIVE", None)
    ]
    names = sorted({s.shape for s in active})
    mapping = {s.shape: s for s in active}
    opts = [(n, n) for n in names] if names else [("No shapes found", "")]
    return opts, mapping


def _load_oke_images(ce_client, comp_client, compartment_id, k8_version):
    minor = _k8_minor(k8_version)
    options = []

    try:
        np_opts = ce_client.get_node_pool_options(node_pool_option_id="all").data
        sources = list(getattr(np_opts, "sources", []) or [])
        for s in sources:
            image_id = getattr(s, "image_id", "")
            source_name = getattr(s, "source_name", "")
            if not image_id:
                continue
            if minor and minor not in source_name:
                continue
            options.append((f"{source_name} | {image_id[:24]}...", image_id))
    except Exception:
        pass

    if not options:
        try:
            imgs = _list_all(comp_client.list_images, compartment_id)
            for img in imgs:
                name = getattr(img, "display_name", "") or ""
                image_id = getattr(img, "id", "")
                if not image_id:
                    continue
                if "OKE" not in name.upper():
                    continue
                if minor and minor not in name:
                    continue
                options.append((f"{name} | {image_id[:24]}...", image_id))
        except Exception:
            pass

    if not options:
        return [("No compatible images discovered", "")]

    dedup = {}
    for lbl, iid in options:
        dedup[iid] = lbl
    return [(lbl, iid) for iid, lbl in dedup.items()]


def _is_image_shape_compatible(comp_client, image_id, shape_name):
    try:
        entries = _list_all(
            comp_client.list_image_shape_compatibility_entries, image_id=image_id
        )
        shapes = {e.shape for e in entries}
        return shape_name in shapes
    except Exception:
        return False


def _on_shape_change(pool_state):
    _refresh_flex_for_pool(pool_state)


def _refresh_region_data(*_):
    issues_html.value = ""
    status_html.value = "<div class='ok-strip'><b>Cell 3</b>: loading region-scoped versions/shapes/images...</div>"

    region = region_dd.value
    comp_id = bench_comp_dd.value

    ce_client, comp_client = _make_region_clients(region)

    k8_versions = _load_k8_versions(ce_client)
    _safe_set_options(
        k8_dd,
        [(v, v) for v in k8_versions],
        preferred=_env("KUBERNETES_VERSION", k8_versions[0]),
    )

    global shape_map
    s_opts, s_map = _load_shapes(comp_client, comp_id)
    shape_map = s_map

    for pool in all_pools:
        _safe_set_options(
            pool["shape_dd"], s_opts, preferred=_env(f"{pool['prefix']}_SHAPE", "")
        )
        _refresh_flex_for_pool(pool)

    i_opts = _load_oke_images(ce_client, comp_client, comp_id, k8_dd.value)
    for pool in all_pools:
        _safe_set_options(
            pool["image_dd"], i_opts, preferred=_env(f"{pool['prefix']}_IMAGE_ID", "")
        )

    status_html.value = "<div class='ok-strip'><b>Cell 3</b>: ready. Review selections and click Apply.</div>"


def _refresh_images_only(*_):
    region = region_dd.value
    comp_id = bench_comp_dd.value
    ce_client, comp_client = _make_region_clients(region)
    i_opts = _load_oke_images(ce_client, comp_client, comp_id, k8_dd.value)
    for pool in all_pools:
        _safe_set_options(pool["image_dd"], i_opts, preferred=pool["image_dd"].value)


def _validate_and_apply(_):
    issues = []
    apply_out.clear_output()
    issues_html.value = ""

    region = region_dd.value
    home_region = home_region_dd.value
    bench_comp = bench_comp_dd.value
    parent_comp = parent_comp_dd.value
    iam_comp = iam_comp_dd.value

    target = target_dd.value
    scenarios = scenario_dd.value
    repeats = repeats_in.value
    image_mode = image_mode_dd.value
    enable_metrics = metrics_server_cb.value
    k8_version = k8_dd.value

    burst_replicas = burst_replicas_in.value
    steady_replicas = steady_replicas_in.value

    workload_image = workload_image_in.value.strip()
    workload_base_replicas = workload_base_replicas_in.value
    workload_cpu_request = workload_cpu_request_in.value.strip()
    workload_mem_request = workload_mem_request_in.value.strip()
    workload_cpu_limit = workload_cpu_limit_in.value.strip()
    workload_mem_limit = workload_mem_limit_in.value.strip()

    if not _k8_ge(k8_version, 1, 31):
        issues.append(f"Kubernetes version must be >=1.31. Selected: {k8_version}")

    if not region:
        issues.append("Region is required.")
    if not bench_comp:
        issues.append("Benchmark compartment is required.")
    if not parent_comp:
        issues.append("Parent compartment is required.")
    if not iam_comp:
        issues.append("IAM policy compartment is required.")
    if not home_region:
        issues.append("Home region is required for strict IAM mode.")

    if burst_replicas < 1:
        issues.append("SCENARIO_BURST_REPLICAS must be >= 1")
    if steady_replicas < 1:
        issues.append("SCENARIO_STEADY_REPLICAS must be >= 1")

    if not workload_image:
        issues.append("WORKLOAD_IMAGE must not be empty")
    if not workload_cpu_request or not workload_mem_request:
        issues.append("WORKLOAD_CPU_REQUEST and WORKLOAD_MEM_REQUEST must not be empty")
    if not workload_cpu_limit or not workload_mem_limit:
        issues.append("WORKLOAD_CPU_LIMIT and WORKLOAD_MEM_LIMIT must not be empty")

    pools = {
        "CA_BASELINE_POOL": _pool_values(ca_base_state),
        "CA_AUTOSCALED_POOL": _pool_values(ca_auto_state),
        "KPO_BASELINE_POOL": _pool_values(kpo_base_state),
        "KPO_AUTOSCALED_POOL": _pool_values(kpo_auto_state),
    }

    for name, p in pools.items():
        if not p["shape"]:
            issues.append(f"{name}: shape is required.")
        if not p["image_id"]:
            issues.append(f"{name}: image_id is required.")
        if "size" in p and p["size"] < 1:
            issues.append(f"{name}: size must be >=1.")
        if "min_size" in p:
            if p["min_size"] > p["max_size"]:
                issues.append(f"{name}: min_size cannot exceed max_size.")
            if p["min_size"] < 0 or p["max_size"] < 0:
                issues.append(f"{name}: min/max must be >=0.")
        if _is_flex(p["shape"]):
            if p["flex_ocpus"] is None or p["flex_ocpus"] <= 0:
                issues.append(f"{name}: flex_ocpus must be > 0 for Flex shapes.")
            if p["flex_memory_gb"] is None or p["flex_memory_gb"] <= 0:
                issues.append(f"{name}: flex_memory_gb must be > 0 for Flex shapes.")

    if target in ("both", "kpo") and pools["KPO_BASELINE_POOL"]["size"] < 2:
        issues.append(
            "KPO_BASELINE_POOL: size must be >=2 for default KPO chart readiness "
            "(replicaCount=2 with required pod anti-affinity)."
        )

    if target == "both":
        ca_auto = pools["CA_AUTOSCALED_POOL"]
        kpo_auto = pools["KPO_AUTOSCALED_POOL"]
        if (
            ca_auto["min_size"] != kpo_auto["min_size"]
            or ca_auto["max_size"] != kpo_auto["max_size"]
        ):
            issues.append(
                "Fairness check failed: CA and KPO autoscaled min/max must match in TARGET_CLUSTER=both."
            )
        if ca_auto["shape"] != kpo_auto["shape"]:
            issues.append(
                "Fairness check failed: CA and KPO autoscaled shapes must match in TARGET_CLUSTER=both."
            )
        if _is_flex(ca_auto["shape"]) and _is_flex(kpo_auto["shape"]):
            if abs((ca_auto["flex_ocpus"] or 0) - (kpo_auto["flex_ocpus"] or 0)) > 1e-9:
                issues.append(
                    "Fairness check failed: CA and KPO autoscaled Flex OCPU must match."
                )
            if (
                abs(
                    (ca_auto["flex_memory_gb"] or 0) - (kpo_auto["flex_memory_gb"] or 0)
                )
                > 1e-9
            ):
                issues.append(
                    "Fairness check failed: CA and KPO autoscaled Flex memory must match."
                )

    ce_client, comp_client = _make_region_clients(region)
    target_pools = []
    if target in ("both", "ca"):
        target_pools += [
            ("CA_BASELINE_POOL", pools["CA_BASELINE_POOL"]),
            ("CA_AUTOSCALED_POOL", pools["CA_AUTOSCALED_POOL"]),
        ]
    if target in ("both", "kpo"):
        target_pools += [
            ("KPO_BASELINE_POOL", pools["KPO_BASELINE_POOL"]),
            ("KPO_AUTOSCALED_POOL", pools["KPO_AUTOSCALED_POOL"]),
        ]

    for name, p in target_pools:
        if p["image_id"] and p["shape"]:
            ok = _is_image_shape_compatible(comp_client, p["image_id"], p["shape"])
            if not ok:
                issues.append(
                    f"{name}: selected image is not shape-compatible for {p['shape']}."
                )

    if issues:
        issues_html.value = (
            "<div class='issues-strip'><b>Validation failed:</b><br>- "
            + "<br>- ".join(issues)
            + "</div>"
        )
        with apply_out:
            print("Cell 3 apply failed. Fix validation issues and re-apply.")
        return

    _setenv("BENCHMARK_REGION", region)
    _setenv("REGION", region)
    _setenv("HOME_REGION", home_region)
    _setenv("BENCHMARK_COMPARTMENT_OCID", bench_comp)
    _setenv("PARENT_COMPARTMENT_OCID", parent_comp)
    _setenv("IAM_POLICY_COMPARTMENT_OCID", iam_comp)

    _setenv("TARGET_CLUSTER", target)
    _setenv("TARGET_SCENARIOS", scenarios)
    _setenv("REPEATS", repeats)
    _setenv("IMAGE_MODE", image_mode)
    _setenv("ENABLE_METRICS_SERVER", str(enable_metrics).lower())
    _setenv("KUBERNETES_VERSION", k8_version)

    _setenv("SCENARIO_BURST_REPLICAS", burst_replicas)
    _setenv("SCENARIO_STEADY_REPLICAS", steady_replicas)

    _setenv("WORKLOAD_IMAGE", workload_image)
    _setenv("WORKLOAD_BASE_REPLICAS", workload_base_replicas)
    _setenv("WORKLOAD_CPU_REQUEST", workload_cpu_request)
    _setenv("WORKLOAD_MEM_REQUEST", workload_mem_request)
    _setenv("WORKLOAD_CPU_LIMIT", workload_cpu_limit)
    _setenv("WORKLOAD_MEM_LIMIT", workload_mem_limit)

    for name, p in pools.items():
        _setenv(f"{name}_SHAPE", p["shape"])
        _setenv(f"{name}_IMAGE_ID", p["image_id"])
        if "size" in p:
            _setenv(f"{name}_SIZE", p["size"])
        if "min_size" in p:
            _setenv(f"{name}_MIN_SIZE", p["min_size"])
            _setenv(f"{name}_MAX_SIZE", p["max_size"])
        _setenv(
            f"{name}_FLEX_OCPUS", "" if p["flex_ocpus"] is None else p["flex_ocpus"]
        )
        _setenv(
            f"{name}_FLEX_MEMORY_GB",
            "" if p["flex_memory_gb"] is None else p["flex_memory_gb"],
        )

    _setenv("CA_EXPECTED_BENCHMARK_NODE_FLOOR", pools["CA_AUTOSCALED_POOL"]["min_size"])
    _setenv(
        "KPO_EXPECTED_BENCHMARK_NODE_FLOOR", pools["KPO_AUTOSCALED_POOL"]["min_size"]
    )

    summary = {
        "BENCHMARK_REGION": region,
        "HOME_REGION": home_region,
        "BENCHMARK_COMPARTMENT_OCID": bench_comp,
        "PARENT_COMPARTMENT_OCID": parent_comp,
        "IAM_POLICY_COMPARTMENT_OCID": iam_comp,
        "TARGET_CLUSTER": target,
        "TARGET_SCENARIOS": scenarios,
        "REPEATS": repeats,
        "IMAGE_MODE": image_mode,
        "ENABLE_METRICS_SERVER": enable_metrics,
        "KUBERNETES_VERSION": k8_version,
        "SCENARIO_BURST_REPLICAS": burst_replicas,
        "SCENARIO_STEADY_REPLICAS": steady_replicas,
        "WORKLOAD_IMAGE": workload_image,
        "CA_BASELINE_POOL": pools["CA_BASELINE_POOL"],
        "CA_AUTOSCALED_POOL": pools["CA_AUTOSCALED_POOL"],
        "KPO_BASELINE_POOL": pools["KPO_BASELINE_POOL"],
        "KPO_AUTOSCALED_POOL": pools["KPO_AUTOSCALED_POOL"],
    }

    issues_html.value = "<div class='ok-strip'><b>Validation passed.</b> Selections persisted to environment.</div>"
    with apply_out:
        print("Cell 3 complete: Selections applied.")
        print(json.dumps(summary, indent=2))


top_section = _section(
    "Global Selection",
    _row("Benchmark region", region_dd),
    _row("Home region (IAM)", home_region_dd),
    _row("Benchmark compartment", bench_comp_dd),
    _row("Parent compartment", parent_comp_dd),
    _row("IAM policy compartment", iam_comp_dd),
    _row("Target cluster(s)", target_dd),
    _row("Scenarios", scenario_dd),
    _row("Repeats", repeats_in),
    _row("Image mode", image_mode_dd),
    _row("Enable Metrics Server", metrics_server_cb),
    _row("Kubernetes version", k8_dd),
)

benchmark_knobs_section = _section(
    "Benchmark Knobs",
    _row("Burst replicas", burst_replicas_in),
    _row("Steady replicas", steady_replicas_in),
)

workload_section = _section(
    "Workload Profile",
    _row("Workload image", workload_image_in),
    _row("Base replicas", workload_base_replicas_in),
    _row("CPU request", workload_cpu_request_in),
    _row("Memory request", workload_mem_request_in),
    _row("CPU limit", workload_cpu_limit_in),
    _row("Memory limit", workload_mem_limit_in),
)

apply_btn = widgets.Button(
    description="Apply Selections", button_style="primary", icon="check"
)
apply_btn.layout.width = "auto"
apply_btn.on_click(_validate_and_apply)
apply_row = widgets.HBox(
    [apply_btn], layout=widgets.Layout(width="100%", justify_content="center")
)

ui = widgets.VBox(
    [
        status_html,
        top_section,
        benchmark_knobs_section,
        workload_section,
        ca_base_box,
        ca_auto_box,
        kpo_base_box,
        kpo_auto_box,
        apply_row,
        issues_html,
        apply_out,
    ],
    layout=widgets.Layout(width="100%"),
    _dom_classes=["nb3-container"],
)

region_dd.observe(_refresh_region_data, names="value")
bench_comp_dd.observe(_refresh_region_data, names="value")
k8_dd.observe(_refresh_images_only, names="value")

for pool in all_pools:
    pool["shape_dd"].observe(lambda ch, p=pool: _on_shape_change(p), names="value")

_refresh_region_data()
display(ui)

print("Cell 3 loaded. Select values, then click 'Apply Selections'.")
print(
    "Note: advanced/internal defaults remain in Cell 2 SSOT and are intentionally not exposed here."
)
print("Next: Run Cell 4.")

In [ ]:
# Cell 4 — Generate both Terraform stacks (tf/ca + tf/kpo) in one pass from Cell 3 SSOT
# Updated: baseline-mapped system add-ons (CoreDNS + CA add-on), endpoint mode defaults, OCI provider args.

import os
import re
import json
from pathlib import Path
from datetime import datetime, timezone


# ---------------------------
# Helpers
# ---------------------------
def _env(k, d=""):
    return os.environ.get(k, d)


def _require(k):
    v = _env(k, "").strip()
    if not v:
        raise RuntimeError(f"Missing required environment variable: {k}")
    return v


def _as_bool(v, default=False):
    if v is None:
        return default
    return str(v).strip().lower() in {"1", "true", "yes", "y", "on"}


def _as_int(v, name):
    try:
        return int(v)
    except Exception:
        raise RuntimeError(f"Expected integer for {name}, got: {v}")


def _as_float(v, name):
    try:
        return float(v)
    except Exception:
        raise RuntimeError(f"Expected float for {name}, got: {v}")


def _csv_list(v: str):
    return [x.strip() for x in str(v).split(",") if x.strip()]


def _is_flex(shape):
    return str(shape).endswith(".Flex")


def _safe_name(s, max_len=36):
    x = re.sub(r"[^a-z0-9-]+", "-", s.lower())
    x = re.sub(r"-+", "-", x).strip("-")
    if not x:
        x = "bench"
    return x[:max_len].rstrip("-")


def _dns_label(seed, max_len=15):
    x = re.sub(r"[^a-z0-9]+", "", seed.lower())
    if not x:
        x = "oke"
    if not x[0].isalpha():
        x = "a" + x
    return x[:max_len]


def _find_repo_root(start: Path) -> Path:
    p = start.resolve()
    for c in [p] + list(p.parents):
        if (c / "references" / "terraform-oci-oke-main").exists():
            return c
    raise RuntimeError(
        "Could not find repo root containing references/terraform-oci-oke-main"
    )


def _load_pool(prefix: str, autoscaled: bool):
    shape = _require(f"{prefix}_SHAPE")
    image_id = _require(f"{prefix}_IMAGE_ID")

    out = {
        "shape": shape,
        "image_id": image_id,
    }

    if autoscaled:
        out["min_size"] = _as_int(_require(f"{prefix}_MIN_SIZE"), f"{prefix}_MIN_SIZE")
        out["max_size"] = _as_int(_require(f"{prefix}_MAX_SIZE"), f"{prefix}_MAX_SIZE")
        if out["min_size"] > out["max_size"]:
            raise RuntimeError(f"{prefix}: min_size cannot be greater than max_size")
    else:
        out["size"] = _as_int(_require(f"{prefix}_SIZE"), f"{prefix}_SIZE")
        if out["size"] < 1:
            raise RuntimeError(f"{prefix}: size must be >= 1")

    if _is_flex(shape):
        out["flex_ocpus"] = _as_float(
            _require(f"{prefix}_FLEX_OCPUS"), f"{prefix}_FLEX_OCPUS"
        )
        out["flex_memory_gb"] = _as_float(
            _require(f"{prefix}_FLEX_MEMORY_GB"), f"{prefix}_FLEX_MEMORY_GB"
        )
        if out["flex_ocpus"] <= 0 or out["flex_memory_gb"] <= 0:
            raise RuntimeError(f"{prefix}: flex ocpus/memory must be > 0")
    else:
        out["flex_ocpus"] = None
        out["flex_memory_gb"] = None

    return out


def _mk_worker_pool_entry(
    pool_name,
    pool,
    autoscaled,
    allow_autoscaler=False,
    autoscale=False,
    extra_labels=None,
):
    labels = {
        "benchmark.oracle.com/profile": pool_name,
    }
    if extra_labels:
        labels.update(extra_labels)

    entry = {
        "description": pool_name,
        "mode": "node-pool",
        "shape": pool["shape"],
        "image_type": "custom",
        "image_id": pool["image_id"],
        "node_labels": labels,
    }

    if _is_flex(pool["shape"]):
        entry["ocpus"] = pool["flex_ocpus"]
        entry["memory"] = pool["flex_memory_gb"]

    if autoscaled:
        entry["size"] = pool["min_size"]
        entry["min_size"] = pool["min_size"]
        entry["max_size"] = pool["max_size"]
        entry["autoscale"] = autoscale
        entry["allow_autoscaler"] = allow_autoscaler
        entry["ignore_initial_pool_size"] = True
    else:
        entry["size"] = pool["size"]
        entry["allow_autoscaler"] = allow_autoscaler

    return entry


def _write_text(path: Path, content: str):
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(content, encoding="utf-8")


def _write_json(path: Path, data):
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(data, indent=2, sort_keys=True) + "\n", encoding="utf-8")


# ---------------------------
# Read SSOT values from Cell 2/3
# ---------------------------
TENANCY_ID = _require("TENANCY_OCID")
REGION = _require("BENCHMARK_REGION")
HOME_REGION = _require("HOME_REGION")
COMPARTMENT_ID = _require("BENCHMARK_COMPARTMENT_OCID")
PARENT_COMPARTMENT_OCID = _require("PARENT_COMPARTMENT_OCID")
IAM_POLICY_COMPARTMENT_OCID = _require("IAM_POLICY_COMPARTMENT_OCID")
OCI_PROFILE = _require("OCI_PROFILE")
OCI_CONFIG_FILE = _require("OCI_CONFIG_FILE")
K8S_VERSION = _require("KUBERNETES_VERSION")
RUN_ID = _require("RUN_ID")

TARGET_CLUSTER = _env("TARGET_CLUSTER", "both").strip().lower()
TARGET_SCENARIOS = _env("TARGET_SCENARIOS", "all").strip().lower()
REPEATS = _as_int(_env("REPEATS", "3"), "REPEATS")
IMAGE_MODE = _env("IMAGE_MODE", "pin").strip().lower()
ENABLE_METRICS_SERVER = _as_bool(_env("ENABLE_METRICS_SERVER", "false"), False)

CA_NAMESPACE = _env("CA_NAMESPACE", "kube-system")
CA_SERVICE_ACCOUNT = _env("CA_SERVICE_ACCOUNT", "cluster-autoscaler")
KPO_NAMESPACE = _env("KPO_NAMESPACE", "karpenter")
KPO_SERVICE_ACCOUNT = _env("KPO_SERVICE_ACCOUNT", "karpenter")
KPO_NPN_IP_COUNT = _as_int(_env("KPO_NPN_IP_COUNT", "16"), "KPO_NPN_IP_COUNT")
KPO_CPU_PER_OCPU_FACTOR = _as_float(
    _env("KPO_CPU_PER_OCPU_FACTOR", "2.0"), "KPO_CPU_PER_OCPU_FACTOR"
)

CONTROL_PLANE_ENDPOINT_MODE = (
    _env("CONTROL_PLANE_ENDPOINT_MODE", "public").strip().lower()
)
CONTROL_PLANE_ALLOWED_CIDRS_RAW = _env("CONTROL_PLANE_ALLOWED_CIDRS", "").strip()

if not re.search(r"v?1\.(3[1-9]|[4-9][0-9])", K8S_VERSION):
    raise RuntimeError(
        f"Kubernetes version must be >= 1.31 for this benchmark plan. Got: {K8S_VERSION}"
    )
if KPO_NPN_IP_COUNT < 1 or KPO_NPN_IP_COUNT > 256:
    raise RuntimeError(
        f"KPO_NPN_IP_COUNT must be within [1,256]. Got: {KPO_NPN_IP_COUNT}"
    )
if (KPO_NPN_IP_COUNT & (KPO_NPN_IP_COUNT - 1)) != 0:
    raise RuntimeError(
        f"KPO_NPN_IP_COUNT must be a power of two. Got: {KPO_NPN_IP_COUNT}"
    )
if KPO_CPU_PER_OCPU_FACTOR <= 0:
    raise RuntimeError(
        f"KPO_CPU_PER_OCPU_FACTOR must be > 0. Got: {KPO_CPU_PER_OCPU_FACTOR}"
    )
if CONTROL_PLANE_ENDPOINT_MODE not in {"public", "private"}:
    raise RuntimeError(
        f"CONTROL_PLANE_ENDPOINT_MODE must be public|private. Got: {CONTROL_PLANE_ENDPOINT_MODE}"
    )

CONTROL_PLANE_IS_PUBLIC = CONTROL_PLANE_ENDPOINT_MODE == "public"
ASSIGN_PUBLIC_IP_TO_CONTROL_PLANE = CONTROL_PLANE_ENDPOINT_MODE == "public"

if CONTROL_PLANE_ENDPOINT_MODE == "public":
    if CONTROL_PLANE_ALLOWED_CIDRS_RAW:
        CONTROL_PLANE_ALLOWED_CIDRS = _csv_list(CONTROL_PLANE_ALLOWED_CIDRS_RAW)
    else:
        CONTROL_PLANE_ALLOWED_CIDRS = ["0.0.0.0/0"]
    if not CONTROL_PLANE_ALLOWED_CIDRS:
        raise RuntimeError(
            "Public control plane mode requires non-empty CONTROL_PLANE_ALLOWED_CIDRS."
        )
else:
    CONTROL_PLANE_ALLOWED_CIDRS = _csv_list(CONTROL_PLANE_ALLOWED_CIDRS_RAW)

# Make later Terraform subprocesses honor selected OCI config/profile
os.environ["OCI_CLI_CONFIG_FILE"] = OCI_CONFIG_FILE
os.environ["OCI_CONFIG_FILE_PROFILE"] = OCI_PROFILE
os.environ["TF_VAR_config_file_profile"] = OCI_PROFILE

ca_baseline = _load_pool("CA_BASELINE_POOL", autoscaled=False)
ca_autoscaled = _load_pool("CA_AUTOSCALED_POOL", autoscaled=True)
kpo_baseline = _load_pool("KPO_BASELINE_POOL", autoscaled=False)
kpo_autoscaled = _load_pool("KPO_AUTOSCALED_POOL", autoscaled=True)

if TARGET_CLUSTER == "both":
    if ca_autoscaled["shape"] != kpo_autoscaled["shape"]:
        raise RuntimeError(
            "Fairness gate: CA and KPO autoscaled shapes must match in TARGET_CLUSTER=both"
        )
    if (
        ca_autoscaled["min_size"] != kpo_autoscaled["min_size"]
        or ca_autoscaled["max_size"] != kpo_autoscaled["max_size"]
    ):
        raise RuntimeError(
            "Fairness gate: CA and KPO autoscaled min/max must match in TARGET_CLUSTER=both"
        )
    if _is_flex(ca_autoscaled["shape"]):
        if (
            abs(
                (ca_autoscaled["flex_ocpus"] or 0) - (kpo_autoscaled["flex_ocpus"] or 0)
            )
            > 1e-9
        ):
            raise RuntimeError(
                "Fairness gate: CA and KPO autoscaled flex OCPU must match in TARGET_CLUSTER=both"
            )
        if (
            abs(
                (ca_autoscaled["flex_memory_gb"] or 0)
                - (kpo_autoscaled["flex_memory_gb"] or 0)
            )
            > 1e-9
        ):
            raise RuntimeError(
                "Fairness gate: CA and KPO autoscaled flex memory must match in TARGET_CLUSTER=both"
            )

# ---------------------------
# Paths
# ---------------------------
cwd = Path.cwd()
repo_root = _find_repo_root(cwd)
module_src = repo_root / "references" / "terraform-oci-oke-main"

tf_root = repo_root / "tf"
ca_dir = tf_root / "ca"
kpo_dir = tf_root / "kpo"

for d in [ca_dir, kpo_dir]:
    d.mkdir(parents=True, exist_ok=True)

ca_module_src_rel = os.path.relpath(module_src, ca_dir)
kpo_module_src_rel = os.path.relpath(module_src, kpo_dir)

# ---------------------------
# Common Terraform file templates
# ---------------------------
versions_tf = """terraform {
  required_version = ">= 1.3.0"

  required_providers {
    oci = {
      source  = "oracle/oci"
      version = ">= 7.30.0"
    }
  }
}
"""

providers_tf = """variable "benchmark" {
  type = any
}

provider "oci" {
  config_file_profile = var.benchmark.config_file_profile
  tenancy_ocid        = var.benchmark.tenancy_id
  region              = var.benchmark.region
}

provider "oci" {
  alias               = "home"
  config_file_profile = var.benchmark.config_file_profile
  tenancy_ocid        = var.benchmark.tenancy_id
  region              = var.benchmark.home_region
}
"""


def _main_tf(module_source_rel: str) -> str:
    return f"""module "oke" {{
  source = "{module_source_rel}"

  providers = {{
    oci      = oci
    oci.home = oci.home
  }}

  tenancy_id             = var.benchmark.tenancy_id
  compartment_id         = var.benchmark.compartment_id
  worker_compartment_id  = var.benchmark.worker_compartment_id
  network_compartment_id = var.benchmark.network_compartment_id

  region      = var.benchmark.region
  home_region = var.benchmark.home_region

  create_iam_resources         = var.benchmark.create_iam_resources
  create_iam_autoscaler_policy = var.benchmark.create_iam_autoscaler_policy
  create_iam_worker_policy     = var.benchmark.create_iam_worker_policy
  create_iam_operator_policy   = var.benchmark.create_iam_operator_policy
  create_iam_kms_policy        = var.benchmark.create_iam_kms_policy

  create_vcn    = var.benchmark.create_vcn
  vcn_name      = var.benchmark.vcn_name
  vcn_dns_label = var.benchmark.vcn_dns_label

  create_cluster     = var.benchmark.create_cluster
  cluster_type       = var.benchmark.cluster_type
  cluster_name       = var.benchmark.cluster_name
  kubernetes_version = var.benchmark.kubernetes_version

  cni_type                          = var.benchmark.cni_type
  control_plane_is_public           = var.benchmark.control_plane_is_public
  assign_public_ip_to_control_plane = var.benchmark.assign_public_ip_to_control_plane
  control_plane_allowed_cidrs       = var.benchmark.control_plane_allowed_cidrs

  create_bastion  = var.benchmark.create_bastion
  create_operator = var.benchmark.create_operator

  output_detail = var.benchmark.output_detail

  worker_pool_mode = var.benchmark.worker_pool_mode
  worker_pool_size = var.benchmark.worker_pool_size
  worker_pools     = var.benchmark.worker_pools

  cluster_addons           = var.benchmark.cluster_addons
  cluster_addons_to_remove = var.benchmark.cluster_addons_to_remove
}}
"""


outputs_tf = """output "cluster_id" {
  value = module.oke.cluster_id
}

output "cluster_endpoints" {
  value = module.oke.cluster_endpoints
}

output "apiserver_private_host" {
  value = module.oke.apiserver_private_host
}

output "worker_pool_ids" {
  value = module.oke.worker_pool_ids
}

output "worker_pools" {
  value = module.oke.worker_pools
}

output "vcn_id" {
  value = module.oke.vcn_id
}

output "control_plane_subnet_id" {
  value = module.oke.control_plane_subnet_id
}

output "worker_subnet_id" {
  value = module.oke.worker_subnet_id
}

output "pod_subnet_id" {
  value = module.oke.pod_subnet_id
}

output "int_lb_subnet_id" {
  value = module.oke.int_lb_subnet_id
}

output "pub_lb_subnet_id" {
  value = module.oke.pub_lb_subnet_id
}

output "policy_statements" {
  value = module.oke.policy_statements
}

output "dynamic_group_ids" {
  value = module.oke.dynamic_group_ids
}
"""

# ---------------------------
# Build stack data: CA
# ---------------------------
run_seed = _safe_name(RUN_ID, max_len=24)
ca_cluster_name = _safe_name(f"ca-{run_seed}", max_len=32)
kpo_cluster_name = _safe_name(f"kpo-{run_seed}", max_len=32)

ca_worker_pools = {
    "ca-baseline": _mk_worker_pool_entry(
        "ca-baseline",
        ca_baseline,
        autoscaled=False,
        allow_autoscaler=False,
        autoscale=False,
        extra_labels={
            "benchmark.oracle.com/role": "baseline",
            "benchmark.oracle.com/autoscaler": "ca",
        },
    ),
    "ca-benchmark-autoscaled": _mk_worker_pool_entry(
        "ca-benchmark-autoscaled",
        ca_autoscaled,
        autoscaled=True,
        allow_autoscaler=True,
        autoscale=True,
        extra_labels={
            "benchmark.oracle.com/role": "benchmark",
            "benchmark.oracle.com/autoscaler": "ca",
        },
    ),
}

ca_discovery = (
    f"compartmentId:{COMPARTMENT_ID},"
    f"nodepoolTags:cluster_autoscaler=managed,"
    f"min:{ca_autoscaled['min_size']},max:{ca_autoscaled['max_size']}"
)

baseline_selector_json = json.dumps({"benchmark.oracle.com/role": "baseline"})

ca_cluster_addons = {
    "ClusterAutoscaler": {
        "remove_addon_resources_on_delete": True,
        "override_existing": True,
        "configurations": [
            {"key": "authType", "value": "workload"},
            {"key": "numOfReplicas", "value": "2"},
            {"key": "nodeGroupAutoDiscovery", "value": ca_discovery},
            {"key": "scaleDownEnabled", "value": "true"},
            {"key": "maxNodeProvisionTime", "value": "25m"},
            {"key": "scanInterval", "value": "10s"},
            {"key": "unremovableNodeRecheckTimeout", "value": "5m"},
            {"key": "nodeSelectors", "value": baseline_selector_json},
        ],
    },
    "CoreDNS": {
        "remove_addon_resources_on_delete": True,
        "override_existing": True,
        "configurations": [
            {"key": "nodeSelectors", "value": baseline_selector_json},
        ],
    },
}

skip_metrics_dep = _as_bool(
    _env("METRICS_SERVER_SKIP_ADDON_DEPENDENCIES_CHECK", "false"), False
)

if ENABLE_METRICS_SERVER:
    if not skip_metrics_dep:
        ca_cluster_addons["CertManager"] = {
            "remove_addon_resources_on_delete": True,
            "override_existing": True,
            "configurations": [
                {"key": "numOfReplicas", "value": "2"},
            ],
        }
    ms_cfg = []
    if skip_metrics_dep:
        ms_cfg.append({"key": "skipAddonDependenciesCheck", "value": "true"})
    ca_cluster_addons["KubernetesMetricsServer"] = {
        "remove_addon_resources_on_delete": True,
        "override_existing": True,
        "configurations": ms_cfg,
    }

ca_benchmark = {
    "oci_config_file": OCI_CONFIG_FILE,
    "config_file_profile": OCI_PROFILE,
    "tenancy_id": TENANCY_ID,
    "compartment_id": COMPARTMENT_ID,
    "worker_compartment_id": COMPARTMENT_ID,
    "network_compartment_id": COMPARTMENT_ID,
    "region": REGION,
    "home_region": HOME_REGION,
    "create_iam_resources": True,
    "create_iam_autoscaler_policy": "never",
    "create_iam_worker_policy": "never",
    "create_iam_operator_policy": "never",
    "create_iam_kms_policy": "never",
    "create_vcn": True,
    "vcn_name": f"{ca_cluster_name}-vcn",
    "vcn_dns_label": _dns_label(ca_cluster_name),
    "create_cluster": True,
    "cluster_type": "enhanced",
    "cluster_name": ca_cluster_name,
    "kubernetes_version": K8S_VERSION,
    "cni_type": "npn",
    "control_plane_is_public": CONTROL_PLANE_IS_PUBLIC,
    "assign_public_ip_to_control_plane": ASSIGN_PUBLIC_IP_TO_CONTROL_PLANE,
    "control_plane_allowed_cidrs": CONTROL_PLANE_ALLOWED_CIDRS,
    "create_bastion": False,
    "create_operator": False,
    "output_detail": True,
    "worker_pool_mode": "node-pool",
    "worker_pool_size": 0,
    "worker_pools": ca_worker_pools,
    "cluster_addons": ca_cluster_addons,
    "cluster_addons_to_remove": {},
}

# ---------------------------
# Build stack data: KPO
# ---------------------------
kpo_worker_pools = {
    "kpo-baseline": _mk_worker_pool_entry(
        "kpo-baseline",
        kpo_baseline,
        autoscaled=False,
        allow_autoscaler=False,
        autoscale=False,
        extra_labels={
            "benchmark.oracle.com/role": "baseline",
            "benchmark.oracle.com/autoscaler": "kpo",
        },
    ),
}

kpo_cluster_addons = {
    "CoreDNS": {
        "remove_addon_resources_on_delete": True,
        "override_existing": True,
        "configurations": [
            {"key": "nodeSelectors", "value": baseline_selector_json},
        ],
    }
}
if ENABLE_METRICS_SERVER:
    if not skip_metrics_dep:
        kpo_cluster_addons["CertManager"] = {
            "remove_addon_resources_on_delete": True,
            "override_existing": True,
            "configurations": [
                {"key": "numOfReplicas", "value": "2"},
            ],
        }
    ms_cfg = []
    if skip_metrics_dep:
        ms_cfg.append({"key": "skipAddonDependenciesCheck", "value": "true"})
    kpo_cluster_addons["KubernetesMetricsServer"] = {
        "remove_addon_resources_on_delete": True,
        "override_existing": True,
        "configurations": ms_cfg,
    }

kpo_benchmark = {
    "oci_config_file": OCI_CONFIG_FILE,
    "config_file_profile": OCI_PROFILE,
    "tenancy_id": TENANCY_ID,
    "compartment_id": COMPARTMENT_ID,
    "worker_compartment_id": COMPARTMENT_ID,
    "network_compartment_id": COMPARTMENT_ID,
    "region": REGION,
    "home_region": HOME_REGION,
    "create_iam_resources": True,
    "create_iam_autoscaler_policy": "never",
    "create_iam_worker_policy": "never",
    "create_iam_operator_policy": "never",
    "create_iam_kms_policy": "never",
    "create_vcn": True,
    "vcn_name": f"{kpo_cluster_name}-vcn",
    "vcn_dns_label": _dns_label(kpo_cluster_name),
    "create_cluster": True,
    "cluster_type": "enhanced",
    "cluster_name": kpo_cluster_name,
    "kubernetes_version": K8S_VERSION,
    "cni_type": "npn",
    "control_plane_is_public": CONTROL_PLANE_IS_PUBLIC,
    "assign_public_ip_to_control_plane": ASSIGN_PUBLIC_IP_TO_CONTROL_PLANE,
    "control_plane_allowed_cidrs": CONTROL_PLANE_ALLOWED_CIDRS,
    "create_bastion": False,
    "create_operator": False,
    "output_detail": True,
    "worker_pool_mode": "node-pool",
    "worker_pool_size": 0,
    "worker_pools": kpo_worker_pools,
    "cluster_addons": kpo_cluster_addons,
    "cluster_addons_to_remove": {},
}

# ---------------------------
# Write Terraform stacks
# ---------------------------
for stack_dir, module_rel, benchmark in [
    (ca_dir, ca_module_src_rel, ca_benchmark),
    (kpo_dir, kpo_module_src_rel, kpo_benchmark),
]:
    _write_text(stack_dir / "versions.tf", versions_tf)
    _write_text(stack_dir / "providers.tf", providers_tf)
    _write_text(stack_dir / "main.tf", _main_tf(module_rel))
    _write_text(stack_dir / "outputs.tf", outputs_tf)
    _write_json(stack_dir / "terraform.auto.tfvars.json", {"benchmark": benchmark})

# ---------------------------
# KPO template artifacts for Cell 8
# ---------------------------
kpo_values_lines = [
    "settings:",
    f'  clusterCompartmentId: "{COMPARTMENT_ID}"',
    f'  vcnCompartmentId: "{COMPARTMENT_ID}"',
    '  apiserverEndpoint: "__APISERVER_ENDPOINT__"',
    "  ociVcnIpNative: true",
    "  ipFamilies:",
    "    - IPv4",
]
if _is_flex(kpo_autoscaled["shape"]):
    kpo_values_lines.extend(
        [
            "  flexibleShapeConfigs:",
            f"    - ocpus: {int(kpo_autoscaled['flex_ocpus']) if float(kpo_autoscaled['flex_ocpus']).is_integer() else kpo_autoscaled['flex_ocpus']}",
            f"      memoryInGbs: {int(kpo_autoscaled['flex_memory_gb']) if float(kpo_autoscaled['flex_memory_gb']).is_integer() else kpo_autoscaled['flex_memory_gb']}",
        ]
    )
kpo_values_yaml = "\n".join(kpo_values_lines) + "\n"

if IMAGE_MODE == "filter":
    image_filter_compartment = _env("KPO_IMAGE_FILTER_COMPARTMENT_ID", "").strip()
    filter_block = [
        "        imageFilter:",
        '          osFilter: "Oracle Linux"',
        f"          osVersionFilter: \"{_env('KPO_IMAGE_OS_VERSION_FILTER', '8')}\"",
    ]
    if image_filter_compartment:
        filter_block.append(f'          compartmentId: "{image_filter_compartment}"')
    image_config_block = "\n".join(filter_block)
else:
    image_config_block = f"        imageId: \"{kpo_autoscaled['image_id']}\""

shape_cfg = ""
if _is_flex(kpo_autoscaled["shape"]):
    shape_cfg = f"""
  shapeConfigs:
    - ocpus: {int(kpo_autoscaled['flex_ocpus']) if float(kpo_autoscaled['flex_ocpus']).is_integer() else kpo_autoscaled['flex_ocpus']}
      memoryInGbs: {int(kpo_autoscaled['flex_memory_gb']) if float(kpo_autoscaled['flex_memory_gb']).is_integer() else kpo_autoscaled['flex_memory_gb']}
""".rstrip(
        "\n"
    )

kpo_ocinodeclass_yaml = f"""apiVersion: oci.oraclecloud.com/v1beta1
kind: OCINodeClass
metadata:
  name: benchmark-ocinodeclass
spec:{shape_cfg}
  volumeConfig:
    bootVolumeConfig:
      imageConfig:
        imageType: OKEImage
{image_config_block}
  networkConfig:
    primaryVnicConfig:
      subnetConfig:
        subnetId: "__WORKER_SUBNET_ID__"
    secondaryVnicConfigs:
      - subnetConfig:
          subnetId: "__POD_SUBNET_ID__"
        ipCount: {KPO_NPN_IP_COUNT}
""".replace(
    "spec:\n  volumeConfig",
    (
        "spec:\n  volumeConfig"
        if not shape_cfg
        else "spec:" + shape_cfg + "\n  volumeConfig"
    ),
)

nodepool_limits = ""
if _is_flex(kpo_autoscaled["shape"]):
    # Karpenter NodePool cpu limits are Kubernetes CPU cores.
    # OCI flexible shapes are configured in OCPUs, so apply a conversion factor.
    cpu_limit = (
        kpo_autoscaled["max_size"]
        * kpo_autoscaled["flex_ocpus"]
        * KPO_CPU_PER_OCPU_FACTOR
    )
    mem_limit = kpo_autoscaled["max_size"] * kpo_autoscaled["flex_memory_gb"]
    cpu_limit_txt = (
        str(int(cpu_limit)) if float(cpu_limit).is_integer() else str(cpu_limit)
    )
    mem_limit_txt = (
        str(int(mem_limit)) if float(mem_limit).is_integer() else str(mem_limit)
    )
    nodepool_limits = f"""
  limits:
    cpu: "{cpu_limit_txt}"
    memory: "{mem_limit_txt}Gi"
""".rstrip(
        "\n"
    )

kpo_nodepool_yaml = f"""apiVersion: karpenter.sh/v1
kind: NodePool
metadata:
  name: benchmark-nodepool
  annotations:
    benchmark.oracle.com/min-size: "{kpo_autoscaled['min_size']}"
    benchmark.oracle.com/max-size: "{kpo_autoscaled['max_size']}"
spec:
  template:
    metadata:
      labels:
        benchmark.oracle.com/role: "benchmark"
        benchmark.oracle.com/autoscaler: "kpo"
    spec:
      nodeClassRef:
        group: oci.oraclecloud.com
        kind: OCINodeClass
        name: benchmark-ocinodeclass
      requirements:
        - key: karpenter.sh/capacity-type
          operator: In
          values: ["on-demand"]
        - key: oci.oraclecloud.com/instance-shape
          operator: In
          values: ["{kpo_autoscaled['shape']}"]
  disruption:
    consolidationPolicy: WhenEmpty
    consolidateAfter: 5m{nodepool_limits}
"""

_write_text(kpo_dir / "manifests" / "kpo-values.yaml.tmpl", kpo_values_yaml)
_write_text(kpo_dir / "manifests" / "ocinodeclass.yaml.tmpl", kpo_ocinodeclass_yaml)
_write_text(kpo_dir / "manifests" / "nodepool.yaml.tmpl", kpo_nodepool_yaml)

# ---------------------------
# IAM policy templates
# ---------------------------
ca_policy = f"""# CA workload identity policy template (fill __CA_CLUSTER_OCID__)
Allow any-user to manage cluster-node-pools in compartment id {COMPARTMENT_ID} where all {{
  request.principal.type = 'workload',
  request.principal.namespace = '{CA_NAMESPACE}',
  request.principal.service_account = '{CA_SERVICE_ACCOUNT}',
  request.principal.cluster_id = '__CA_CLUSTER_OCID__'
}}
Allow any-user to manage instance-family in compartment id {COMPARTMENT_ID} where all {{
  request.principal.type = 'workload',
  request.principal.namespace = '{CA_NAMESPACE}',
  request.principal.service_account = '{CA_SERVICE_ACCOUNT}',
  request.principal.cluster_id = '__CA_CLUSTER_OCID__'
}}
Allow any-user to use subnets in compartment id {COMPARTMENT_ID} where all {{
  request.principal.type = 'workload',
  request.principal.namespace = '{CA_NAMESPACE}',
  request.principal.service_account = '{CA_SERVICE_ACCOUNT}',
  request.principal.cluster_id = '__CA_CLUSTER_OCID__'
}}
Allow any-user to read virtual-network-family in compartment id {COMPARTMENT_ID} where all {{
  request.principal.type = 'workload',
  request.principal.namespace = '{CA_NAMESPACE}',
  request.principal.service_account = '{CA_SERVICE_ACCOUNT}',
  request.principal.cluster_id = '__CA_CLUSTER_OCID__'
}}
Allow any-user to use vnics in compartment id {COMPARTMENT_ID} where all {{
  request.principal.type = 'workload',
  request.principal.namespace = '{CA_NAMESPACE}',
  request.principal.service_account = '{CA_SERVICE_ACCOUNT}',
  request.principal.cluster_id = '__CA_CLUSTER_OCID__'
}}
Allow any-user to inspect compartments in tenancy where all {{
  request.principal.type = 'workload',
  request.principal.namespace = '{CA_NAMESPACE}',
  request.principal.service_account = '{CA_SERVICE_ACCOUNT}',
  request.principal.cluster_id = '__CA_CLUSTER_OCID__'
}}
"""

kpo_policy = f"""# KPO workload identity policy template (fill __KPO_CLUSTER_OCID__)
Allow any-user to manage instance-family in compartment id {COMPARTMENT_ID} where all {{
  request.principal.type = 'workload',
  request.principal.namespace = '{KPO_NAMESPACE}',
  request.principal.service_account = '{KPO_SERVICE_ACCOUNT}',
  request.principal.cluster_id = '__KPO_CLUSTER_OCID__'
}}
Allow any-user to manage volumes in compartment id {COMPARTMENT_ID} where all {{
  request.principal.type = 'workload',
  request.principal.namespace = '{KPO_NAMESPACE}',
  request.principal.service_account = '{KPO_SERVICE_ACCOUNT}',
  request.principal.cluster_id = '__KPO_CLUSTER_OCID__'
}}
Allow any-user to manage volume-attachments in compartment id {COMPARTMENT_ID} where all {{
  request.principal.type = 'workload',
  request.principal.namespace = '{KPO_NAMESPACE}',
  request.principal.service_account = '{KPO_SERVICE_ACCOUNT}',
  request.principal.cluster_id = '__KPO_CLUSTER_OCID__'
}}
Allow any-user to manage virtual-network-family in compartment id {COMPARTMENT_ID} where all {{
  request.principal.type = 'workload',
  request.principal.namespace = '{KPO_NAMESPACE}',
  request.principal.service_account = '{KPO_SERVICE_ACCOUNT}',
  request.principal.cluster_id = '__KPO_CLUSTER_OCID__'
}}
Allow any-user to inspect compartments in tenancy where all {{
  request.principal.type = 'workload',
  request.principal.namespace = '{KPO_NAMESPACE}',
  request.principal.service_account = '{KPO_SERVICE_ACCOUNT}',
  request.principal.cluster_id = '__KPO_CLUSTER_OCID__'
}}
"""

if IMAGE_MODE == "filter" and _env("KPO_IMAGE_FILTER_COMPARTMENT_ID", "").strip():
    img_comp = _env("KPO_IMAGE_FILTER_COMPARTMENT_ID").strip()
    kpo_policy += f"""
Allow any-user to read instance-images in compartment id {img_comp} where all {{
  request.principal.type = 'workload',
  request.principal.namespace = '{KPO_NAMESPACE}',
  request.principal.service_account = '{KPO_SERVICE_ACCOUNT}',
  request.principal.cluster_id = '__KPO_CLUSTER_OCID__'
}}
"""

kpo_nodes_join = f"""# KPO node registration template
# Dynamic group rule:
# ALL {{instance.compartment.id = '{COMPARTMENT_ID}'}}

# Policy:
Allow dynamic-group __KPO_NODES_DYNAMIC_GROUP_NAME__ to {{CLUSTER_JOIN}} in compartment id {COMPARTMENT_ID}
where {{ target.cluster.id = '__KPO_CLUSTER_OCID__' }}
"""

telemetry_policy = """# Telemetry policy template
# Attach to the principal/group that will query oci_oke metrics:
Allow group __TELEMETRY_GROUP_NAME__ to read metrics in tenancy
"""

_write_text(ca_dir / "iam" / "ca_workload_identity.policy.tmpl", ca_policy)
_write_text(kpo_dir / "iam" / "kpo_workload_identity.policy.tmpl", kpo_policy)
_write_text(kpo_dir / "iam" / "kpo_nodes_cluster_join.policy.tmpl", kpo_nodes_join)
_write_text(tf_root / "iam" / "telemetry.policy.tmpl", telemetry_policy)

# ---------------------------
# Stack metadata
# ---------------------------
generation_summary = {
    "generated_at_utc": datetime.now(timezone.utc).isoformat().replace("+00:00", "Z"),
    "repo_root": str(repo_root),
    "tf_root": str(tf_root),
    "module_source": str(module_src),
    "target_cluster": TARGET_CLUSTER,
    "target_scenarios": TARGET_SCENARIOS,
    "repeats": REPEATS,
    "image_mode": IMAGE_MODE,
    "enable_metrics_server": ENABLE_METRICS_SERVER,
    "control_plane_endpoint_mode": CONTROL_PLANE_ENDPOINT_MODE,
    "control_plane_is_public": CONTROL_PLANE_IS_PUBLIC,
    "assign_public_ip_to_control_plane": ASSIGN_PUBLIC_IP_TO_CONTROL_PLANE,
    "control_plane_allowed_cidrs": CONTROL_PLANE_ALLOWED_CIDRS,
    "region": REGION,
    "home_region": HOME_REGION,
    "kubernetes_version": K8S_VERSION,
    "kpo_cpu_per_ocpu_factor": KPO_CPU_PER_OCPU_FACTOR,
    "system_addons_mapped_to_baseline": True,
    "ca_stack_dir": str(ca_dir),
    "kpo_stack_dir": str(kpo_dir),
}
_write_json(tf_root / "generation-summary.json", generation_summary)

# ---------------------------
# Print summary
# ---------------------------
generated_files = [
    ca_dir / "versions.tf",
    ca_dir / "providers.tf",
    ca_dir / "main.tf",
    ca_dir / "outputs.tf",
    ca_dir / "terraform.auto.tfvars.json",
    ca_dir / "iam" / "ca_workload_identity.policy.tmpl",
    kpo_dir / "versions.tf",
    kpo_dir / "providers.tf",
    kpo_dir / "main.tf",
    kpo_dir / "outputs.tf",
    kpo_dir / "terraform.auto.tfvars.json",
    kpo_dir / "manifests" / "kpo-values.yaml.tmpl",
    kpo_dir / "manifests" / "ocinodeclass.yaml.tmpl",
    kpo_dir / "manifests" / "nodepool.yaml.tmpl",
    kpo_dir / "iam" / "kpo_workload_identity.policy.tmpl",
    kpo_dir / "iam" / "kpo_nodes_cluster_join.policy.tmpl",
    tf_root / "iam" / "telemetry.policy.tmpl",
    tf_root / "generation-summary.json",
]

print("Cell 4 complete: Terraform stacks generated for both CA and KPO.")
print(f"- CA stack:  {ca_dir}")
print(f"- KPO stack: {kpo_dir}")
print(f"- Module source: {module_src}")
print("\nGenerated files:")
for f in generated_files:
    print(f"- {f}")

print("\nKey contract checks baked into generated config:")
print("- create_vcn=true, create_cluster=true, cluster_type=enhanced")
print("- create_iam_resources=true (strict self-contained benchmark mode)")
print(
    f"- control plane endpoint mode: {CONTROL_PLANE_ENDPOINT_MODE} (control_plane_is_public={CONTROL_PLANE_IS_PUBLIC}, assign_public_ip_to_control_plane={ASSIGN_PUBLIC_IP_TO_CONTROL_PLANE})"
)
print(f"- control_plane_allowed_cidrs={CONTROL_PLANE_ALLOWED_CIDRS}")
print(f"- KPO NodePool cpu/OCPU conversion factor: {KPO_CPU_PER_OCPU_FACTOR}")
print(
    "- CA uses ClusterAutoscaler add-on with authType=workload and nodeGroupAutoDiscovery"
)
print(
    "- CoreDNS and CA add-on are pinned to baseline nodes for clean autoscaled-pool scale-down"
)
print(
    "- Terraform autoscaled CA pool uses autoscale=true + ignore_initial_pool_size=true"
)
print("- KPO stack includes Helm values + OCINodeClass + NodePool templates")
print(
    "- OCI provider blocks use supported args: config_file_profile + tenancy_ocid + region"
)
if ENABLE_METRICS_SERVER:
    if skip_metrics_dep:
        print(
            "- Metrics Server add-on enabled with skipAddonDependenciesCheck=true (standalone cert-manager path)"
        )
    else:
        print("- Metrics Server add-on enabled with CertManager add-on dependency")

print("\nNext: Run Cell 5")

In [ ]:
# Cell 5 — Preflight Gates (blocking): config + doc/version + OCI checks + terraform checks
# Updated:
# - provider gate checks unsupported config_file argument correctly (line-level regex)
# - terraform fmt auto-fix before fmt -check
# - terraform init -upgrade to refresh providers after plugin reset

import os
import re
import json
import subprocess
from pathlib import Path

import oci


def _env(k, d=""):
    return os.environ.get(k, d)


def _require_env(k):
    v = _env(k, "").strip()
    if not v:
        raise RuntimeError(f"Missing required environment variable: {k}")
    return v


def _k8_minor(version: str) -> str:
    m = re.search(r"v?(\d+)\.(\d+)", version)
    if not m:
        return ""
    return f"{m.group(1)}.{m.group(2)}"


def _k8_ge(version: str, maj: int, minr: int) -> bool:
    m = re.search(r"v?(\d+)\.(\d+)", version)
    if not m:
        return False
    return (int(m.group(1)), int(m.group(2))) >= (maj, minr)


def _load_json(path: Path):
    return json.loads(path.read_text(encoding="utf-8"))


def _run(cmd, cwd=None):
    print(f"\n$ {' '.join(cmd)}")
    p = subprocess.run(cmd, cwd=cwd, text=True, capture_output=True)
    if p.stdout:
        print(p.stdout)
    if p.stderr:
        print(p.stderr)
    return p.returncode


def _bool(x):
    return str(x).strip().lower() in {"1", "true", "yes", "y", "on"}


def _assert(cond, ok_msg, fail_msg, oks, errs):
    if cond:
        oks.append(ok_msg)
    else:
        errs.append(fail_msg)


def _contains_addon(addons: dict, addon_name: str) -> bool:
    return isinstance(addons, dict) and addon_name in addons


def _addon_cfg_map(addon_obj):
    cfgs = addon_obj.get("configurations", []) if isinstance(addon_obj, dict) else []
    out = {}
    for c in cfgs:
        if isinstance(c, dict) and "key" in c:
            out[c["key"]] = c.get("value", "")
    return out


cwd = Path.cwd().resolve()


def _find_repo_root(start: Path) -> Path:
    for p in [start] + list(start.parents):
        if (p / "references" / "terraform-oci-oke-main").exists():
            return p
    raise RuntimeError(
        "Could not locate repo root containing references/terraform-oci-oke-main"
    )


repo_root = _find_repo_root(cwd)
tf_root = repo_root / "tf"
ca_dir = tf_root / "ca"
kpo_dir = tf_root / "kpo"

target = _env("TARGET_CLUSTER", "both").strip().lower()
if target not in {"ca", "kpo", "both"}:
    raise RuntimeError(f"TARGET_CLUSTER must be ca|kpo|both, got: {target}")

targets = ["ca", "kpo"] if target == "both" else [target]

print("=== Cell 5 Preflight ===")
print(f"Repo root: {repo_root}")
print(f"Selected target(s): {targets}")

os.environ["OCI_CLI_CONFIG_FILE"] = _require_env("OCI_CONFIG_FILE")
os.environ["OCI_CONFIG_FILE_PROFILE"] = _require_env("OCI_PROFILE")
os.environ["TF_VAR_config_file_profile"] = _require_env("OCI_PROFILE")

required_files = [
    ca_dir / "main.tf",
    ca_dir / "providers.tf",
    ca_dir / "terraform.auto.tfvars.json",
    kpo_dir / "main.tf",
    kpo_dir / "providers.tf",
    kpo_dir / "terraform.auto.tfvars.json",
    repo_root
    / "references"
    / "docs"
    / "ContEng"
    / "Tasks"
    / "contengconfiguringclusteraddons-supportedversions.md",
]
for f in required_files:
    if not f.exists():
        raise RuntimeError(f"Required file missing: {f}")

ca_bm = _load_json(ca_dir / "terraform.auto.tfvars.json")["benchmark"]
kpo_bm = _load_json(kpo_dir / "terraform.auto.tfvars.json")["benchmark"]

supported_addon_versions = {
    "1.32": {
        "ClusterAutoscaler": "1.32.5",
        "CertManager": "1.17.1",
        "KubernetesMetricsServer": "0.7.2",
    },
    "1.33": {
        "ClusterAutoscaler": "1.33.3",
        "CertManager": "1.17.1",
        "KubernetesMetricsServer": "0.7.2",
    },
    "1.34": {
        "ClusterAutoscaler": "1.34.2",
        "CertManager": "1.17.1",
        "KubernetesMetricsServer": "0.7.2",
    },
}

oks = []
errs = []

k8 = _require_env("KUBERNETES_VERSION")
bench_region = _require_env("BENCHMARK_REGION")
bench_compartment = _require_env("BENCHMARK_COMPARTMENT_OCID")
home_region = _require_env("HOME_REGION")
tenancy_ocid = _require_env("TENANCY_OCID")
enable_metrics = _bool(_env("ENABLE_METRICS_SERVER", "false"))

_assert(
    _k8_ge(k8, 1, 31),
    f"Kubernetes version gate passed ({k8} >= 1.31).",
    f"Kubernetes version gate failed: {k8} must be >= 1.31.",
    oks,
    errs,
)

k8_minor = _k8_minor(k8)
_assert(
    k8_minor in supported_addon_versions,
    f"Addon support table found for Kubernetes {k8_minor}.",
    f"No addon support mapping for Kubernetes {k8_minor}. Update supported_addon_versions map from docs.",
    oks,
    errs,
)

for name, bm in [("ca", ca_bm), ("kpo", kpo_bm)]:
    _assert(
        bm.get("create_vcn") is True,
        f"{name}: create_vcn=true",
        f"{name}: create_vcn must be true",
        oks,
        errs,
    )
    _assert(
        bm.get("create_cluster") is True,
        f"{name}: create_cluster=true",
        f"{name}: create_cluster must be true",
        oks,
        errs,
    )
    _assert(
        str(bm.get("cluster_type", "")).lower() == "enhanced",
        f"{name}: cluster_type=enhanced",
        f"{name}: cluster_type must be enhanced",
        oks,
        errs,
    )
    _assert(
        str(bm.get("cni_type", "")).lower() == "npn",
        f"{name}: cni_type=npn (VCN-native).",
        f"{name}: cni_type must be npn for this rebuild.",
        oks,
        errs,
    )
    _assert(
        str(bm.get("kubernetes_version", "")) == k8,
        f"{name}: kubernetes_version matches SSOT ({k8})",
        f"{name}: kubernetes_version mismatch vs SSOT",
        oks,
        errs,
    )
    _assert(
        bm.get("create_iam_resources") is True,
        f"{name}: create_iam_resources=true",
        f"{name}: create_iam_resources must be true in benchmark mode",
        oks,
        errs,
    )
    _assert(
        bool(home_region),
        f"{name}: HOME_REGION present for IAM operations",
        f"{name}: HOME_REGION missing",
        oks,
        errs,
    )

for name, p in [("ca", ca_dir / "providers.tf"), ("kpo", kpo_dir / "providers.tf")]:
    txt = p.read_text(encoding="utf-8")
    _assert(
        "config_file_profile" in txt,
        f"{name}: provider uses config_file_profile.",
        f"{name}: provider block missing config_file_profile.",
        oks,
        errs,
    )
    has_unsupported_config_file = re.search(r"(?m)^\s*config_file\s*=", txt) is not None
    _assert(
        not has_unsupported_config_file,
        f"{name}: provider does not use unsupported config_file arg.",
        f"{name}: provider block still contains unsupported config_file arg.",
        oks,
        errs,
    )

if "ca" in targets:
    ca_addons = ca_bm.get("cluster_addons", {})
    wp = ca_bm.get("worker_pools", {})

    _assert(
        _contains_addon(ca_addons, "ClusterAutoscaler"),
        "ca: ClusterAutoscaler add-on configured.",
        "ca: ClusterAutoscaler add-on missing.",
        oks,
        errs,
    )

    if _contains_addon(ca_addons, "ClusterAutoscaler"):
        ca_cfg = _addon_cfg_map(ca_addons["ClusterAutoscaler"])
        _assert(
            ca_cfg.get("authType") == "workload",
            "ca: authType=workload",
            "ca: ClusterAutoscaler authType must be workload",
            oks,
            errs,
        )

        replicas = (
            int(ca_cfg.get("numOfReplicas", "0"))
            if str(ca_cfg.get("numOfReplicas", "")).isdigit()
            else 0
        )
        _assert(
            replicas >= 2,
            f"ca: numOfReplicas={replicas} (>=2)",
            f"ca: numOfReplicas must be >=2 (got {replicas})",
            oks,
            errs,
        )

        has_nodes = "nodes" in ca_cfg and str(ca_cfg.get("nodes", "")).strip() != ""
        has_ngad = (
            "nodeGroupAutoDiscovery" in ca_cfg
            and str(ca_cfg.get("nodeGroupAutoDiscovery", "")).strip() != ""
        )
        _assert(
            has_nodes != has_ngad,
            "ca: exactly one of nodes/nodeGroupAutoDiscovery is set.",
            "ca: must set exactly one of nodes or nodeGroupAutoDiscovery (not both/none).",
            oks,
            errs,
        )

        if has_ngad:
            _assert(
                _k8_ge(k8, 1, 32),
                f"ca: nodeGroupAutoDiscovery gate passed for Kubernetes {k8}.",
                f"ca: nodeGroupAutoDiscovery requires supported CA addon versions; enforce Kubernetes >=1.32 in this workflow.",
                oks,
                errs,
            )

    autoscaled_pools = [
        v for _, v in wp.items() if isinstance(v, dict) and v.get("autoscale") is True
    ]
    _assert(
        len(autoscaled_pools) >= 1,
        "ca: autoscaled worker pool exists.",
        "ca: no autoscaled worker pool found (autoscale=true).",
        oks,
        errs,
    )
    for idx, p in enumerate(autoscaled_pools):
        _assert(
            p.get("ignore_initial_pool_size") is True,
            f"ca: autoscaled pool #{idx+1} has ignore_initial_pool_size=true.",
            f"ca: autoscaled pool #{idx+1} must set ignore_initial_pool_size=true.",
            oks,
            errs,
        )

    if enable_metrics:
        _assert(
            _contains_addon(ca_addons, "KubernetesMetricsServer"),
            "ca: KubernetesMetricsServer add-on present.",
            "ca: ENABLE_METRICS_SERVER=true but KubernetesMetricsServer add-on missing.",
            oks,
            errs,
        )
        if _contains_addon(ca_addons, "KubernetesMetricsServer"):
            ms_cfg = _addon_cfg_map(ca_addons["KubernetesMetricsServer"])
            has_cert = _contains_addon(ca_addons, "CertManager")
            skip_dep = (
                str(ms_cfg.get("skipAddonDependenciesCheck", "false")).lower() == "true"
            )
            _assert(
                has_cert or skip_dep,
                "ca: Metrics Server dependency gate passed (CertManager present or skipAddonDependenciesCheck=true).",
                "ca: Metrics Server dependency gate failed (need CertManager add-on or skipAddonDependenciesCheck=true).",
                oks,
                errs,
            )

if "kpo" in targets:
    kpo_addons = kpo_bm.get("cluster_addons", {})
    _assert(
        (kpo_dir / "manifests" / "kpo-values.yaml.tmpl").exists(),
        "kpo: kpo-values.yaml.tmpl present.",
        "kpo: missing kpo-values.yaml.tmpl",
        oks,
        errs,
    )
    _assert(
        (kpo_dir / "manifests" / "ocinodeclass.yaml.tmpl").exists(),
        "kpo: ocinodeclass.yaml.tmpl present.",
        "kpo: missing ocinodeclass.yaml.tmpl",
        oks,
        errs,
    )
    _assert(
        (kpo_dir / "manifests" / "nodepool.yaml.tmpl").exists(),
        "kpo: nodepool.yaml.tmpl present.",
        "kpo: missing nodepool.yaml.tmpl",
        oks,
        errs,
    )
    _assert(
        (kpo_dir / "iam" / "kpo_workload_identity.policy.tmpl").exists(),
        "kpo: workload identity policy template present.",
        "kpo: missing kpo_workload_identity.policy.tmpl",
        oks,
        errs,
    )
    _assert(
        (kpo_dir / "iam" / "kpo_nodes_cluster_join.policy.tmpl").exists(),
        "kpo: cluster join policy template present.",
        "kpo: missing kpo_nodes_cluster_join.policy.tmpl",
        oks,
        errs,
    )

    kpo_values_text = (kpo_dir / "manifests" / "kpo-values.yaml.tmpl").read_text(
        encoding="utf-8"
    )
    oci_nc_text = (kpo_dir / "manifests" / "ocinodeclass.yaml.tmpl").read_text(
        encoding="utf-8"
    )
    _assert(
        "__APISERVER_ENDPOINT__" in kpo_values_text,
        "kpo: apiserver endpoint placeholder present.",
        "kpo: kpo-values.yaml.tmpl must include __APISERVER_ENDPOINT__ placeholder.",
        oks,
        errs,
    )
    _assert(
        "ociVcnIpNative: true" in kpo_values_text,
        "kpo: Helm values set ociVcnIpNative=true.",
        "kpo: kpo-values.yaml.tmpl must set ociVcnIpNative=true for VCN-native.",
        oks,
        errs,
    )
    _assert(
        "__WORKER_SUBNET_ID__" in oci_nc_text,
        "kpo: worker subnet placeholder present.",
        "kpo: ocinodeclass.yaml.tmpl must include __WORKER_SUBNET_ID__ placeholder.",
        oks,
        errs,
    )
    _assert(
        "__POD_SUBNET_ID__" in oci_nc_text,
        "kpo: pod subnet placeholder present.",
        "kpo: ocinodeclass.yaml.tmpl must include __POD_SUBNET_ID__ placeholder.",
        oks,
        errs,
    )
    _assert(
        "secondaryVnicConfigs" in oci_nc_text,
        "kpo: OCINodeClass has secondaryVnicConfigs for NPN.",
        "kpo: ocinodeclass.yaml.tmpl must include secondaryVnicConfigs for VCN-native.",
        oks,
        errs,
    )
    ip_count_match = re.search(r"ipCount:\s*(\d+)", oci_nc_text)
    if ip_count_match:
        ip_count = int(ip_count_match.group(1))
        _assert(
            (ip_count & (ip_count - 1)) == 0,
            f"kpo: ipCount={ip_count} is power-of-two.",
            f"kpo: ipCount must be power-of-two, got {ip_count}.",
            oks,
            errs,
        )
        _assert(
            ip_count <= 16,
            f"kpo: ipCount={ip_count} (safe default <=16).",
            f"kpo: ipCount should be <=16 unless CNI add-on >=3.2.0 is verified.",
            oks,
            errs,
        )
    else:
        errs.append("kpo: ipCount not found in ocinodeclass.yaml.tmpl")

    kpo_policy_text = (kpo_dir / "iam" / "kpo_workload_identity.policy.tmpl").read_text(
        encoding="utf-8"
    )
    kpo_join_text = (kpo_dir / "iam" / "kpo_nodes_cluster_join.policy.tmpl").read_text(
        encoding="utf-8"
    )
    _assert(
        "request.principal.type = 'workload'" in kpo_policy_text,
        "kpo: workload identity policy uses workload principal conditions.",
        "kpo: workload identity policy missing workload condition block.",
        oks,
        errs,
    )
    _assert(
        "Allow dynamic-group __KPO_NODES_DYNAMIC_GROUP_NAME__ to {CLUSTER_JOIN}"
        in kpo_join_text,
        "kpo: CLUSTER_JOIN statement present.",
        "kpo: cluster join policy missing CLUSTER_JOIN statement.",
        oks,
        errs,
    )
    _assert(
        "target.cluster.id = '__KPO_CLUSTER_OCID__'" in kpo_join_text,
        "kpo: CLUSTER_JOIN policy scoped to target.cluster.id.",
        "kpo: cluster join policy must scope CLUSTER_JOIN by target.cluster.id.",
        oks,
        errs,
    )

    if enable_metrics:
        _assert(
            _contains_addon(kpo_addons, "KubernetesMetricsServer"),
            "kpo: KubernetesMetricsServer add-on present.",
            "kpo: ENABLE_METRICS_SERVER=true but KubernetesMetricsServer add-on missing.",
            oks,
            errs,
        )
        if _contains_addon(kpo_addons, "KubernetesMetricsServer"):
            ms_cfg = _addon_cfg_map(kpo_addons["KubernetesMetricsServer"])
            has_cert = _contains_addon(kpo_addons, "CertManager")
            skip_dep = (
                str(ms_cfg.get("skipAddonDependenciesCheck", "false")).lower() == "true"
            )
            _assert(
                has_cert or skip_dep,
                "kpo: Metrics Server dependency gate passed (CertManager present or skipAddonDependenciesCheck=true).",
                "kpo: Metrics Server dependency gate failed (need CertManager add-on or skipAddonDependenciesCheck=true).",
                oks,
                errs,
            )

try:
    cfg = oci.config.from_file(
        file_location=_require_env("OCI_CONFIG_FILE"),
        profile_name=_require_env("OCI_PROFILE"),
    )
    cfg["region"] = bench_region

    idc = oci.identity.IdentityClient(cfg)
    ce = oci.container_engine.ContainerEngineClient(cfg)
    compute = oci.core.ComputeClient(cfg)
    limits = oci.limits.LimitsClient(cfg)

    subs = oci.pagination.list_call_get_all_results(
        idc.list_region_subscriptions, tenancy_ocid
    ).data
    subscribed_regions = {s.region_name for s in subs}
    _assert(
        bench_region in subscribed_regions,
        f"OCI: benchmark region {bench_region} is subscribed.",
        f"OCI: benchmark region {bench_region} is not subscribed in tenancy.",
        oks,
        errs,
    )

    try:
        ce_opts = ce.get_cluster_options(cluster_option_id="all").data
        ce_versions = set(getattr(ce_opts, "kubernetes_versions", []) or [])
        _assert(
            k8 in ce_versions,
            f"OCI CE: Kubernetes version {k8} available in region {bench_region}.",
            f"OCI CE: Kubernetes version {k8} not available in region {bench_region}.",
            oks,
            errs,
        )
    except Exception as e:
        errs.append(f"OCI CE options check failed: {e}")

    required_shapes = set()
    if "ca" in targets:
        for p in ca_bm.get("worker_pools", {}).values():
            if isinstance(p, dict) and p.get("shape"):
                required_shapes.add(p["shape"])
    if "kpo" in targets:
        for p in kpo_bm.get("worker_pools", {}).values():
            if isinstance(p, dict) and p.get("shape"):
                required_shapes.add(p["shape"])

    try:
        shape_objs = oci.pagination.list_call_get_all_results(
            compute.list_shapes, bench_compartment
        ).data
        available_shapes = {s.shape for s in shape_objs if getattr(s, "shape", None)}
        for shp in sorted(required_shapes):
            _assert(
                shp in available_shapes,
                f"OCI Compute: required shape available: {shp}",
                f"OCI Compute: required shape not available in region/compartment: {shp}",
                oks,
                errs,
            )
    except Exception as e:
        errs.append(f"OCI Compute shape availability check failed: {e}")

    try:
        svc_resp = oci.pagination.list_call_get_all_results(
            limits.list_services, tenancy_ocid
        ).data
        svc_names = {
            (getattr(s, "name", None) or getattr(s, "service_name", None) or "").lower()
            for s in svc_resp
        }
        _assert(
            "compute" in svc_names,
            "OCI Limits: compute service visible at tenancy scope.",
            "OCI Limits: compute service not returned by list_services.",
            oks,
            errs,
        )

        lv = oci.pagination.list_call_get_all_results(
            limits.list_limit_values,
            tenancy_ocid,
            service_name="compute",
            scope_type="REGION",
        ).data
        _assert(
            len(lv) > 0,
            "OCI Limits: compute limit values retrieved at tenancy scope.",
            "OCI Limits: no compute limit values returned at tenancy scope.",
            oks,
            errs,
        )

    except Exception as e:
        errs.append(f"OCI Limits check failed (tenancy-scoped APIs): {e}")

except Exception as e:
    errs.append(f"OCI client initialization/auth check failed: {e}")

if k8_minor in supported_addon_versions:
    if "ca" in targets:
        _assert(
            "ClusterAutoscaler" in supported_addon_versions[k8_minor],
            f"Doc gate: ClusterAutoscaler supported for Kubernetes {k8_minor}.",
            f"Doc gate: ClusterAutoscaler support not found for Kubernetes {k8_minor}.",
            oks,
            errs,
        )
    if enable_metrics:
        _assert(
            "KubernetesMetricsServer" in supported_addon_versions[k8_minor],
            f"Doc gate: KubernetesMetricsServer supported for Kubernetes {k8_minor}.",
            f"Doc gate: KubernetesMetricsServer support not found for Kubernetes {k8_minor}.",
            oks,
            errs,
        )
        _assert(
            "CertManager" in supported_addon_versions[k8_minor],
            f"Doc gate: CertManager supported for Kubernetes {k8_minor}.",
            f"Doc gate: CertManager support not found for Kubernetes {k8_minor}.",
            oks,
            errs,
        )

print("\n=== Preflight Gate Results (before Terraform commands) ===")
for x in oks:
    print(f"[PASS] {x}")
if errs:
    for x in errs:
        print(f"[FAIL] {x}")
    raise RuntimeError(
        f"Preflight failed with {len(errs)} issue(s) before Terraform checks."
    )

tf_errs = []

for t in targets:
    stack_dir = ca_dir if t == "ca" else kpo_dir
    print(f"\n=== Terraform checks for target: {t} ({stack_dir}) ===")

    steps = [
        [
            "terraform",
            f"-chdir={stack_dir}",
            "init",
            "-upgrade",
            "-input=false",
            "-no-color",
        ],
        ["terraform", f"-chdir={stack_dir}", "fmt", "-recursive"],
        ["terraform", f"-chdir={stack_dir}", "fmt", "-check", "-recursive"],
        ["terraform", f"-chdir={stack_dir}", "validate", "-no-color"],
        [
            "terraform",
            f"-chdir={stack_dir}",
            "plan",
            "-input=false",
            "-no-color",
            "-out=tfplan",
        ],
    ]

    for cmd in steps:
        rc = _run(cmd)
        if rc != 0:
            tf_errs.append(f"{t}: command failed ({' '.join(cmd)})")
            break

if tf_errs:
    print("\n=== Terraform Preflight Failures ===")
    for e in tf_errs:
        print(f"[FAIL] {e}")
    raise RuntimeError(f"Cell 5 failed with {len(tf_errs)} Terraform error(s).")

print("\nCell 5 complete: all preflight gates passed.")
print("Next: Run Cell 6.")

In [ ]:
# Cell 6 — Image discovery and resolution validation
# Primary source: OKE node pool options
# Fallback/verification: Compute image APIs
# Default mode: IMAGE_MODE=pin (reproducible image IDs)

import os
import re
import json
from pathlib import Path
from datetime import datetime, timezone

import oci


# -----------------------------
# Helpers
# -----------------------------
def _env(k, d=""):
    return os.environ.get(k, d)


def _require(k):
    v = _env(k, "").strip()
    if not v:
        raise RuntimeError(f"Missing required env var: {k}")
    return v


def _k8_minor(version: str) -> str:
    m = re.search(r"v?(\d+)\.(\d+)", version)
    if not m:
        raise RuntimeError(f"Invalid Kubernetes version: {version}")
    return f"{m.group(1)}.{m.group(2)}"


def _list_all(fn, *args, **kwargs):
    return oci.pagination.list_call_get_all_results(fn, *args, **kwargs).data


def _find_repo_root(start: Path) -> Path:
    for p in [start.resolve()] + list(start.resolve().parents):
        if (p / "references" / "terraform-oci-oke-main").exists():
            return p
    raise RuntimeError("Could not locate repo root")


def _load_json(path: Path):
    return json.loads(path.read_text(encoding="utf-8"))


def _safe_write_json(path: Path, data):
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(data, indent=2, sort_keys=True) + "\n", encoding="utf-8")


# -----------------------------
# Context
# -----------------------------
repo_root = _find_repo_root(Path.cwd())
tf_root = repo_root / "tf"
ca_tfvars = tf_root / "ca" / "terraform.auto.tfvars.json"
kpo_tfvars = tf_root / "kpo" / "terraform.auto.tfvars.json"

if not ca_tfvars.exists() or not kpo_tfvars.exists():
    raise RuntimeError("Missing generated tfvars files. Run Cell 4 first.")

target_cluster = _env("TARGET_CLUSTER", "both").strip().lower()
if target_cluster not in {"ca", "kpo", "both"}:
    raise RuntimeError(f"TARGET_CLUSTER must be ca|kpo|both, got: {target_cluster}")
selected_targets = ["ca", "kpo"] if target_cluster == "both" else [target_cluster]

image_mode = _env("IMAGE_MODE", "pin").strip().lower()
if image_mode not in {"pin", "filter"}:
    raise RuntimeError(f"IMAGE_MODE must be pin|filter, got: {image_mode}")

k8_version = _require("KUBERNETES_VERSION")
k8_minor = _k8_minor(k8_version)

region = _require("BENCHMARK_REGION")
profile = _require("OCI_PROFILE")
config_file = _require("OCI_CONFIG_FILE")

run_id = _require("RUN_ID")
output_dir = Path(_require("OUTPUT_DIR"))

os.environ["OCI_CLI_CONFIG_FILE"] = config_file
os.environ["OCI_CONFIG_FILE_PROFILE"] = profile

cfg = oci.config.from_file(file_location=config_file, profile_name=profile)
cfg["region"] = region

ce = oci.container_engine.ContainerEngineClient(cfg)
compute = oci.core.ComputeClient(cfg)

ca_bm = _load_json(ca_tfvars)["benchmark"]
kpo_bm = _load_json(kpo_tfvars)["benchmark"]


# -----------------------------
# Collect image usages from generated stacks
# -----------------------------
def _collect_usages(stack_name: str, bm: dict):
    usages = []
    pools = bm.get("worker_pools", {})
    for pool_name, pool_cfg in pools.items():
        if not isinstance(pool_cfg, dict):
            continue
        image_id = (pool_cfg.get("image_id") or "").strip()
        shape = (pool_cfg.get("shape") or "").strip()
        if image_id and shape:
            usages.append(
                {
                    "stack": stack_name,
                    "pool": pool_name,
                    "shape": shape,
                    "image_id": image_id,
                    "autoscale": bool(pool_cfg.get("autoscale", False)),
                }
            )
    return usages


usages = []
if "ca" in selected_targets:
    usages.extend(_collect_usages("ca", ca_bm))
if "kpo" in selected_targets:
    usages.extend(_collect_usages("kpo", kpo_bm))

if not usages:
    raise RuntimeError("No image usages found in generated stack tfvars.")

usage_by_image = {}
for u in usages:
    usage_by_image.setdefault(u["image_id"], []).append(u)

# -----------------------------
# Primary source: OKE node pool options
# -----------------------------
oke_sources_by_id = {}
try:
    np_opts = ce.get_node_pool_options(node_pool_option_id="all").data
    for s in list(getattr(np_opts, "sources", []) or []):
        iid = getattr(s, "image_id", None)
        if not iid:
            continue
        oke_sources_by_id[iid] = {
            "source_name": getattr(s, "source_name", ""),
            "source_type": getattr(s, "source_type", ""),
            "image_id": iid,
        }
except Exception as e:
    raise RuntimeError(
        f"Failed reading OKE node pool options (primary image source): {e}"
    )

# -----------------------------
# Validate each selected image ID
# -----------------------------
errors = []
results = []


def _display_has_k8(display_name: str, minor: str) -> bool:
    s = (display_name or "").lower()
    return ("oke" in s) and (minor in s)


for image_id, image_usages in usage_by_image.items():
    discovered_in_oke_source = image_id in oke_sources_by_id
    source_name = oke_sources_by_id.get(image_id, {}).get("source_name", "")
    minor_match_primary = k8_minor in (source_name or "")

    # fallback/detail source
    try:
        img = compute.get_image(image_id).data
    except Exception as e:
        errors.append(f"{image_id}: compute.get_image failed: {e}")
        continue

    display_name = getattr(img, "display_name", "")
    base_image_id = getattr(img, "base_image_id", None)
    freeform_tags = getattr(img, "freeform_tags", {}) or {}
    os_name = getattr(img, "operating_system", "")
    os_version = getattr(img, "operating_system_version", "")

    # Shape compatibility check for each usage
    try:
        compat_entries = _list_all(
            compute.list_image_shape_compatibility_entries, image_id=image_id
        )
        compat_shapes = {getattr(x, "shape", "") for x in compat_entries}
    except Exception as e:
        compat_shapes = set()
        errors.append(f"{image_id}: failed to read image shape compatibility: {e}")

    for u in image_usages:
        if u["shape"] not in compat_shapes:
            errors.append(
                f"{image_id}: shape incompatibility for {u['stack']}/{u['pool']} -> required shape {u['shape']} not in compatibility list."
            )

    # OKE derivation checks when image not discovered directly in OKE sources
    if not discovered_in_oke_source:
        tag_k8 = str(freeform_tags.get("k8s_version", "")).strip()
        tag_match = tag_k8 in {k8_version, k8_minor}
        name_match = _display_has_k8(display_name, k8_minor)

        parent_name_match = False
        if base_image_id:
            try:
                parent = compute.get_image(base_image_id).data
                parent_name_match = _display_has_k8(
                    getattr(parent, "display_name", ""), k8_minor
                )
            except Exception:
                parent_name_match = False

        if not (tag_match or name_match or parent_name_match):
            errors.append(
                f"{image_id}: not in OKE node pool sources and no k8s_version tag/base-image evidence for Kubernetes {k8_minor}."
            )

    # KPO + filter + Ubuntu rule (from KPO docs)
    if image_mode == "filter":
        is_ubuntu = (
            "ubuntu" in (display_name or "").lower()
            or "ubuntu" in (os_name or "").lower()
        )
        if is_ubuntu and any(u["stack"] == "kpo" for u in image_usages):
            errors.append(
                f"{image_id}: IMAGE_MODE=filter with Ubuntu for KPO is not supported; use pinned imageId path."
            )

    results.append(
        {
            "image_id": image_id,
            "discovered_in_oke_nodepool_options": discovered_in_oke_source,
            "primary_source_name": source_name,
            "k8_minor_match_in_primary_source": minor_match_primary,
            "display_name": display_name,
            "operating_system": os_name,
            "operating_system_version": os_version,
            "base_image_id": base_image_id,
            "freeform_tags": freeform_tags,
            "usages": image_usages,
        }
    )

# -----------------------------
# Persist resolved image state
# -----------------------------
# For benchmark reproducibility in pin mode, persist per-pool image IDs.
for u in usages:
    env_key = (
        f"RESOLVED_{u['stack'].upper()}_{u['pool'].upper().replace('-', '_')}_IMAGE_ID"
    )
    os.environ[env_key] = u["image_id"]

resolution_payload = {
    "generated_at_utc": datetime.now(timezone.utc).isoformat().replace("+00:00", "Z"),
    "run_id": run_id,
    "region": region,
    "kubernetes_version": k8_version,
    "k8_minor": k8_minor,
    "target_cluster": target_cluster,
    "image_mode": image_mode,
    "oke_primary_source_count": len(oke_sources_by_id),
    "validated_image_count": len(results),
    "results": results,
    "errors": errors,
}

artifact_dir = output_dir / run_id
artifact_dir.mkdir(parents=True, exist_ok=True)
resolution_path = artifact_dir / "image_selection.json"
_safe_write_json(resolution_path, resolution_payload)

print("Cell 6 image discovery summary:")
print(
    f"- Primary source images discovered (OKE node pool options): {len(oke_sources_by_id)}"
)
print(f"- Validated selected image IDs: {len(results)}")
print(f"- Artifact: {resolution_path}")

if image_mode == "pin":
    print("- IMAGE_MODE=pin: explicit image IDs are retained for reproducibility.")
else:
    print("- IMAGE_MODE=filter: drift-mode validation path executed.")

if errors:
    print("\nImage validation issues:")
    for e in errors:
        print(f"- {e}")
    raise RuntimeError(f"Cell 6 failed with {len(errors)} image validation issue(s).")

print("\nCell 6 complete: image discovery/selection validation passed.")
print("Next: Run Cell 7.")

In [ ]:
# Cell 7 — Provision selected target(s): terraform apply + output capture + kubeconfig refs

import os
import re
import json
import subprocess
from pathlib import Path
from datetime import datetime, timezone


def _env(k, d=""):
    return os.environ.get(k, d)


def _require(k):
    v = _env(k, "").strip()
    if not v:
        raise RuntimeError(f"Missing required env var: {k}")
    return v


def _find_repo_root(start: Path) -> Path:
    p = start.resolve()
    for c in [p] + list(p.parents):
        if (c / "references" / "terraform-oci-oke-main").exists():
            return c
    raise RuntimeError(
        "Could not locate repo root containing references/terraform-oci-oke-main"
    )


def _load_json(path: Path):
    return json.loads(path.read_text(encoding="utf-8"))


def _write_json(path: Path, data):
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(data, indent=2, sort_keys=True) + "\n", encoding="utf-8")


def _run(cmd, cwd=None, env=None):
    print(f"\n$ {' '.join(cmd)}")
    rc = subprocess.run(cmd, cwd=cwd, env=env, text=True).returncode
    if rc != 0:
        raise RuntimeError(f"Command failed (exit {rc}): {' '.join(cmd)}")
    return rc


def _run_json(cmd, cwd=None, env=None):
    print(f"\n$ {' '.join(cmd)}")
    p = subprocess.run(cmd, cwd=cwd, env=env, text=True, capture_output=True)
    if p.stdout:
        print(p.stdout)
    if p.stderr:
        print(p.stderr)
    if p.returncode != 0:
        raise RuntimeError(f"Command failed (exit {p.returncode}): {' '.join(cmd)}")
    try:
        return json.loads(p.stdout or "{}")
    except Exception as e:
        raise RuntimeError(
            f"Failed to parse JSON output from command: {' '.join(cmd)}; error: {e}"
        )


def _tf_out_val(tf_output_obj, name, default=None):
    x = tf_output_obj.get(name, {})
    if isinstance(x, dict) and "value" in x:
        return x["value"]
    return default


def _normalize_endpoint(endpoint: str) -> str:
    if not endpoint:
        return ""
    e = endpoint.strip()
    if not e:
        return ""
    if not e.startswith("http://") and not e.startswith("https://"):
        e = f"https://{e}"
    return e


def _extract_endpoint(cluster_endpoints, apiserver_private_host, prefer="private"):
    endpoint = ""

    if isinstance(cluster_endpoints, dict) and prefer == "private":
        for k in ("private_endpoint", "privateEndpoint", "private", "endpoint_private"):
            v = cluster_endpoints.get(k)
            if isinstance(v, str) and v.strip():
                endpoint = v.strip()
                break

    if isinstance(cluster_endpoints, dict) and prefer == "public":
        for k in ("public_endpoint", "publicEndpoint", "public", "endpoint_public"):
            v = cluster_endpoints.get(k)
            if isinstance(v, str) and v.strip():
                endpoint = v.strip()
                break

    if (
        not endpoint
        and prefer == "private"
        and isinstance(apiserver_private_host, str)
        and apiserver_private_host.strip()
    ):
        host = apiserver_private_host.strip()
        if host.startswith("http://") or host.startswith("https://"):
            endpoint = host
        else:
            endpoint = f"https://{host}:6443"

    return _normalize_endpoint(endpoint)


def _kubeconfig_server(kubeconfig_path: Path) -> str:
    text = kubeconfig_path.read_text(encoding="utf-8")
    for ln in text.splitlines():
        m = re.match(r"^\s*server:\s*(\S+)\s*$", ln)
        if m:
            return m.group(1).strip()
    return ""


# ---------------------------
# Context
# ---------------------------
repo_root = _find_repo_root(Path.cwd())
tf_root = repo_root / "tf"

target_cluster = _env("TARGET_CLUSTER", "both").strip().lower()
if target_cluster not in {"ca", "kpo", "both"}:
    raise RuntimeError(f"TARGET_CLUSTER must be ca|kpo|both, got: {target_cluster}")

selected_targets = ["ca", "kpo"] if target_cluster == "both" else [target_cluster]

run_id = _require("RUN_ID")
output_dir = Path(_require("OUTPUT_DIR"))
artifact_dir = output_dir / run_id
artifact_dir.mkdir(parents=True, exist_ok=True)

region = _require("BENCHMARK_REGION")
oci_profile = _require("OCI_PROFILE")
oci_config_file = _require("OCI_CONFIG_FILE")
kubeconfig_endpoint_mode = _env("KUBECONFIG_ENDPOINT_MODE", "auto").strip().lower()
if kubeconfig_endpoint_mode not in {"auto", "public", "private"}:
    raise RuntimeError(
        f"KUBECONFIG_ENDPOINT_MODE must be auto|public|private, got: {kubeconfig_endpoint_mode}"
    )

os.environ["OCI_CLI_CONFIG_FILE"] = oci_config_file
os.environ["OCI_CONFIG_FILE_PROFILE"] = oci_profile
os.environ["TF_VAR_config_file_profile"] = oci_profile

image_selection_path = artifact_dir / "image_selection.json"
if not image_selection_path.exists():
    raise RuntimeError(
        f"Missing image selection artifact from Cell 6: {image_selection_path}"
    )

image_selection = _load_json(image_selection_path)

print("=== Cell 7 Provision ===")
print(f"Repo root: {repo_root}")
print(f"Selected target(s): {selected_targets}")
print(f"Run artifact dir: {artifact_dir}")

kubeconfig_dir = artifact_dir / "kubeconfigs"
kubeconfig_dir.mkdir(parents=True, exist_ok=True)

target_results = {}
capacity_selection = {}

for t in selected_targets:
    stack_dir = tf_root / t
    tfvars_path = stack_dir / "terraform.auto.tfvars.json"
    tfplan_path = stack_dir / "tfplan"

    if not stack_dir.exists():
        raise RuntimeError(f"Stack dir not found: {stack_dir}")
    if not tfvars_path.exists():
        raise RuntimeError(f"Missing tfvars: {tfvars_path}")

    print(f"\n=== Applying target: {t} ({stack_dir}) ===")

    if tfplan_path.exists():
        _run(
            [
                "terraform",
                f"-chdir={stack_dir}",
                "apply",
                "-input=false",
                "-no-color",
                str(tfplan_path),
            ]
        )
        apply_mode = "tfplan"
    else:
        print(
            f"[WARN] Plan file not found for {t}: {tfplan_path}; applying with -auto-approve fallback."
        )
        _run(
            [
                "terraform",
                f"-chdir={stack_dir}",
                "apply",
                "-input=false",
                "-no-color",
                "-auto-approve",
            ]
        )
        apply_mode = "auto-approve-fallback"

    tf_outputs = _run_json(["terraform", f"-chdir={stack_dir}", "output", "-json"])
    tf_output_artifact = artifact_dir / f"{t}_terraform_output.json"
    _write_json(tf_output_artifact, tf_outputs)

    cluster_id = _tf_out_val(tf_outputs, "cluster_id", "")
    if not cluster_id:
        raise RuntimeError(f"{t}: terraform output missing cluster_id")

    cluster_endpoints = _tf_out_val(tf_outputs, "cluster_endpoints", {})
    apiserver_private_host = _tf_out_val(tf_outputs, "apiserver_private_host", "")
    worker_pool_ids = _tf_out_val(tf_outputs, "worker_pool_ids", {})
    worker_pools = _tf_out_val(tf_outputs, "worker_pools", {})
    worker_subnet_id = _tf_out_val(tf_outputs, "worker_subnet_id", "")
    pod_subnet_id = _tf_out_val(tf_outputs, "pod_subnet_id", "")
    control_plane_subnet_id = _tf_out_val(tf_outputs, "control_plane_subnet_id", "")
    int_lb_subnet_id = _tf_out_val(tf_outputs, "int_lb_subnet_id", "")
    pub_lb_subnet_id = _tf_out_val(tf_outputs, "pub_lb_subnet_id", "")
    vcn_id = _tf_out_val(tf_outputs, "vcn_id", "")

    apiserver_private_endpoint = _extract_endpoint(
        cluster_endpoints, apiserver_private_host, prefer="private"
    )
    apiserver_public_endpoint = _extract_endpoint(
        cluster_endpoints, apiserver_private_host, prefer="public"
    )

    if kubeconfig_endpoint_mode == "public":
        kube_endpoint_flag = "PUBLIC_ENDPOINT"
    elif kubeconfig_endpoint_mode == "private":
        kube_endpoint_flag = "PRIVATE_ENDPOINT"
    else:
        kube_endpoint_flag = (
            "PUBLIC_ENDPOINT" if apiserver_public_endpoint else "PRIVATE_ENDPOINT"
        )

    kubeconfig_mode_label = (
        "public" if kube_endpoint_flag == "PUBLIC_ENDPOINT" else "private"
    )
    apiserver_endpoint = (
        (apiserver_public_endpoint or apiserver_private_endpoint)
        if kubeconfig_mode_label == "public"
        else (apiserver_private_endpoint or apiserver_public_endpoint)
    )

    kubeconfig_path = kubeconfig_dir / f"{t}-{kubeconfig_mode_label}.kubeconfig"
    _run(
        [
            "oci",
            "--config-file",
            oci_config_file,
            "--profile",
            oci_profile,
            "--region",
            region,
            "ce",
            "cluster",
            "create-kubeconfig",
            "--cluster-id",
            cluster_id,
            "--file",
            str(kubeconfig_path),
            "--token-version",
            "2.0.0",
            "--kube-endpoint",
            kube_endpoint_flag,
            "--overwrite",
        ]
    )

    kubeconfig_server = _kubeconfig_server(kubeconfig_path)

    os.environ[f"{t.upper()}_CLUSTER_ID"] = cluster_id
    os.environ[f"{t.upper()}_KUBECONFIG"] = str(kubeconfig_path)
    os.environ[f"{t.upper()}_KUBECONFIG_SERVER"] = kubeconfig_server
    os.environ[f"{t.upper()}_APISERVER_ENDPOINT"] = apiserver_endpoint
    os.environ[f"{t.upper()}_APISERVER_PRIVATE_ENDPOINT"] = apiserver_private_endpoint
    os.environ[f"{t.upper()}_APISERVER_PUBLIC_ENDPOINT"] = apiserver_public_endpoint
    os.environ[f"{t.upper()}_WORKER_SUBNET_ID"] = str(worker_subnet_id or "")
    os.environ[f"{t.upper()}_POD_SUBNET_ID"] = str(pod_subnet_id or "")
    os.environ[f"{t.upper()}_VCN_ID"] = str(vcn_id or "")

    bm = _load_json(tfvars_path).get("benchmark", {})
    pools = bm.get("worker_pools", {})
    pool_capacity = {}
    for pool_name, pool_cfg in pools.items():
        if not isinstance(pool_cfg, dict):
            continue
        pool_capacity[pool_name] = {
            "shape": pool_cfg.get("shape"),
            "image_id": pool_cfg.get("image_id"),
            "size": pool_cfg.get("size"),
            "min_size": pool_cfg.get("min_size"),
            "max_size": pool_cfg.get("max_size"),
            "ocpus": pool_cfg.get("ocpus"),
            "memory": pool_cfg.get("memory"),
            "autoscale": pool_cfg.get("autoscale"),
            "allow_autoscaler": pool_cfg.get("allow_autoscaler"),
            "ignore_initial_pool_size": pool_cfg.get("ignore_initial_pool_size"),
        }

    capacity_selection[t] = {
        "kubernetes_version": bm.get("kubernetes_version"),
        "cluster_name": bm.get("cluster_name"),
        "worker_pools": pool_capacity,
    }

    target_results[t] = {
        "apply_mode": apply_mode,
        "stack_dir": str(stack_dir),
        "cluster_id": cluster_id,
        "cluster_endpoints": cluster_endpoints,
        "apiserver_private_host": apiserver_private_host,
        "apiserver_private_endpoint": apiserver_private_endpoint,
        "apiserver_public_endpoint": apiserver_public_endpoint,
        "apiserver_endpoint": apiserver_endpoint,
        "kubeconfig_endpoint": kubeconfig_mode_label,
        "kubeconfig_server": kubeconfig_server,
        "worker_pool_ids": worker_pool_ids,
        "worker_pools": worker_pools,
        "vcn_id": vcn_id,
        "control_plane_subnet_id": control_plane_subnet_id,
        "worker_subnet_id": worker_subnet_id,
        "pod_subnet_id": pod_subnet_id,
        "int_lb_subnet_id": int_lb_subnet_id,
        "pub_lb_subnet_id": pub_lb_subnet_id,
        "kubeconfig_path": str(kubeconfig_path),
        "terraform_output_artifact": str(tf_output_artifact),
    }

provision_summary = {
    "generated_at_utc": datetime.now(timezone.utc).isoformat().replace("+00:00", "Z"),
    "run_id": run_id,
    "region": region,
    "target_cluster": target_cluster,
    "selected_targets": selected_targets,
    "kubeconfig_endpoint_mode": kubeconfig_endpoint_mode,
    "image_mode": _env("IMAGE_MODE", "pin"),
    "image_selection_artifact": str(image_selection_path),
    "image_selection_validated_count": image_selection.get("validated_image_count", 0),
    "image_selection_primary_source_count": image_selection.get(
        "oke_primary_source_count", 0
    ),
    "capacity_selection": capacity_selection,
    "targets": target_results,
}

provision_path = artifact_dir / "provisioning.json"
_write_json(provision_path, provision_summary)

os.environ["PROVISIONING_ARTIFACT"] = str(provision_path)

print("\nCell 7 provisioning summary:")
for t in selected_targets:
    r = target_results[t]
    print(f"- {t}: cluster_id={r['cluster_id']}")
    print(f"  kubeconfig={r['kubeconfig_path']}")
    print(f"  kubeconfig_endpoint={r['kubeconfig_endpoint']}")
    print(f"  kubeconfig_server={r['kubeconfig_server']}")
    print(f"  apiserver_endpoint={r['apiserver_endpoint']}")
    print(f"  apiserver_public_endpoint={r['apiserver_public_endpoint']}")
    print(f"  apiserver_private_endpoint={r['apiserver_private_endpoint']}")
    print(f"  worker_subnet_id={r['worker_subnet_id']}")
    print(f"  pod_subnet_id={r['pod_subnet_id']}")

print(f"- Provisioning artifact: {provision_path}")
print("\nCell 7 complete: selected target(s) provisioned and outputs persisted.")
print("Next: Run Cell 8.")

In [ ]:
# Cell 8 — Post-Provision Autoscaler Setup and Health (CA + KPO, with IAM bootstrap)
import os
import re
import json
import subprocess
from pathlib import Path
from datetime import datetime, timezone
from urllib.parse import urlparse
import oci


def _env(k, d=""):
    return os.environ.get(k, d)


def _env_nonempty(k, d=""):
    v = os.environ.get(k, "")
    if v is None:
        return d
    s = str(v).strip()
    return s if s else d


def _require(k):
    v = _env(k, "").strip()
    if not v:
        raise RuntimeError(f"Missing required env var: {k}")
    return v


def _bool(v, default=False):
    if v is None:
        return default
    return str(v).strip().lower() in {"1", "true", "yes", "y", "on"}


def _find_repo_root(start: Path) -> Path:
    p = start.resolve()
    for c in [p] + list(p.parents):
        if (c / "references" / "terraform-oci-oke-main").exists():
            return c
    raise RuntimeError(
        "Could not locate repo root containing references/terraform-oci-oke-main"
    )


def _load_json(path: Path):
    return json.loads(path.read_text(encoding="utf-8"))


def _write_json(path: Path, data):
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(data, indent=2, sort_keys=True) + "\n", encoding="utf-8")


def _run(cmd, cwd=None, env=None, capture=False):
    print(f"\n$ {' '.join(cmd)}")
    if capture:
        p = subprocess.run(cmd, cwd=cwd, env=env, text=True, capture_output=True)
        if p.stdout:
            print(p.stdout)
        if p.stderr:
            print(p.stderr)
        if p.returncode != 0:
            raise RuntimeError(f"Command failed (exit {p.returncode}): {' '.join(cmd)}")
        return p.stdout
    rc = subprocess.run(cmd, cwd=cwd, env=env, text=True).returncode
    if rc != 0:
        raise RuntimeError(f"Command failed (exit {rc}): {' '.join(cmd)}")
    return ""


def _sanitize_name(name: str, max_len: int = 120) -> str:
    out = re.sub(r"[^A-Za-z0-9_.-]+", "-", name).strip("-")
    if len(out) > max_len:
        out = out[:max_len]
    return out or "bench-policy"


def _sanitized_env_name(env_key: str, default: str) -> str:
    return _sanitize_name(_env_nonempty(env_key, default))


def _version_tuple(v: str):
    m = re.search(r"(\d+)\.(\d+)\.(\d+)", v)
    if not m:
        return None
    return (int(m.group(1)), int(m.group(2)), int(m.group(3)))


def _parse_ip_count(ocinodeclass_text: str) -> int:
    m = re.search(r"(?m)^\s*ipCount:\s*(\d+)\s*$", ocinodeclass_text)
    if not m:
        raise RuntimeError("Could not find ipCount in rendered OCINodeClass manifest.")
    return int(m.group(1))


def _is_power_of_two(n: int) -> bool:
    return n > 0 and (n & (n - 1)) == 0


def _extract_apiserver_host(endpoint: str) -> str:
    if not endpoint:
        return ""
    s = endpoint.strip()
    if s.startswith("http://") or s.startswith("https://"):
        parsed = urlparse(s)
        return parsed.hostname or ""
    if ":" in s:
        return s.split(":")[0]
    return s


def _read_kubeconfig_server(kubeconfig_path: str) -> str:
    text = Path(kubeconfig_path).read_text(encoding="utf-8")
    for ln in text.splitlines():
        m = re.match(r"^\s*server:\s*(\S+)\s*$", ln)
        if m:
            return m.group(1).strip()
    return ""


def _ensure_dynamic_group(
    identity_client,
    compartment_id: str,
    dg_name: str,
    matching_rule: str,
    description: str,
):
    matching_rule = (matching_rule or "").strip()
    if not matching_rule:
        raise RuntimeError(
            "KPO dynamic group matching rule resolved to empty string. "
            "Set KPO_NODES_DYNAMIC_GROUP_RULE or keep it unset to use the default rule."
        )
    existing = None
    page = None
    while True:
        resp = identity_client.list_dynamic_groups(
            compartment_id=compartment_id, page=page
        )
        for dg in resp.data:
            if dg.name == dg_name:
                existing = dg
                break
        if existing or not resp.has_next_page:
            break
        page = resp.next_page
    if existing is None:
        details = oci.identity.models.CreateDynamicGroupDetails(
            compartment_id=compartment_id,
            name=dg_name,
            description=description,
            matching_rule=matching_rule,
        )
        created = identity_client.create_dynamic_group(details).data
        return {
            "action": "created",
            "id": created.id,
            "name": created.name,
            "matching_rule": created.matching_rule,
        }
    if (existing.matching_rule or "").strip() != matching_rule.strip() or (
        existing.description or ""
    ) != description:
        update = oci.identity.models.UpdateDynamicGroupDetails(
            description=description,
            matching_rule=matching_rule,
        )
        updated = identity_client.update_dynamic_group(existing.id, update).data
        return {
            "action": "updated",
            "id": updated.id,
            "name": updated.name,
            "matching_rule": updated.matching_rule,
        }
    return {
        "action": "unchanged",
        "id": existing.id,
        "name": existing.name,
        "matching_rule": existing.matching_rule,
    }


def _ensure_policy(
    identity_client,
    compartment_id: str,
    policy_name: str,
    statements: list[str],
    description: str,
):
    existing = None
    page = None
    while True:
        resp = identity_client.list_policies(compartment_id=compartment_id, page=page)
        for p in resp.data:
            if p.name == policy_name:
                existing = p
                break
        if existing or not resp.has_next_page:
            break
        page = resp.next_page
    normalized = [s.strip() for s in statements if s.strip()]
    if not normalized:
        raise RuntimeError(f"Policy {policy_name} has no statements after rendering.")
    if existing is None:
        details = oci.identity.models.CreatePolicyDetails(
            compartment_id=compartment_id,
            name=policy_name,
            description=description,
            statements=normalized,
        )
        created = identity_client.create_policy(details).data
        return {
            "action": "created",
            "id": created.id,
            "name": created.name,
            "statements": normalized,
        }
    existing_set = [s.strip() for s in (existing.statements or [])]
    if existing_set != normalized or (existing.description or "") != description:
        update = oci.identity.models.UpdatePolicyDetails(
            description=description,
            statements=normalized,
        )
        updated = identity_client.update_policy(existing.id, update).data
        return {
            "action": "updated",
            "id": updated.id,
            "name": updated.name,
            "statements": normalized,
        }
    return {
        "action": "unchanged",
        "id": existing.id,
        "name": existing.name,
        "statements": normalized,
    }


def _render_template(path: Path, replacements: dict[str, str]) -> str:
    text = path.read_text(encoding="utf-8")
    for k, v in replacements.items():
        text = text.replace(k, v)
    return text


def _policy_statements_from_text(text: str) -> list[str]:
    lines = []
    acc = []
    for raw in text.splitlines():
        s = raw.strip()
        if not s or s.startswith("#"):
            continue
        acc.append(s)
    blob = "\n".join(acc)
    blocks = []
    cur = []
    depth = 0
    for ln in blob.splitlines():
        if ln.startswith("Allow ") and not cur:
            cur = [ln]
            depth += ln.count("{") - ln.count("}")
            if depth <= 0:
                blocks.append(" ".join(cur))
                cur = []
                depth = 0
            continue
        if cur:
            cur.append(ln)
            depth += ln.count("{") - ln.count("}")
            if depth <= 0:
                blocks.append(" ".join(cur))
                cur = []
                depth = 0
    lines.extend([re.sub(r"\s+", " ", b).strip() for b in blocks if b.strip()])
    return lines


def _normalize_statements_for_policy_compartment(
    statements: list[str], policy_compartment_id: str
) -> list[str]:
    normalized = [s.strip() for s in statements if s.strip()]
    if not normalized:
        return normalized
    if policy_compartment_id.startswith("ocid1.tenancy."):
        return normalized
    rewritten = []
    rewrite_count = 0
    target = f"in compartment id {policy_compartment_id}"
    for stmt in normalized:
        updated = re.sub(r"\bin tenancy\b", target, stmt)
        if updated != stmt:
            rewrite_count += 1
        rewritten.append(updated)
    if rewrite_count:
        print(
            f"Normalized {rewrite_count} tenancy-scoped policy statement(s) "
            f"to policy compartment {policy_compartment_id}"
        )
    return rewritten


def _detect_vcn_native_cni_version(kubeconfig: str) -> str:
    out = _run(
        [
            "kubectl",
            "--kubeconfig",
            kubeconfig,
            "-n",
            "kube-system",
            "get",
            "ds",
            "-l",
            "app=vcn-native-ip-cni",
            "-o",
            "json",
        ],
        capture=True,
    )
    obj = json.loads(out or "{}")
    items = obj.get("items", [])
    if not items:
        raise RuntimeError(
            "VCN-native CNI daemonset not found with label app=vcn-native-ip-cni"
        )
    containers = (
        items[0]
        .get("spec", {})
        .get("template", {})
        .get("spec", {})
        .get("containers", [])
    )
    if not containers:
        raise RuntimeError("VCN-native CNI daemonset has no containers")
    image = containers[0].get("image", "")
    m = re.search(r":([0-9]+\.[0-9]+\.[0-9]+)", image)
    if not m:
        raise RuntimeError(f"Could not parse CNI version from image: {image}")
    return m.group(1)


def _k8s_api_reachability_check(kubeconfig: str, timeout_seconds: int = 15):
    cmd = [
        "kubectl",
        "--kubeconfig",
        kubeconfig,
        "get",
        "--raw",
        "/version",
        f"--request-timeout={timeout_seconds}s",
    ]
    p = subprocess.run(cmd, text=True, capture_output=True)
    if p.returncode == 0:
        return True, ""
    err = (p.stderr or p.stdout or "").strip()
    if len(err) > 500:
        err = err[-500:]
    return False, err or f"kubectl exited with {p.returncode}"


def _extract_addon_names(list_addons_obj: dict) -> set[str]:
    names = set()
    items = list_addons_obj.get("data", [])
    if isinstance(items, list):
        for item in items:
            if not isinstance(item, dict):
                continue
            for key in ("name", "addon-name", "addon_name"):
                val = item.get(key)
                if isinstance(val, str) and val.strip():
                    names.add(val.strip())
    return names


def _get_cluster_addon_names(
    oci_config_file: str, oci_profile: str, region: str, cluster_id: str
) -> set[str]:
    try:
        out = _run(
            [
                "oci",
                "--config-file",
                oci_config_file,
                "--profile",
                oci_profile,
                "--region",
                region,
                "ce",
                "cluster",
                "list-addons",
                "--cluster-id",
                cluster_id,
                "--all",
                "--output",
                "json",
            ],
            capture=True,
        )
        parsed = json.loads(out or "{}")
        names = _extract_addon_names(parsed)
        if names:
            return names
    except RuntimeError as e:
        print(
            f"Warning: list-addons call failed, trying get-addon fallback. Details: {e}"
        )
    try:
        out = _run(
            [
                "oci",
                "--config-file",
                oci_config_file,
                "--profile",
                oci_profile,
                "--region",
                region,
                "ce",
                "cluster",
                "get-addon",
                "--cluster-id",
                cluster_id,
                "--addon-name",
                "ClusterAutoscaler",
                "--output",
                "json",
            ],
            capture=True,
        )
        parsed = json.loads(out or "{}")
        data = parsed.get("data", {})
        addon_name = ""
        if isinstance(data, dict):
            addon_name = (
                data.get("name")
                or data.get("addon-name")
                or data.get("addon_name")
                or ""
            ).strip()
        if addon_name:
            return {addon_name}
        return {"ClusterAutoscaler"}
    except RuntimeError:
        return set()


def _resolve_kpo_chart_path(repo_root: Path, explicit_path: str) -> str:
    if explicit_path:
        p = Path(explicit_path).expanduser()
        if p.exists():
            return str(p)
        raise RuntimeError(
            f"KPO_HELM_CHART was set but path does not exist: {explicit_path}. "
            "Set KPO_HELM_CHART to a valid chart dir or .tgz file."
        )
    local_checkout_chart = (
        repo_root / "references" / "karpenter-provider-oci-main" / "chart"
    )
    if local_checkout_chart.exists():
        return str(local_checkout_chart)
    release_tgz_candidates = sorted(
        list((repo_root / "references").glob("karpenter-provider-oci*.tgz"))
        + list(repo_root.glob("karpenter-provider-oci*.tgz")),
        key=lambda x: x.stat().st_mtime,
        reverse=True,
    )
    if release_tgz_candidates:
        return str(release_tgz_candidates[0])
    raise RuntimeError(
        "Unable to resolve KPO Helm chart path. "
        "Set KPO_HELM_CHART to either: "
        "(1) local chart dir (for example references/karpenter-provider-oci-main/chart), or "
        "(2) downloaded chart tarball path from KPO releases (*.tgz)."
    )


def _instance_ocid_from_provider_id(provider_id: str) -> str:
    if not provider_id:
        return ""
    s = provider_id.strip()
    if s.startswith("oci://"):
        s = s[len("oci://") :]
    parts = [p for p in s.split("/") if p]
    if not parts:
        return ""
    if parts[-1].startswith("ocid1.instance."):
        return parts[-1]
    for p in parts:
        if p.startswith("ocid1.instance."):
            return p
    return ""


def _discover_kpo_nsg_ids_from_baseline_nodes(
    kubeconfig: str,
    worker_subnet_id: str,
    pod_subnet_id: str,
    oci_config_file: str,
    oci_profile: str,
    region: str,
):
    nodes_obj = json.loads(
        _run(
            [
                "kubectl",
                "--kubeconfig",
                kubeconfig,
                "get",
                "nodes",
                "-l",
                "benchmark.oracle.com/autoscaler=kpo,benchmark.oracle.com/role=baseline",
                "-o",
                "json",
            ],
            capture=True,
        )
        or "{}"
    )
    nodes = nodes_obj.get("items", []) or []
    if not nodes:
        raise RuntimeError(
            "Cannot discover NSGs: no baseline KPO nodes found. "
            "Ensure baseline node pool is Ready before Cell 8."
        )
    worker_nsg_id = ""
    pod_nsg_id = ""
    for node in nodes:
        provider_id = (node.get("spec", {}) or {}).get("providerID", "")
        instance_id = _instance_ocid_from_provider_id(provider_id)
        if not instance_id:
            continue
        vnics_obj = json.loads(
            _run(
                [
                    "oci",
                    "--config-file",
                    oci_config_file,
                    "--profile",
                    oci_profile,
                    "--region",
                    region,
                    "compute",
                    "instance",
                    "list-vnics",
                    "--instance-id",
                    instance_id,
                    "--all",
                    "--output",
                    "json",
                ],
                capture=True,
            )
            or "{}"
        )
        for vnic in vnics_obj.get("data", []) or []:
            subnet = vnic.get("subnet-id", "")
            nsg_ids = vnic.get("nsg-ids", []) or []
            if subnet == worker_subnet_id and nsg_ids and not worker_nsg_id:
                worker_nsg_id = nsg_ids[0]
            if subnet == pod_subnet_id and nsg_ids and not pod_nsg_id:
                pod_nsg_id = nsg_ids[0]
        if worker_nsg_id and pod_nsg_id:
            break
    if not worker_nsg_id or not pod_nsg_id:
        raise RuntimeError(
            "Failed to discover worker/pod NSG IDs from baseline nodes. "
            f"worker_nsg_id={worker_nsg_id or '<missing>'}, pod_nsg_id={pod_nsg_id or '<missing>'}"
        )
    return worker_nsg_id, pod_nsg_id


def _inject_nsgs_into_ocinodeclass(
    ocinodeclass_text: str,
    worker_subnet_id: str,
    pod_subnet_id: str,
    worker_nsg_id: str,
    pod_nsg_id: str,
) -> str:
    def _norm_subnet(v: str) -> str:
        # Accept quoted/unquoted values and trim inline comments.
        x = (v or "").split("#", 1)[0].strip()
        x = x.strip('"').strip("'").strip()
        return x

    lines = ocinodeclass_text.splitlines()
    # If already present for both target NSGs, return unchanged.
    has_worker_nsg = worker_nsg_id in ocinodeclass_text
    has_pod_nsg = pod_nsg_id in ocinodeclass_text
    injected_worker = has_worker_nsg
    injected_pod = has_pod_nsg
    out = []
    for line in lines:
        out.append(line)
        m = re.match(r"^(\s*)subnetId:\s*(.+?)\s*$", line)
        if not m:
            continue
        subnet_indent = m.group(1)
        subnet_val = _norm_subnet(m.group(2))
        # subnetId is nested under subnetConfig, so sibling keys are 2 spaces less indented.
        sibling_indent = subnet_indent[:-2] if len(subnet_indent) >= 2 else ""
        list_indent = sibling_indent + "  "
        if subnet_val == worker_subnet_id and not injected_worker:
            out.append(f"{sibling_indent}networkSecurityGroupConfigs:")
            out.append(f'{list_indent}- networkSecurityGroupId: "{worker_nsg_id}"')
            injected_worker = True
            continue
        if subnet_val == pod_subnet_id and not injected_pod:
            out.append(f"{sibling_indent}networkSecurityGroupConfigs:")
            out.append(f'{list_indent}- networkSecurityGroupId: "{pod_nsg_id}"')
            injected_pod = True
            continue
    if not injected_worker or not injected_pod:
        raise RuntimeError(
            "Failed to inject NSG blocks into OCINodeClass manifest. "
            f"worker_injected={injected_worker}, pod_injected={injected_pod}, "
            "template/subnet values may have changed unexpectedly."
        )
    return "\n".join(out) + "\n"


def _extract_nodepool_limits(nodepool_text: str):
    cpu_match = re.search(r'(?m)^\s*cpu:\s*"?([^"\n]+)"?\s*$', nodepool_text)
    mem_match = re.search(r'(?m)^\s*memory:\s*"?([^"\n]+)"?\s*$', nodepool_text)
    return (
        cpu_match.group(1).strip() if cpu_match else "",
        mem_match.group(1).strip() if mem_match else "",
    )


# ---------------------------
# Context and inputs
# ---------------------------
repo_root = _find_repo_root(Path.cwd())
tf_root = repo_root / "tf"
target_cluster = _env_nonempty("TARGET_CLUSTER", "both").strip().lower()
if target_cluster not in {"ca", "kpo", "both"}:
    raise RuntimeError(f"TARGET_CLUSTER must be ca|kpo|both, got: {target_cluster}")
selected_targets = ["ca", "kpo"] if target_cluster == "both" else [target_cluster]
run_id = _require("RUN_ID")
output_dir = Path(_require("OUTPUT_DIR"))
artifact_dir = output_dir / run_id
provisioning_path = artifact_dir / "provisioning.json"
if not provisioning_path.exists():
    raise RuntimeError(
        f"Missing provisioning artifact from Cell 7: {provisioning_path}"
    )
provisioning = _load_json(provisioning_path)
region = _require("BENCHMARK_REGION")
home_region = _require("HOME_REGION")
oci_profile = _require("OCI_PROFILE")
oci_config_file = _require("OCI_CONFIG_FILE")
compartment_id = _require("BENCHMARK_COMPARTMENT_OCID")
iam_policy_compartment_id = _require("IAM_POLICY_COMPARTMENT_OCID")
tenancy_ocid = _require("TENANCY_OCID")
ca_namespace = _env_nonempty("CA_NAMESPACE", "kube-system")
ca_service_account = _env_nonempty("CA_SERVICE_ACCOUNT", "cluster-autoscaler")
kpo_namespace = _env_nonempty("KPO_NAMESPACE", "karpenter")
kpo_service_account = _env_nonempty("KPO_SERVICE_ACCOUNT", "karpenter")
kpo_chart_path = (
    _resolve_kpo_chart_path(repo_root, _env_nonempty("KPO_HELM_CHART", "").strip())
    if "kpo" in selected_targets
    else ""
)
kpo_release_name = _env_nonempty("KPO_HELM_RELEASE", "karpenter").strip()
allow_api_unreachable = _bool(
    _env_nonempty("HEALTHCHECK_ALLOW_API_UNREACHABLE", "false")
)
k8s_api_check_timeout_seconds = int(
    _env_nonempty("K8S_API_CHECK_TIMEOUT_SECONDS", "15")
)
if "kpo" in selected_targets:
    print(f"Using KPO chart source: {kpo_chart_path}")
os.environ["OCI_CLI_CONFIG_FILE"] = oci_config_file
os.environ["OCI_CONFIG_FILE_PROFILE"] = oci_profile
cfg_home = oci.config.from_file(file_location=oci_config_file, profile_name=oci_profile)
cfg_home["region"] = home_region
identity = oci.identity.IdentityClient(cfg_home)
summary = {
    "generated_at_utc": datetime.now(timezone.utc).isoformat().replace("+00:00", "Z"),
    "run_id": run_id,
    "selected_targets": selected_targets,
    "iam": {},
    "health": {},
    "applied_manifests": {},
    "k8s_connectivity": {},
}
# ---------------------------
# Kubernetes API reachability preflight
# ---------------------------
unreachable_targets = []
for t in selected_targets:
    tgt = provisioning["targets"][t]
    kubeconfig_path = tgt["kubeconfig_path"]
    api_endpoint = tgt.get("apiserver_endpoint", "")
    public_endpoint = tgt.get("apiserver_public_endpoint", "")
    private_endpoint = tgt.get("apiserver_private_endpoint", "")
    kubeconfig_endpoint = tgt.get("kubeconfig_endpoint", "")
    kubeconfig_server = _read_kubeconfig_server(kubeconfig_path)
    reachable, error_text = _k8s_api_reachability_check(
        kubeconfig=kubeconfig_path,
        timeout_seconds=k8s_api_check_timeout_seconds,
    )
    summary["k8s_connectivity"][t] = {
        "kubeconfig_path": kubeconfig_path,
        "kubeconfig_endpoint": kubeconfig_endpoint,
        "kubeconfig_server": kubeconfig_server,
        "apiserver_endpoint": api_endpoint,
        "apiserver_public_endpoint": public_endpoint,
        "apiserver_private_endpoint": private_endpoint,
        "reachable": reachable,
        "error": error_text,
    }
    if not reachable:
        unreachable_targets.append(t)
if unreachable_targets and not allow_api_unreachable:
    details = []
    for t in unreachable_targets:
        c = summary["k8s_connectivity"][t]
        details.append(
            f"{t}: kubeconfig_endpoint={c.get('kubeconfig_endpoint','')}, "
            f"kubeconfig_server={c.get('kubeconfig_server','')}, "
            f"apiserver_endpoint={c.get('apiserver_endpoint','')}, "
            f"public={c.get('apiserver_public_endpoint','')}, "
            f"private={c.get('apiserver_private_endpoint','')}, "
            f"error={c.get('error','')}"
        )
    raise RuntimeError(
        "Kubernetes API is unreachable for selected target(s) using current kubeconfig(s). "
        "If kubeconfig endpoint is private, run from a host with VCN reachability (for example VPN/bastion/instance in VCN). "
        "If kubeconfig endpoint is public, confirm your network can reach the public API endpoint and required CIDRs are allowed. "
        "or set HEALTHCHECK_ALLOW_API_UNREACHABLE=true to run IAM/OCI-only checks and skip kubectl/helm operations.\n"
        + "\n".join(details)
    )
# ---------------------------
# IAM bootstrap from templates
# ---------------------------
if "ca" in selected_targets:
    ca_cluster_id = provisioning["targets"]["ca"]["cluster_id"]
    ca_policy_name = _sanitized_env_name(
        "CA_WORKLOAD_POLICY_NAME", f"ca-workload-{run_id[:16]}"
    )
    ca_template = tf_root / "ca" / "iam" / "ca_workload_identity.policy.tmpl"
    ca_rendered = _render_template(ca_template, {"__CA_CLUSTER_OCID__": ca_cluster_id})
    ca_statements = _normalize_statements_for_policy_compartment(
        _policy_statements_from_text(ca_rendered),
        iam_policy_compartment_id,
    )
    ca_policy = _ensure_policy(
        identity_client=identity,
        compartment_id=iam_policy_compartment_id,
        policy_name=ca_policy_name,
        statements=ca_statements,
        description=f"CA workload identity policy for run {run_id}",
    )
    summary["iam"]["ca_workload_policy"] = ca_policy
if "kpo" in selected_targets:
    kpo_cluster_id = provisioning["targets"]["kpo"]["cluster_id"]
    kpo_nodes_dynamic_group_name = _sanitized_env_name(
        "KPO_NODES_DYNAMIC_GROUP_NAME", f"kpo-nodes-{run_id[:16]}"
    )
    kpo_nodes_policy_name = _sanitized_env_name(
        "KPO_NODES_POLICY_NAME", f"kpo-cluster-join-{run_id[:16]}"
    )
    kpo_workload_policy_name = _sanitized_env_name(
        "KPO_WORKLOAD_POLICY_NAME", f"kpo-workload-{run_id[:16]}"
    )
    dg_rule = _env_nonempty(
        "KPO_NODES_DYNAMIC_GROUP_RULE",
        f"ALL {{instance.compartment.id = '{compartment_id}'}}",
    )
    dg_desc = f"KPO launched nodes for run {run_id}"
    dg = _ensure_dynamic_group(
        identity_client=identity,
        compartment_id=tenancy_ocid,
        dg_name=kpo_nodes_dynamic_group_name,
        matching_rule=dg_rule,
        description=dg_desc,
    )
    join_template = tf_root / "kpo" / "iam" / "kpo_nodes_cluster_join.policy.tmpl"
    join_rendered = _render_template(
        join_template,
        {
            "__KPO_NODES_DYNAMIC_GROUP_NAME__": kpo_nodes_dynamic_group_name,
            "__KPO_CLUSTER_OCID__": kpo_cluster_id,
        },
    )
    join_statements = _policy_statements_from_text(join_rendered)
    join_policy = _ensure_policy(
        identity_client=identity,
        compartment_id=iam_policy_compartment_id,
        policy_name=kpo_nodes_policy_name,
        statements=join_statements,
        description=f"KPO node CLUSTER_JOIN policy for run {run_id}",
    )
    workload_template = tf_root / "kpo" / "iam" / "kpo_workload_identity.policy.tmpl"
    workload_rendered = _render_template(
        workload_template, {"__KPO_CLUSTER_OCID__": kpo_cluster_id}
    )
    workload_statements = _normalize_statements_for_policy_compartment(
        _policy_statements_from_text(workload_rendered),
        iam_policy_compartment_id,
    )
    workload_policy = _ensure_policy(
        identity_client=identity,
        compartment_id=iam_policy_compartment_id,
        policy_name=kpo_workload_policy_name,
        statements=workload_statements,
        description=f"KPO workload identity policy for run {run_id}",
    )
    summary["iam"]["kpo_nodes_dynamic_group"] = dg
    summary["iam"]["kpo_nodes_join_policy"] = join_policy
    summary["iam"]["kpo_workload_policy"] = workload_policy
# ---------------------------
# CA health checks
# ---------------------------
if "ca" in selected_targets:
    ca_kubeconfig = provisioning["targets"]["ca"]["kubeconfig_path"]
    addon_names = _get_cluster_addon_names(
        oci_config_file=oci_config_file,
        oci_profile=oci_profile,
        region=region,
        cluster_id=provisioning["targets"]["ca"]["cluster_id"],
    )
    if "ClusterAutoscaler" not in addon_names:
        raise RuntimeError(
            "CA path health gate failed: ClusterAutoscaler addon not found on CA cluster"
        )
    if "ca" in unreachable_targets:
        summary["health"]["ca"] = {
            "cluster_autoscaler_addon_present": True,
            "detected_cluster_addons": sorted(addon_names),
            "k8s_api_reachable": False,
            "kubernetes_health_check_skipped": True,
            "skip_reason": "API unreachable; set up network access for selected kubeconfig endpoint.",
        }
    else:
        dep_json = _run(
            [
                "kubectl",
                "--kubeconfig",
                ca_kubeconfig,
                "-n",
                "kube-system",
                "get",
                "deploy",
                "-o",
                "json",
            ],
            capture=True,
        )
        dep_obj = json.loads(dep_json or "{}")
        ca_deps = [
            i
            for i in dep_obj.get("items", [])
            if "cluster-autoscaler" in i.get("metadata", {}).get("name", "")
        ]
        if not ca_deps:
            raise RuntimeError(
                "CA path health gate failed: no cluster-autoscaler deployment found in kube-system"
            )
        available = ca_deps[0].get("status", {}).get("availableReplicas", 0)
        if available < 1:
            raise RuntimeError(
                "CA path health gate failed: cluster-autoscaler deployment has no available replicas"
            )
        summary["health"]["ca"] = {
            "cluster_autoscaler_addon_present": True,
            "detected_cluster_addons": sorted(addon_names),
            "k8s_api_reachable": True,
            "cluster_autoscaler_available_replicas": available,
        }
# ---------------------------
# KPO install + VCN-native gates + CRDs/resources
# ---------------------------
if "kpo" in selected_targets:
    kpo_target = provisioning["targets"]["kpo"]
    kpo_kubeconfig = kpo_target["kubeconfig_path"]
    if "kpo" in unreachable_targets:
        print(
            "Skipping KPO Helm install and CRD checks because Kubernetes API is unreachable."
        )
        summary["health"]["kpo"] = {
            "k8s_api_reachable": False,
            "kpo_install_skipped": True,
            "skip_reason": "API unreachable; set up network access for selected kubeconfig endpoint.",
        }
    else:
        values_template = tf_root / "kpo" / "manifests" / "kpo-values.yaml.tmpl"
        values_text = values_template.read_text(encoding="utf-8")
        api_host = _extract_apiserver_host(
            kpo_target.get("apiserver_private_endpoint", "")
            or kpo_target.get("apiserver_endpoint", "")
        )
        if not api_host:
            raise RuntimeError(
                "KPO path: failed to derive API server host for Helm values"
            )
        values_text = values_text.replace("__APISERVER_ENDPOINT__", api_host)
        rendered_dir = artifact_dir / "rendered" / "kpo"
        rendered_dir.mkdir(parents=True, exist_ok=True)
        values_path = rendered_dir / "kpo-values.yaml"
        values_path.write_text(values_text, encoding="utf-8")
        _run(
            [
                "helm",
                "upgrade",
                "--install",
                kpo_release_name,
                kpo_chart_path,
                "--namespace",
                kpo_namespace,
                "--create-namespace",
                "--kubeconfig",
                kpo_kubeconfig,
                "-f",
                str(values_path),
            ]
        )
        _run(
            [
                "kubectl",
                "--kubeconfig",
                kpo_kubeconfig,
                "-n",
                kpo_namespace,
                "rollout",
                "status",
                "deployment",
                "-l",
                "app.kubernetes.io/name=karpenter",
                "--timeout=600s",
            ]
        )
        for crd in [
            "nodepools.karpenter.sh",
            "nodeclaims.karpenter.sh",
            "ocinodeclasses.oci.oraclecloud.com",
        ]:
            _run(["kubectl", "--kubeconfig", kpo_kubeconfig, "get", "crd", crd])
        ocinodeclass_template = tf_root / "kpo" / "manifests" / "ocinodeclass.yaml.tmpl"
        nodepool_template = tf_root / "kpo" / "manifests" / "nodepool.yaml.tmpl"
        worker_subnet_id = kpo_target.get("worker_subnet_id", "")
        pod_subnet_id = kpo_target.get("pod_subnet_id", "")
        if not worker_subnet_id:
            raise RuntimeError("KPO path: worker_subnet_id missing from Cell 7 outputs")
        if not pod_subnet_id:
            raise RuntimeError("KPO path: pod_subnet_id missing from Cell 7 outputs")
        worker_nsg_id, pod_nsg_id = _discover_kpo_nsg_ids_from_baseline_nodes(
            kubeconfig=kpo_kubeconfig,
            worker_subnet_id=worker_subnet_id,
            pod_subnet_id=pod_subnet_id,
            oci_config_file=oci_config_file,
            oci_profile=oci_profile,
            region=region,
        )
        ocinodeclass_text = _render_template(
            ocinodeclass_template,
            {
                "__WORKER_SUBNET_ID__": worker_subnet_id,
                "__POD_SUBNET_ID__": pod_subnet_id,
            },
        )
        ocinodeclass_text = _inject_nsgs_into_ocinodeclass(
            ocinodeclass_text=ocinodeclass_text,
            worker_subnet_id=worker_subnet_id,
            pod_subnet_id=pod_subnet_id,
            worker_nsg_id=worker_nsg_id,
            pod_nsg_id=pod_nsg_id,
        )
        nodepool_text = nodepool_template.read_text(encoding="utf-8")
        ocinodeclass_path = rendered_dir / "ocinodeclass.yaml"
        nodepool_path = rendered_dir / "nodepool.yaml"
        ocinodeclass_path.write_text(ocinodeclass_text, encoding="utf-8")
        nodepool_path.write_text(nodepool_text, encoding="utf-8")
        configured_cpu_limit, configured_memory_limit = _extract_nodepool_limits(
            nodepool_text
        )
        ip_count = _parse_ip_count(ocinodeclass_text)
        if not _is_power_of_two(ip_count):
            raise RuntimeError(
                f"KPO VCN-native gate failed: ipCount must be power-of-two. Got {ip_count}"
            )
        cni_ver = _detect_vcn_native_cni_version(kpo_kubeconfig)
        cni_tuple = _version_tuple(cni_ver)
        if cni_tuple is None:
            raise RuntimeError(
                f"KPO VCN-native gate failed: unable to parse OCI VCN-native CNI version ({cni_ver})"
            )
        if cni_tuple < (3, 0, 0):
            raise RuntimeError(
                f"KPO VCN-native gate failed: add-on version must be >=3.0.0 (got {cni_ver})"
            )
        if cni_tuple < (3, 2, 0) and ip_count > 16:
            raise RuntimeError(
                f"KPO VCN-native gate failed: add-on {cni_ver} requires ipCount <=16 (got {ip_count})"
            )
        _run(
            [
                "kubectl",
                "--kubeconfig",
                kpo_kubeconfig,
                "apply",
                "-f",
                str(ocinodeclass_path),
            ]
        )
        _run(
            [
                "kubectl",
                "--kubeconfig",
                kpo_kubeconfig,
                "apply",
                "-f",
                str(nodepool_path),
            ]
        )
        _run(
            [
                "kubectl",
                "--kubeconfig",
                kpo_kubeconfig,
                "get",
                "ocinodeclass",
                "benchmark-ocinodeclass",
                "-o",
                "yaml",
            ]
        )
        _run(
            [
                "kubectl",
                "--kubeconfig",
                kpo_kubeconfig,
                "get",
                "nodepool",
                "benchmark-nodepool",
                "-o",
                "yaml",
            ]
        )
        _run(["kubectl", "--kubeconfig", kpo_kubeconfig, "get", "nodeclaims", "-A"])
        summary["health"]["kpo"] = {
            "controller_namespace": kpo_namespace,
            "k8s_api_reachable": True,
            "vcn_native_cni_version": cni_ver,
            "ip_count": ip_count,
            "ip_count_power_of_two": True,
            "vcn_native_gate_passed": True,
            "worker_nsg_id": worker_nsg_id,
            "pod_nsg_id": pod_nsg_id,
            "nodepool_limits_applied": {
                "cpu": configured_cpu_limit,
                "memory": configured_memory_limit,
            },
        }
        summary["applied_manifests"]["kpo"] = {
            "helm_values": str(values_path),
            "ocinodeclass": str(ocinodeclass_path),
            "nodepool": str(nodepool_path),
        }
post_provision_path = artifact_dir / "post_provisioning.json"
_write_json(post_provision_path, summary)
os.environ["POST_PROVISIONING_ARTIFACT"] = str(post_provision_path)
print("\nCell 8 summary:")
print(json.dumps(summary, indent=2))
print(f"\nCell 8 complete. Artifact: {post_provision_path}")
print("Next: Run Cell 9.")

In [ ]:
# Cell 9 — Deploy Benchmark Workload (identical profile for CA and KPO)

import os
import json
import subprocess
from pathlib import Path
from datetime import datetime, timezone


def _env(k, d=""):
    return os.environ.get(k, d)


def _require(k):
    v = _env(k, "").strip()
    if not v:
        raise RuntimeError(f"Missing required env var: {k}")
    return v


def _run(cmd, capture=False):
    print(f"\n$ {' '.join(cmd)}")
    if capture:
        p = subprocess.run(cmd, text=True, capture_output=True)
        if p.stdout:
            print(p.stdout)
        if p.stderr:
            print(p.stderr)
        if p.returncode != 0:
            raise RuntimeError(f"Command failed (exit {p.returncode}): {' '.join(cmd)}")
        return p.stdout
    rc = subprocess.run(cmd, text=True).returncode
    if rc != 0:
        raise RuntimeError(f"Command failed (exit {rc}): {' '.join(cmd)}")
    return ""


def _run_json(cmd):
    out = _run(cmd, capture=True)
    return json.loads(out or "{}")


def _write_json(path: Path, data):
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(data, indent=2, sort_keys=True) + "\n", encoding="utf-8")


def _load_json(path: Path):
    return json.loads(path.read_text(encoding="utf-8"))


run_id = _require("RUN_ID")
output_dir = Path(_require("OUTPUT_DIR"))
artifact_dir = output_dir / run_id
provisioning_path = artifact_dir / "provisioning.json"

if not provisioning_path.exists():
    raise RuntimeError(
        f"Missing provisioning artifact from Cell 7: {provisioning_path}"
    )

provisioning = _load_json(provisioning_path)

target_cluster = _env("TARGET_CLUSTER", "both").strip().lower()
if target_cluster not in {"ca", "kpo", "both"}:
    raise RuntimeError(f"TARGET_CLUSTER must be ca|kpo|both, got: {target_cluster}")
selected_targets = ["ca", "kpo"] if target_cluster == "both" else [target_cluster]

target_scenarios = _env("TARGET_SCENARIOS", "all").strip().lower()

workload_base_replicas = int(_env("WORKLOAD_BASE_REPLICAS", "0"))
if workload_base_replicas < 0:
    raise RuntimeError("WORKLOAD_BASE_REPLICAS must be >= 0")

workload_image = _env("WORKLOAD_IMAGE", "registry.k8s.io/pause:3.9").strip()
workload_cpu_request = _env("WORKLOAD_CPU_REQUEST", "500m").strip()
workload_mem_request = _env("WORKLOAD_MEM_REQUEST", "512Mi").strip()
workload_cpu_limit = _env("WORKLOAD_CPU_LIMIT", workload_cpu_request).strip()
workload_mem_limit = _env("WORKLOAD_MEM_LIMIT", workload_mem_request).strip()

workload_name = _env("WORKLOAD_NAME", "benchmark-load").strip()
ns_prefix = _env("WORKLOAD_NAMESPACE_PREFIX", "benchmark").strip()
kpo_workload_nodepool = _env("KPO_WORKLOAD_NODEPOOL", "benchmark-nodepool").strip()
if not kpo_workload_nodepool:
    raise RuntimeError("KPO_WORKLOAD_NODEPOOL must not be empty")

render_dir = artifact_dir / "rendered" / "workload"
render_dir.mkdir(parents=True, exist_ok=True)

summary = {
    "generated_at_utc": datetime.now(timezone.utc).isoformat().replace("+00:00", "Z"),
    "run_id": run_id,
    "target_cluster": target_cluster,
    "selected_targets": selected_targets,
    "target_scenarios": target_scenarios,
    "workload_profile": {
        "name": workload_name,
        "image": workload_image,
        "base_replicas": workload_base_replicas,
        "resources": {
            "requests": {"cpu": workload_cpu_request, "memory": workload_mem_request},
            "limits": {"cpu": workload_cpu_limit, "memory": workload_mem_limit},
        },
    },
    "targets": {},
}

print("=== Cell 9 Workload Deploy ===")
print(f"run_id: {run_id}")
print(f"selected_targets: {selected_targets}")
print(f"base_replicas: {workload_base_replicas}")
print(f"image: {workload_image}")
print(f"kpo_nodepool_selector: {kpo_workload_nodepool}")

for t in selected_targets:
    if t not in provisioning.get("targets", {}):
        raise RuntimeError(f"Target {t} missing in provisioning.json")

    tgt = provisioning["targets"][t]
    kubeconfig = tgt["kubeconfig_path"]
    ns = f"{ns_prefix}-{t}"
    deploy_name = workload_name

    if t == "kpo":
        selector_items = [
            ("karpenter.sh/nodepool", kpo_workload_nodepool),
        ]
    else:
        selector_items = [
            ("benchmark.oracle.com/autoscaler", t),
            ("benchmark.oracle.com/role", "benchmark"),
        ]
    selector_dict = {k: v for k, v in selector_items}
    node_selector_yaml = "\n".join([f'        {k}: "{v}"' for k, v in selector_items])

    manifest = f"""apiVersion: v1
kind: Namespace
metadata:
  name: {ns}
---
apiVersion: apps/v1
kind: Deployment
metadata:
  name: {deploy_name}
  namespace: {ns}
  labels:
    app.kubernetes.io/name: {deploy_name}
    benchmark.oracle.com/autoscaler: \"{t}\"
spec:
  replicas: {workload_base_replicas}
  selector:
    matchLabels:
      app.kubernetes.io/name: {deploy_name}
  template:
    metadata:
      labels:
        app.kubernetes.io/name: {deploy_name}
        benchmark.oracle.com/autoscaler: \"{t}\"
    spec:
      nodeSelector:
{node_selector_yaml}
      containers:
      - name: {deploy_name}
        image: {workload_image}
        imagePullPolicy: IfNotPresent
        resources:
          requests:
            cpu: \"{workload_cpu_request}\"
            memory: \"{workload_mem_request}\"
          limits:
            cpu: \"{workload_cpu_limit}\"
            memory: \"{workload_mem_limit}\"
"""

    manifest_path = render_dir / f"{t}-workload.yaml"
    manifest_path.write_text(manifest, encoding="utf-8")

    _run(["kubectl", "--kubeconfig", kubeconfig, "apply", "-f", str(manifest_path)])

    dep_obj = _run_json(
        [
            "kubectl",
            "--kubeconfig",
            kubeconfig,
            "-n",
            ns,
            "get",
            "deploy",
            deploy_name,
            "-o",
            "json",
        ]
    )
    pods_obj = _run_json(
        [
            "kubectl",
            "--kubeconfig",
            kubeconfig,
            "-n",
            ns,
            "get",
            "pods",
            "-l",
            f"app.kubernetes.io/name={deploy_name}",
            "-o",
            "json",
        ]
    )

    pod_items = pods_obj.get("items", [])
    pending = sum(1 for p in pod_items if p.get("status", {}).get("phase") == "Pending")
    running = sum(1 for p in pod_items if p.get("status", {}).get("phase") == "Running")

    summary["targets"][t] = {
        "namespace": ns,
        "deployment": deploy_name,
        "kubeconfig": kubeconfig,
        "manifest_path": str(manifest_path),
        "replicas_spec": dep_obj.get("spec", {}).get("replicas", 0),
        "replicas_available": dep_obj.get("status", {}).get("availableReplicas", 0),
        "pod_count": len(pod_items),
        "pod_pending": pending,
        "pod_running": running,
        "node_selector": selector_dict,
    }

    os.environ[f"WORKLOAD_NAMESPACE_{t.upper()}"] = ns
    os.environ[f"WORKLOAD_DEPLOYMENT_{t.upper()}"] = deploy_name

artifact_path = artifact_dir / "workload_deployments.json"
_write_json(artifact_path, summary)
os.environ["WORKLOAD_DEPLOYMENT_ARTIFACT"] = str(artifact_path)

print("\nCell 9 summary:")
print(json.dumps(summary, indent=2))
print(f"\nCell 9 complete. Artifact: {artifact_path}")

if workload_base_replicas == 0:
    print("Note: WORKLOAD_BASE_REPLICAS=0, so no NodeClaims are expected yet.")
    print("Cell 10 should drive scenario replicas and trigger autoscaler behavior.")
print("Next: Run Cell 10.")

In [ ]:
# Cell 10 — Burst + Scale-Down only (separate artifact set)

import os
import json
import time
import subprocess
from pathlib import Path
from datetime import datetime, timezone


def _env(k, d=""):
    return os.environ.get(k, d)


def _require(k):
    v = _env(k, "").strip()
    if not v:
        raise RuntimeError(f"Missing required env var: {k}")
    return v


def _bool(v, default=False):
    if v is None:
        return default
    return str(v).strip().lower() in {"1", "true", "yes", "y", "on"}


def _now_utc():
    return datetime.now(timezone.utc).isoformat().replace("+00:00", "Z")


def _run(cmd, capture=False, check=True):
    print(f"\n$ {' '.join(cmd)}")
    if capture:
        p = subprocess.run(cmd, text=True, capture_output=True)
        if p.stdout:
            print(p.stdout)
        if p.stderr:
            print(p.stderr)
        if check and p.returncode != 0:
            raise RuntimeError(f"Command failed (exit {p.returncode}): {' '.join(cmd)}")
        return p.stdout
    rc = subprocess.run(cmd, text=True).returncode
    if check and rc != 0:
        raise RuntimeError(f"Command failed (exit {rc}): {' '.join(cmd)}")
    return ""


def _run_json(cmd):
    out = _run(cmd, capture=True)
    try:
        return json.loads(out or "{}")
    except Exception as e:
        raise RuntimeError(
            f"Failed to parse JSON output from: {' '.join(cmd)} ; error={e}"
        )


def _write_json(path: Path, data):
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(data, indent=2, sort_keys=True) + "\n", encoding="utf-8")


def _append_jsonl(path: Path, obj: dict):
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("a", encoding="utf-8") as f:
        f.write(json.dumps(obj, sort_keys=True) + "\n")


def _deployment_status(kubeconfig: str, ns: str, deploy: str):
    obj = _run_json(
        [
            "kubectl",
            "--kubeconfig",
            kubeconfig,
            "-n",
            ns,
            "get",
            "deploy",
            deploy,
            "-o",
            "json",
        ]
    )
    spec = obj.get("spec", {}) or {}
    status = obj.get("status", {}) or {}
    return {
        "spec_replicas": int(spec.get("replicas", 0) or 0),
        "available_replicas": int(status.get("availableReplicas", 0) or 0),
        "ready_replicas": int(status.get("readyReplicas", 0) or 0),
        "updated_replicas": int(status.get("updatedReplicas", 0) or 0),
        "unavailable_replicas": int(status.get("unavailableReplicas", 0) or 0),
    }


def _pod_phase_counts(kubeconfig: str, ns: str, deploy: str):
    obj = _run_json(
        [
            "kubectl",
            "--kubeconfig",
            kubeconfig,
            "-n",
            ns,
            "get",
            "pods",
            "-l",
            f"app.kubernetes.io/name={deploy}",
            "-o",
            "json",
        ]
    )
    counts = {"Pending": 0, "Running": 0, "Succeeded": 0, "Failed": 0, "Unknown": 0}
    items = obj.get("items", []) or []
    for it in items:
        phase = (it.get("status", {}) or {}).get("phase", "Unknown")
        if phase not in counts:
            phase = "Unknown"
        counts[phase] += 1
    counts["total"] = len(items)
    return counts


def _benchmark_nodes(kubeconfig: str, target: str, kpo_nodepool_name: str):
    if target == "kpo":
        label_selector = f"karpenter.sh/nodepool={kpo_nodepool_name}"
    else:
        label_selector = f"benchmark.oracle.com/autoscaler={target},benchmark.oracle.com/role=benchmark"

    obj = _run_json(
        [
            "kubectl",
            "--kubeconfig",
            kubeconfig,
            "get",
            "nodes",
            "-l",
            label_selector,
            "-o",
            "json",
        ]
    )
    items = obj.get("items", []) or []
    ready = 0
    for n in items:
        for c in (n.get("status", {}) or {}).get("conditions", []) or []:
            if c.get("type") == "Ready" and c.get("status") == "True":
                ready += 1
                break
    return {"selector": label_selector, "total": len(items), "ready": ready}


def _nodeclaims_count(kubeconfig: str):
    obj = _run_json(
        ["kubectl", "--kubeconfig", kubeconfig, "get", "nodeclaims", "-A", "-o", "json"]
    )
    return len(obj.get("items", []) or [])


def _scale_deploy(kubeconfig: str, ns: str, deploy: str, replicas: int):
    _run(
        [
            "kubectl",
            "--kubeconfig",
            kubeconfig,
            "-n",
            ns,
            "scale",
            "deploy",
            deploy,
            f"--replicas={replicas}",
        ]
    )


def _wait_loop(check_fn, timeout_s: int, poll_s: int):
    start = time.time()
    last = {}
    while time.time() - start <= timeout_s:
        ok, snapshot = check_fn()
        last = snapshot
        if ok:
            return True, int(time.time() - start), snapshot
        time.sleep(poll_s)
    return False, int(time.time() - start), last


def _execute_single_profile(
    profile_name: str,
    up_replicas: int,
    hold_seconds: int,
    down_replicas: int,
    repeats: int,
    poll_s: int,
    pending_timeout_s: int,
    ready_timeout_s: int,
    scale_down_timeout_s: int,
    selected_targets: list[str],
    workload: dict,
    kpo_nodepool_name: str,
    expected_floor: dict,
    require_kpo_claims_floor: bool,
    artifact_dir: Path,
    run_id: str,
    precheck_timeout_s: int = 900,
):
    events_jsonl = artifact_dir / f"scenario_events_{profile_name}.jsonl"
    if events_jsonl.exists():
        events_jsonl.unlink()

    summary = {
        "generated_at_utc": _now_utc(),
        "run_id": run_id,
        "profile": profile_name,
        "selected_targets": selected_targets,
        "repeats": repeats,
        "config": {
            "up_replicas": up_replicas,
            "down_replicas": down_replicas,
            "hold_seconds": hold_seconds,
            "poll_seconds": poll_s,
            "pending_timeout_seconds": pending_timeout_s,
            "ready_timeout_seconds": ready_timeout_s,
            "scale_down_timeout_seconds": scale_down_timeout_s,
            "precheck_timeout_seconds": precheck_timeout_s,
            "kpo_nodepool_selector": kpo_nodepool_name,
            "expected_floor": expected_floor,
            "require_kpo_claims_floor": require_kpo_claims_floor,
        },
        "runs": [],
    }

    def _emit(ev: dict):
        _append_jsonl(events_jsonl, ev)
        print(
            f"[event] {ev['target']} r{ev['repeat']} {profile_name} -> {ev['event']} @ {ev['time_utc']}"
        )

    def _snapshot(kubeconfig: str, ns: str, deploy: str, target: str):
        dep = _deployment_status(kubeconfig, ns, deploy)
        pods = _pod_phase_counts(kubeconfig, ns, deploy)
        nodes = _benchmark_nodes(kubeconfig, target, kpo_nodepool_name)
        nodeclaims = _nodeclaims_count(kubeconfig) if target == "kpo" else None
        return {
            "deployment": dep,
            "pods": pods,
            "benchmark_nodes": nodes,
            "nodeclaims": nodeclaims,
        }

    def _at_floor(snap: dict, target: str, floor: int, desired_replicas: int):
        dep = snap["deployment"]
        pods = snap["pods"]
        nodes = snap["benchmark_nodes"]
        nodeclaims = snap.get("nodeclaims")

        base_ok = (
            dep["spec_replicas"] == desired_replicas
            and dep["available_replicas"] <= desired_replicas
            and pods["total"] == desired_replicas
        )
        floor_ok = nodes["ready"] <= floor
        claims_ok = True
        if target == "kpo" and require_kpo_claims_floor:
            claims_ok = (nodeclaims or 0) <= floor
        return base_ok and floor_ok and claims_ok

    for repeat in range(1, repeats + 1):
        for t in selected_targets:
            tgt = workload.get("targets", {}).get(t)
            if not tgt:
                raise RuntimeError(
                    f"{t}: missing workload target in workload_deployments.json"
                )

            kubeconfig = tgt["kubeconfig"]
            ns = tgt["namespace"]
            deploy = tgt["deployment"]
            floor = int(expected_floor.get(t, 0))

            run_rec = {
                "target": t,
                "repeat": repeat,
                "profile": profile_name,
                "started_at_utc": _now_utc(),
                "up_replicas": up_replicas,
                "down_replicas": down_replicas,
                "hold_seconds": hold_seconds,
                "expected_floor": floor,
                "status": "running",
            }

            _emit(
                {
                    "run_id": run_id,
                    "target": t,
                    "repeat": repeat,
                    "scenario": profile_name,
                    "event": "precheck_start",
                    "time_utc": _now_utc(),
                    "details": {"expected_floor": floor},
                }
            )

            def _precheck():
                snap = _snapshot(kubeconfig, ns, deploy, t)
                return _at_floor(snap, t, floor, down_replicas), snap

            ok_pre, pre_elapsed, pre_snap = _wait_loop(
                _precheck, precheck_timeout_s, poll_s
            )
            if not ok_pre:
                _emit(
                    {
                        "run_id": run_id,
                        "target": t,
                        "repeat": repeat,
                        "scenario": profile_name,
                        "event": "precheck_failed",
                        "time_utc": _now_utc(),
                        "details": {"elapsed_seconds": pre_elapsed, **pre_snap},
                    }
                )
                run_rec["status"] = "failed_precheck_floor"
                run_rec["failed_stage"] = "precheck"
                run_rec["ended_at_utc"] = _now_utc()
                summary["runs"].append(run_rec)
                _emit(
                    {
                        "run_id": run_id,
                        "target": t,
                        "repeat": repeat,
                        "scenario": profile_name,
                        "event": "scenario_end",
                        "time_utc": _now_utc(),
                        "details": {"status": run_rec["status"]},
                    }
                )
                continue

            baseline_nodes = pre_snap["benchmark_nodes"]
            run_rec["baseline_benchmark_nodes"] = baseline_nodes

            _emit(
                {
                    "run_id": run_id,
                    "target": t,
                    "repeat": repeat,
                    "scenario": profile_name,
                    "event": "scenario_start",
                    "time_utc": _now_utc(),
                    "details": {
                        "up_replicas": up_replicas,
                        "down_replicas": down_replicas,
                        "baseline_benchmark_nodes_ready": baseline_nodes["ready"],
                        "benchmark_node_selector": baseline_nodes["selector"],
                        "expected_floor": floor,
                    },
                }
            )

            _scale_deploy(kubeconfig, ns, deploy, up_replicas)

            def _pending_check():
                snap = _snapshot(kubeconfig, ns, deploy, t)
                return snap["pods"]["Pending"] > 0, snap

            ok_pending, pending_elapsed, pending_snap = _wait_loop(
                _pending_check, pending_timeout_s, poll_s
            )
            if ok_pending:
                _emit(
                    {
                        "run_id": run_id,
                        "target": t,
                        "repeat": repeat,
                        "scenario": profile_name,
                        "event": "first_pod_pending",
                        "time_utc": _now_utc(),
                        "details": {"elapsed_seconds": pending_elapsed, **pending_snap},
                    }
                )

            def _capacity_check():
                snap = _snapshot(kubeconfig, ns, deploy, t)
                nodes = snap["benchmark_nodes"]
                nodeclaims = snap.get("nodeclaims")
                node_growth = nodes["ready"] > baseline_nodes["ready"]
                claims_growth = t == "kpo" and (nodeclaims or 0) > floor
                return (node_growth or claims_growth), snap

            ok_cap, cap_elapsed, cap_snap = _wait_loop(
                _capacity_check, ready_timeout_s, poll_s
            )
            if ok_cap:
                _emit(
                    {
                        "run_id": run_id,
                        "target": t,
                        "repeat": repeat,
                        "scenario": profile_name,
                        "event": "first_capacity_ready",
                        "time_utc": _now_utc(),
                        "details": {"elapsed_seconds": cap_elapsed, **cap_snap},
                    }
                )

            def _all_running_check():
                snap = _snapshot(kubeconfig, ns, deploy, t)
                dep = snap["deployment"]
                pods = snap["pods"]
                ok = (
                    dep["spec_replicas"] == up_replicas
                    and dep["available_replicas"] >= up_replicas
                    and pods["Running"] >= up_replicas
                    and pods["Pending"] == 0
                )
                return ok, snap

            ok_all, all_elapsed, all_snap = _wait_loop(
                _all_running_check, ready_timeout_s, poll_s
            )
            if ok_all:
                _emit(
                    {
                        "run_id": run_id,
                        "target": t,
                        "repeat": repeat,
                        "scenario": profile_name,
                        "event": "all_running",
                        "time_utc": _now_utc(),
                        "details": {"elapsed_seconds": all_elapsed, **all_snap},
                    }
                )
            else:
                run_rec["status"] = "failed_upscale_timeout"
                run_rec["failed_stage"] = "all_running"

            if hold_seconds > 0:
                print(f"[hold] {t} r{repeat} {profile_name}: sleeping {hold_seconds}s")
                time.sleep(hold_seconds)

            _scale_deploy(kubeconfig, ns, deploy, down_replicas)

            def _scaledown_check():
                snap = _snapshot(kubeconfig, ns, deploy, t)
                return _at_floor(snap, t, floor, down_replicas), snap

            ok_down, down_elapsed, down_snap = _wait_loop(
                _scaledown_check, scale_down_timeout_s, poll_s
            )
            if ok_down:
                _emit(
                    {
                        "run_id": run_id,
                        "target": t,
                        "repeat": repeat,
                        "scenario": profile_name,
                        "event": "scale_down_complete",
                        "time_utc": _now_utc(),
                        "details": {"elapsed_seconds": down_elapsed, **down_snap},
                    }
                )
                if run_rec["status"] == "running":
                    run_rec["status"] = "success"
            else:
                _emit(
                    {
                        "run_id": run_id,
                        "target": t,
                        "repeat": repeat,
                        "scenario": profile_name,
                        "event": "scale_down_issue",
                        "time_utc": _now_utc(),
                        "details": {"elapsed_seconds": down_elapsed, **down_snap},
                    }
                )
                if run_rec["status"] == "running":
                    run_rec["status"] = "failed_scale_down_timeout"

            run_rec["ended_at_utc"] = _now_utc()
            summary["runs"].append(run_rec)

            _emit(
                {
                    "run_id": run_id,
                    "target": t,
                    "repeat": repeat,
                    "scenario": profile_name,
                    "event": "scenario_end",
                    "time_utc": _now_utc(),
                    "details": {"status": run_rec["status"]},
                }
            )

    summary_path = artifact_dir / f"scenario_execution_{profile_name}_summary.json"
    _write_json(summary_path, summary)

    os.environ[f"SCENARIO_EVENTS_{profile_name.upper()}_ARTIFACT"] = str(events_jsonl)
    os.environ[f"SCENARIO_EXECUTION_{profile_name.upper()}_SUMMARY_ARTIFACT"] = str(
        summary_path
    )

    print(f"\nCell 10 ({profile_name}) summary:")
    print(
        json.dumps(
            {
                "run_id": run_id,
                "profile": profile_name,
                "selected_targets": selected_targets,
                "repeats": repeats,
                "event_ledger": str(events_jsonl),
                "summary_json": str(summary_path),
                "run_count": len(summary["runs"]),
            },
            indent=2,
        )
    )
    print(f"\nCell 10 ({profile_name}) complete.")
    print(f"- Event ledger: {events_jsonl}")
    print(f"- Summary: {summary_path}")

    return events_jsonl, summary_path


run_id = _require("RUN_ID")
output_dir = Path(_require("OUTPUT_DIR"))
artifact_dir = output_dir / run_id

workload_path = artifact_dir / "workload_deployments.json"
if not workload_path.exists():
    raise RuntimeError(
        f"Missing workload deployment artifact from Cell 9: {workload_path}"
    )
workload = json.loads(workload_path.read_text(encoding="utf-8"))

target_cluster = _env("TARGET_CLUSTER", "both").strip().lower()
if target_cluster not in {"ca", "kpo", "both"}:
    raise RuntimeError(f"TARGET_CLUSTER must be ca|kpo|both, got: {target_cluster}")
selected_targets = ["ca", "kpo"] if target_cluster == "both" else [target_cluster]

repeats = int(_env("REPEATS", "3"))
poll_s = int(_env("SCENARIO_POLL_SECONDS", "10"))
pending_timeout_s = int(_env("SCENARIO_PENDING_TIMEOUT_SECONDS", "300"))
ready_timeout_s = int(_env("SCENARIO_READY_TIMEOUT_SECONDS", "1800"))
scale_down_timeout_s = int(_env("SCENARIO_SCALE_DOWN_TIMEOUT_SECONDS", "1800"))
precheck_timeout_s = int(_env("SCENARIO_PRECHECK_TIMEOUT_SECONDS", "900"))

kpo_nodepool_name = _env("KPO_WORKLOAD_NODEPOOL", "benchmark-nodepool").strip()
if "kpo" in workload.get("targets", {}):
    sel = workload["targets"]["kpo"].get("node_selector") or {}
    kpo_nodepool_name = str(sel.get("karpenter.sh/nodepool", kpo_nodepool_name)).strip()
if not kpo_nodepool_name:
    raise RuntimeError("KPO nodepool selector cannot be empty")

ca_floor = int(
    _env("CA_EXPECTED_BENCHMARK_NODE_FLOOR", _env("CA_AUTOSCALED_POOL_MIN_SIZE", "0"))
)
kpo_floor = int(
    _env("KPO_EXPECTED_BENCHMARK_NODE_FLOOR", _env("KPO_AUTOSCALED_POOL_MIN_SIZE", "0"))
)
expected_floor = {"ca": ca_floor, "kpo": kpo_floor}
require_kpo_claims_floor = _bool(_env("REQUIRE_KPO_NODECLAIMS_AT_FLOOR", "true"))

burst_replicas = int(_env("SCENARIO_BURST_REPLICAS", "20"))
burst_hold_seconds = int(_env("SCENARIO_BURST_HOLD_SECONDS", "120"))
if repeats < 1:
    raise RuntimeError("REPEATS must be >= 1")
if burst_replicas < 1:
    raise RuntimeError("SCENARIO_BURST_REPLICAS must be >= 1")
if burst_hold_seconds < 0:
    raise RuntimeError("SCENARIO_BURST_HOLD_SECONDS must be >= 0")

print("=== Cell 10 Burst + Scale-Down ===")
print(f"run_id: {run_id}")
print(f"targets: {selected_targets}")
print(f"repeats: {repeats}")
print(f"kpo_nodepool_selector: {kpo_nodepool_name}")
print(f"expected_floor: {expected_floor}")
print(f"precheck_timeout_seconds: {precheck_timeout_s}")

_execute_single_profile(
    profile_name="burst",
    up_replicas=burst_replicas,
    hold_seconds=burst_hold_seconds,
    down_replicas=0,
    repeats=repeats,
    poll_s=poll_s,
    pending_timeout_s=pending_timeout_s,
    ready_timeout_s=ready_timeout_s,
    scale_down_timeout_s=scale_down_timeout_s,
    selected_targets=selected_targets,
    workload=workload,
    kpo_nodepool_name=kpo_nodepool_name,
    expected_floor=expected_floor,
    require_kpo_claims_floor=require_kpo_claims_floor,
    artifact_dir=artifact_dir,
    run_id=run_id,
    precheck_timeout_s=precheck_timeout_s,
)
print("Next: Run Cell 11 for Steady + Scale-Down.")

In [ ]:
# Cell 11 — Steady + Scale-Down only (separate artifact set)

if "_execute_single_profile" not in globals():
    raise RuntimeError(
        "Cell 10 helper not loaded. Run Cell 10 first in this kernel session."
    )

run_id = _require("RUN_ID")
output_dir = Path(_require("OUTPUT_DIR"))
artifact_dir = output_dir / run_id

workload_path = artifact_dir / "workload_deployments.json"
if not workload_path.exists():
    raise RuntimeError(
        f"Missing workload deployment artifact from Cell 9: {workload_path}"
    )
workload = json.loads(workload_path.read_text(encoding="utf-8"))

target_cluster = _env("TARGET_CLUSTER", "both").strip().lower()
if target_cluster not in {"ca", "kpo", "both"}:
    raise RuntimeError(f"TARGET_CLUSTER must be ca|kpo|both, got: {target_cluster}")
selected_targets = ["ca", "kpo"] if target_cluster == "both" else [target_cluster]

repeats = int(_env("REPEATS", "3"))
poll_s = int(_env("SCENARIO_POLL_SECONDS", "10"))
pending_timeout_s = int(_env("SCENARIO_PENDING_TIMEOUT_SECONDS", "300"))
ready_timeout_s = int(_env("SCENARIO_READY_TIMEOUT_SECONDS", "1800"))
scale_down_timeout_s = int(_env("SCENARIO_SCALE_DOWN_TIMEOUT_SECONDS", "1800"))
precheck_timeout_s = int(_env("SCENARIO_PRECHECK_TIMEOUT_SECONDS", "900"))

kpo_nodepool_name = _env("KPO_WORKLOAD_NODEPOOL", "benchmark-nodepool").strip()
if "kpo" in workload.get("targets", {}):
    sel = workload["targets"]["kpo"].get("node_selector") or {}
    kpo_nodepool_name = str(sel.get("karpenter.sh/nodepool", kpo_nodepool_name)).strip()
if not kpo_nodepool_name:
    raise RuntimeError("KPO nodepool selector cannot be empty")

ca_floor = int(
    _env("CA_EXPECTED_BENCHMARK_NODE_FLOOR", _env("CA_AUTOSCALED_POOL_MIN_SIZE", "0"))
)
kpo_floor = int(
    _env("KPO_EXPECTED_BENCHMARK_NODE_FLOOR", _env("KPO_AUTOSCALED_POOL_MIN_SIZE", "0"))
)
expected_floor = {"ca": ca_floor, "kpo": kpo_floor}
require_kpo_claims_floor = _bool(_env("REQUIRE_KPO_NODECLAIMS_AT_FLOOR", "true"))

steady_replicas = int(_env("SCENARIO_STEADY_REPLICAS", "8"))
steady_hold_seconds = int(_env("SCENARIO_STEADY_HOLD_SECONDS", "300"))

if repeats < 1:
    raise RuntimeError("REPEATS must be >= 1")
if steady_replicas < 1:
    raise RuntimeError("SCENARIO_STEADY_REPLICAS must be >= 1")
if steady_hold_seconds < 0:
    raise RuntimeError("SCENARIO_STEADY_HOLD_SECONDS must be >= 0")

print("=== Cell 11 Steady + Scale-Down ===")
print(f"run_id: {run_id}")
print(f"targets: {selected_targets}")
print(f"repeats: {repeats}")
print(f"kpo_nodepool_selector: {kpo_nodepool_name}")
print(f"expected_floor: {expected_floor}")
print(f"precheck_timeout_seconds: {precheck_timeout_s}")

_execute_single_profile(
    profile_name="steady",
    up_replicas=steady_replicas,
    hold_seconds=steady_hold_seconds,
    down_replicas=0,
    repeats=repeats,
    poll_s=poll_s,
    pending_timeout_s=pending_timeout_s,
    ready_timeout_s=ready_timeout_s,
    scale_down_timeout_s=scale_down_timeout_s,
    selected_targets=selected_targets,
    workload=workload,
    kpo_nodepool_name=kpo_nodepool_name,
    expected_floor=expected_floor,
    require_kpo_claims_floor=require_kpo_claims_floor,
    artifact_dir=artifact_dir,
    run_id=run_id,
    precheck_timeout_s=precheck_timeout_s,
)
print("Next: Run Cell 12 To Gather Telemetry.")

In [ ]:
# Cell 12 — Collect Telemetry (OCI Monitoring + Kubernetes evidence)

import os
import json
import csv
import subprocess
from pathlib import Path
from datetime import datetime, timezone


def _env(k, d=""):
    return os.environ.get(k, d)


def _now_utc():
    return datetime.now(timezone.utc).isoformat().replace("+00:00", "Z")


def _run(cmd, check=True, print_output=False):
    print(f"\n$ {' '.join(cmd)}")
    p = subprocess.run(cmd, text=True, capture_output=True)
    if print_output and p.stdout:
        print(p.stdout)
    if print_output and p.stderr:
        print(p.stderr)
    if check and p.returncode != 0:
        raise RuntimeError(
            f"Command failed (exit {p.returncode}): {' '.join(cmd)}\n{p.stderr}"
        )
    return p


def _run_json(cmd, check=True):
    p = _run(cmd, check=check, print_output=False)
    if p.returncode != 0:
        return None, p
    try:
        return json.loads(p.stdout or "{}"), p
    except Exception as e:
        raise RuntimeError(f"Failed to parse JSON from command: {' '.join(cmd)} ; {e}")


def _write_json(path: Path, obj):
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(obj, indent=2, sort_keys=True) + "\n", encoding="utf-8")


def _write_jsonl(path: Path, records):
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as f:
        for r in records:
            f.write(json.dumps(r, sort_keys=True) + "\n")


def _parse_ts(ts):
    if not ts:
        return None
    if ts.endswith("Z"):
        ts = ts[:-1] + "+00:00"
    return datetime.fromisoformat(ts)


def _first_nonempty(*vals):
    for v in vals:
        if v is None:
            continue
        sv = str(v).strip()
        if sv:
            return sv
    return ""


def _candidate_output_roots():
    roots = []
    out_env = _env("OUTPUT_DIR", "").strip()
    if out_env:
        roots.append(Path(os.path.expanduser(out_env)).resolve())

    cwd = Path.cwd().resolve()
    roots.append((cwd / "OCIK8sClusterAutoscalerVsKarpenter" / "artifacts").resolve())
    roots.append((cwd / "artifacts").resolve())

    uniq = []
    seen = set()
    for r in roots:
        k = str(r)
        if k not in seen:
            seen.add(k)
            uniq.append(r)
    return uniq


def _is_valid_run_dir(run_dir: Path):
    if not run_dir.is_dir():
        return False
    needed = [
        run_dir / "workload_deployments.json",
        run_dir / "provisioning.json",
    ]
    if not all(p.exists() for p in needed):
        return False
    # at least one scenario summary should exist
    has_summary = any(
        (run_dir / n).exists()
        for n in [
            "scenario_execution_burst_summary.json",
            "scenario_execution_steady_summary.json",
        ]
    )
    return has_summary


def _discover_run_context():
    run_id_env = _env("RUN_ID", "").strip()
    out_env = _env("OUTPUT_DIR", "").strip()
    roots = _candidate_output_roots()

    # 1) If both env vars are set and valid, use them.
    if run_id_env and out_env:
        out = Path(os.path.expanduser(out_env)).resolve()
        ad = out / run_id_env
        if _is_valid_run_dir(ad):
            return out, run_id_env, ad

    # 2) If RUN_ID is set but OUTPUT_DIR is missing/wrong, find that run_id under candidate roots.
    if run_id_env:
        for r in roots:
            ad = r / run_id_env
            if _is_valid_run_dir(ad):
                return r, run_id_env, ad

    # 3) If OUTPUT_DIR is set, scan it for latest valid run.
    if out_env:
        out = Path(os.path.expanduser(out_env)).resolve()
        if out.exists():
            runs = [p for p in out.glob("ca-vs-kpo-*") if _is_valid_run_dir(p)]
            if runs:
                runs.sort(key=lambda p: p.stat().st_mtime, reverse=True)
                ad = runs[0]
                return out, ad.name, ad

    # 4) Global fallback: scan candidate roots for latest valid run.
    candidates = []
    for r in roots:
        if not r.exists():
            continue
        for p in r.glob("ca-vs-kpo-*"):
            if _is_valid_run_dir(p):
                candidates.append((p.stat().st_mtime, r, p))
    if candidates:
        candidates.sort(key=lambda x: x[0], reverse=True)
        _, out, ad = candidates[0]
        return out, ad.name, ad

    raise RuntimeError(
        "Could not resolve RUN_ID/OUTPUT_DIR from env or artifacts. "
        "Run Cell 2 once in this kernel, then rerun Cell 12."
    )


def _collect_window_from_summaries(summaries):
    starts = []
    ends = []
    for s in summaries:
        for r in s.get("runs", []) or []:
            st = _parse_ts(str(r.get("started_at_utc", "")))
            en = _parse_ts(str(r.get("ended_at_utc", "")))
            if st:
                starts.append(st)
            if en:
                ends.append(en)
    if not starts or not ends:
        raise RuntimeError(
            "Could not derive telemetry time window from scenario summaries."
        )
    start = min(starts).isoformat().replace("+00:00", "Z")
    end = max(ends).isoformat().replace("+00:00", "Z")
    return start, end


def _infer_compartment_id(target_obj):
    cid = _first_nonempty(
        target_obj.get("compartment_id"),
        target_obj.get("compartmentId"),
    )
    if cid:
        return cid
    worker_pools = target_obj.get("worker_pools", {}) or {}
    for _, wp in worker_pools.items():
        cid = _first_nonempty(wp.get("compartment_id"), wp.get("compartmentId"))
        if cid:
            return cid
    return _first_nonempty(
        _env("BENCHMARK_COMPARTMENT_OCID", ""),
        _env("PARENT_COMPARTMENT_OCID", ""),
    )


def _metric_query(metric_name, interval, dimension_key, cluster_id, stat):
    return f'{metric_name}[{interval}]{{{dimension_key} = "{cluster_id}"}}.{stat}()'


# Resolve runtime context robustly (works even after kernel restart)
output_dir, run_id, artifact_dir = _discover_run_context()
os.environ["OUTPUT_DIR"] = str(output_dir)
os.environ["RUN_ID"] = run_id

workload_path = artifact_dir / "workload_deployments.json"
provisioning_path = artifact_dir / "provisioning.json"
workload = json.loads(workload_path.read_text(encoding="utf-8"))
provisioning = json.loads(provisioning_path.read_text(encoding="utf-8"))

burst_summary_path = artifact_dir / "scenario_execution_burst_summary.json"
steady_summary_path = artifact_dir / "scenario_execution_steady_summary.json"

scenario_summaries = []
if burst_summary_path.exists():
    scenario_summaries.append(
        json.loads(burst_summary_path.read_text(encoding="utf-8"))
    )
if steady_summary_path.exists():
    scenario_summaries.append(
        json.loads(steady_summary_path.read_text(encoding="utf-8"))
    )
if not scenario_summaries:
    raise RuntimeError(
        f"No scenario summary found. Expected one of: {burst_summary_path}, {steady_summary_path}"
    )

window_start_utc, window_end_utc = _collect_window_from_summaries(scenario_summaries)

selected_targets = []
for s in scenario_summaries:
    for t in s.get("selected_targets", []) or []:
        if t not in selected_targets:
            selected_targets.append(t)
if not selected_targets:
    selected_targets = list((workload.get("targets", {}) or {}).keys())
if not selected_targets:
    raise RuntimeError("No selected targets found in scenario/workload artifacts.")

oci_config_file = os.path.expanduser(_env("OCI_CONFIG_FILE", "~/.oci/config"))
oci_profile = _env("OCI_PROFILE", "DEFAULT").strip() or "DEFAULT"
region = _first_nonempty(
    provisioning.get("region"),
    _env("BENCHMARK_REGION", ""),
    _env("REGION", ""),
)

ca_namespace = _env("CA_NAMESPACE", "kube-system").strip() or "kube-system"
kpo_namespace = _env("KPO_NAMESPACE", "karpenter").strip() or "karpenter"

post_prov_path = artifact_dir / "post_provisioning.json"
if post_prov_path.exists():
    post_prov = json.loads(post_prov_path.read_text(encoding="utf-8"))
    kpo_namespace = _first_nonempty(
        ((post_prov.get("health") or {}).get("kpo") or {}).get("controller_namespace"),
        kpo_namespace,
    )

telemetry_dir = artifact_dir / "telemetry"
metrics_raw_dir = telemetry_dir / "oci_metrics_raw"
controller_logs_dir = telemetry_dir / "controller_logs"
kpo_snapshots_dir = telemetry_dir / "kpo_snapshots"

metrics_csv_path = telemetry_dir / "oci_metrics.csv"
k8s_events_jsonl_path = telemetry_dir / "k8s_events.jsonl"
summary_path = telemetry_dir / "telemetry_collection_summary.json"

telemetry_dir.mkdir(parents=True, exist_ok=True)
metrics_raw_dir.mkdir(parents=True, exist_ok=True)
controller_logs_dir.mkdir(parents=True, exist_ok=True)
kpo_snapshots_dir.mkdir(parents=True, exist_ok=True)

print("=== Cell 12 Collect Telemetry ===")
print(f"run_id: {run_id}")
print(f"artifact_dir: {artifact_dir}")
print(f"selected_targets: {selected_targets}")
print(f"window_start_utc: {window_start_utc}")
print(f"window_end_utc: {window_end_utc}")
print(f"region: {region}")
print(f"oci_profile: {oci_profile}")
print(f"oci_config_file: {oci_config_file}")

metric_interval = _env("TELEMETRY_METRIC_INTERVAL", "1m").strip() or "1m"
metric_resolution = _env("TELEMETRY_METRIC_RESOLUTION", "1m").strip() or "1m"

metric_specs = [
    {"name": "UnschedulablePods", "dimension_key": "resourceId", "stat": "max"},
    {"name": "NodeState", "dimension_key": "clusterId", "stat": "max"},
    {"name": "KubernetesNodeCondition", "dimension_key": "clusterId", "stat": "max"},
    {"name": "APIServerRequestCount", "dimension_key": "resourceId", "stat": "sum"},
    {"name": "APIServerResponseCount", "dimension_key": "resourceId", "stat": "sum"},
]

errors = []
metrics_rows = []
metrics_queries = []
k8s_event_rows = []
controller_log_files = []
snapshot_files = []

targets_obj = provisioning.get("targets", {}) or {}
workload_targets = workload.get("targets", {}) or {}

for target in selected_targets:
    tgt = targets_obj.get(target) or {}
    w_tgt = workload_targets.get(target) or {}

    cluster_id = _first_nonempty(tgt.get("cluster_id"))
    kubeconfig = _first_nonempty(w_tgt.get("kubeconfig"))
    compartment_id = _infer_compartment_id(tgt)

    if not cluster_id:
        errors.append(
            {
                "target": target,
                "stage": "target_init",
                "error": "Missing cluster_id in provisioning artifact.",
            }
        )
        continue

    if not compartment_id:
        errors.append(
            {
                "target": target,
                "stage": "target_init",
                "error": "Missing compartment_id (and no BENCHMARK_COMPARTMENT_OCID fallback).",
            }
        )
        continue

    # 1) OCI metrics
    for spec in metric_specs:
        metric_name = spec["name"]
        query_text = _metric_query(
            metric_name=metric_name,
            interval=metric_interval,
            dimension_key=spec["dimension_key"],
            cluster_id=cluster_id,
            stat=spec["stat"],
        )

        metrics_queries.append(
            {
                "target": target,
                "cluster_id": cluster_id,
                "compartment_id": compartment_id,
                "metric_name": metric_name,
                "query_text": query_text,
            }
        )

        cmd = [
            "oci",
            "--config-file",
            oci_config_file,
            "--profile",
            oci_profile,
        ]
        if region:
            cmd += ["--region", region]
        cmd += [
            "monitoring",
            "metric-data",
            "summarize-metrics-data",
            "--compartment-id",
            compartment_id,
            "--namespace",
            "oci_oke",
            "--query-text",
            query_text,
            "--start-time",
            window_start_utc,
            "--end-time",
            window_end_utc,
            "--resolution",
            metric_resolution,
            "--output",
            "json",
        ]

        obj, proc = _run_json(cmd, check=False)
        raw_path = metrics_raw_dir / f"{target}_{metric_name}.json"
        if obj is None:
            errors.append(
                {
                    "target": target,
                    "stage": "oci_metrics",
                    "metric": metric_name,
                    "query_text": query_text,
                    "exit_code": proc.returncode,
                    "stderr": (proc.stderr or "").strip(),
                }
            )
            _write_json(
                raw_path,
                {"error": (proc.stderr or "").strip(), "query_text": query_text},
            )
            continue

        _write_json(raw_path, obj)

        data_streams = obj.get("data", []) or []
        for stream in data_streams:
            dims = stream.get("dimensions") or {}
            datapoints = (
                stream.get("aggregated-datapoints")
                or stream.get("aggregatedDatapoints")
                or stream.get("aggregated_datapoints")
                or []
            )
            for dp in datapoints:
                metrics_rows.append(
                    {
                        "run_id": run_id,
                        "target": target,
                        "cluster_id": cluster_id,
                        "compartment_id": compartment_id,
                        "metric_name": metric_name,
                        "query_text": query_text,
                        "timestamp": dp.get("timestamp"),
                        "value": dp.get("value"),
                        "dimensions_json": json.dumps(dims, sort_keys=True),
                    }
                )

    # 2) Kubernetes events
    if kubeconfig:
        events_cmd = [
            "kubectl",
            "--kubeconfig",
            kubeconfig,
            "get",
            "events",
            "-A",
            "-o",
            "json",
        ]
        events_obj, events_proc = _run_json(events_cmd, check=False)
        per_target_events_raw = telemetry_dir / f"k8s_events_{target}.json"

        if events_obj is None:
            errors.append(
                {
                    "target": target,
                    "stage": "k8s_events",
                    "exit_code": events_proc.returncode,
                    "stderr": (events_proc.stderr or "").strip(),
                }
            )
            _write_json(
                per_target_events_raw, {"error": (events_proc.stderr or "").strip()}
            )
        else:
            _write_json(per_target_events_raw, events_obj)
            items = events_obj.get("items", []) or []
            for ev in items:
                md = ev.get("metadata", {}) or {}
                involved = ev.get("involvedObject", {}) or {}
                k8s_event_rows.append(
                    {
                        "run_id": run_id,
                        "target": target,
                        "cluster_id": cluster_id,
                        "namespace": md.get("namespace"),
                        "name": md.get("name"),
                        "type": ev.get("type"),
                        "reason": ev.get("reason"),
                        "message": ev.get("message"),
                        "count": ev.get("count"),
                        "timestamp": (
                            ev.get("eventTime")
                            or ev.get("lastTimestamp")
                            or ev.get("firstTimestamp")
                            or md.get("creationTimestamp")
                        ),
                        "involved_kind": involved.get("kind"),
                        "involved_name": involved.get("name"),
                        "involved_namespace": involved.get("namespace"),
                        "source_component": (ev.get("source", {}) or {}).get(
                            "component"
                        ),
                    }
                )

    # 3) Controller logs
    if kubeconfig:
        ns = ca_namespace if target == "ca" else kpo_namespace

        dep_cmd = [
            "kubectl",
            "--kubeconfig",
            kubeconfig,
            "-n",
            ns,
            "get",
            "deploy",
            "-o",
            "json",
        ]
        dep_obj, dep_proc = _run_json(dep_cmd, check=False)
        deploy_names = []
        if dep_obj is not None:
            deploy_names = [
                ((x.get("metadata") or {}).get("name"))
                for x in dep_obj.get("items", []) or []
            ]
            deploy_names = [d for d in deploy_names if d]
            if target == "ca":
                deploy_names = [
                    d for d in deploy_names if "autoscaler" in d
                ] or deploy_names
            else:
                deploy_names = [
                    d for d in deploy_names if "karpenter" in d
                ] or deploy_names
        else:
            errors.append(
                {
                    "target": target,
                    "stage": "controller_logs",
                    "substage": "list_deployments",
                    "namespace": ns,
                    "exit_code": dep_proc.returncode,
                    "stderr": (dep_proc.stderr or "").strip(),
                }
            )

        for dep in deploy_names:
            log_cmd = [
                "kubectl",
                "--kubeconfig",
                kubeconfig,
                "-n",
                ns,
                "logs",
                f"deploy/{dep}",
                "--all-containers=true",
                "--since-time",
                window_start_utc,
                "--timestamps=true",
            ]
            log_proc = _run(log_cmd, check=False, print_output=False)
            log_path = controller_logs_dir / f"{target}_{ns}_{dep}.log"
            log_path.write_text(
                log_proc.stdout if log_proc.stdout else (log_proc.stderr or ""),
                encoding="utf-8",
            )
            controller_log_files.append(str(log_path))

            if log_proc.returncode != 0:
                errors.append(
                    {
                        "target": target,
                        "stage": "controller_logs",
                        "substage": "fetch_logs",
                        "namespace": ns,
                        "deployment": dep,
                        "exit_code": log_proc.returncode,
                        "stderr": (log_proc.stderr or "").strip(),
                    }
                )

    # 4) KPO snapshots + CA autoscaler status configmap
    if target == "kpo" and kubeconfig:
        for res in ["nodepool", "nodeclaims", "ocinodeclass"]:
            snap_cmd = [
                "kubectl",
                "--kubeconfig",
                kubeconfig,
                "get",
                res,
                "-A",
                "-o",
                "json",
            ]
            snap_obj, snap_proc = _run_json(snap_cmd, check=False)
            snap_path = kpo_snapshots_dir / f"{res}.json"
            if snap_obj is None:
                errors.append(
                    {
                        "target": target,
                        "stage": "kpo_snapshots",
                        "resource": res,
                        "exit_code": snap_proc.returncode,
                        "stderr": (snap_proc.stderr or "").strip(),
                    }
                )
                _write_json(snap_path, {"error": (snap_proc.stderr or "").strip()})
            else:
                _write_json(snap_path, snap_obj)
                snapshot_files.append(str(snap_path))

    if target == "ca" and kubeconfig:
        cm_cmd = [
            "kubectl",
            "--kubeconfig",
            kubeconfig,
            "-n",
            ca_namespace,
            "get",
            "configmap",
            "cluster-autoscaler-status",
            "-o",
            "yaml",
        ]
        cm_proc = _run(cm_cmd, check=False, print_output=False)
        cm_path = telemetry_dir / "ca_cluster_autoscaler_status_configmap.yaml"
        cm_path.write_text(
            cm_proc.stdout if cm_proc.stdout else (cm_proc.stderr or ""),
            encoding="utf-8",
        )
        if cm_proc.returncode != 0:
            errors.append(
                {
                    "target": target,
                    "stage": "ca_status_configmap",
                    "exit_code": cm_proc.returncode,
                    "stderr": (cm_proc.stderr or "").strip(),
                }
            )

# Write metrics CSV
metric_fields = [
    "run_id",
    "target",
    "cluster_id",
    "compartment_id",
    "metric_name",
    "query_text",
    "timestamp",
    "value",
    "dimensions_json",
]
with metrics_csv_path.open("w", encoding="utf-8", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=metric_fields)
    writer.writeheader()
    for row in metrics_rows:
        writer.writerow(row)

# Write K8s events JSONL
_write_jsonl(k8s_events_jsonl_path, k8s_event_rows)

queries_path = telemetry_dir / "oci_metrics_queries.json"
_write_json(queries_path, metrics_queries)

summary = {
    "generated_at_utc": _now_utc(),
    "run_id": run_id,
    "selected_targets": selected_targets,
    "window_start_utc": window_start_utc,
    "window_end_utc": window_end_utc,
    "region": region,
    "artifacts": {
        "metrics_csv": str(metrics_csv_path),
        "metrics_queries": str(queries_path),
        "metrics_raw_dir": str(metrics_raw_dir),
        "k8s_events_jsonl": str(k8s_events_jsonl_path),
        "controller_logs_dir": str(controller_logs_dir),
        "kpo_snapshots_dir": str(kpo_snapshots_dir),
    },
    "counts": {
        "metrics_rows": len(metrics_rows),
        "metrics_queries": len(metrics_queries),
        "k8s_events": len(k8s_event_rows),
        "controller_log_files": len(controller_log_files),
        "snapshot_files": len(snapshot_files),
        "errors": len(errors),
    },
    "errors": errors,
}
_write_json(summary_path, summary)

os.environ["TELEMETRY_DIR"] = str(telemetry_dir)
os.environ["OCI_METRICS_CSV_ARTIFACT"] = str(metrics_csv_path)
os.environ["K8S_EVENTS_ARTIFACT"] = str(k8s_events_jsonl_path)
os.environ["CONTROLLER_LOGS_DIR"] = str(controller_logs_dir)
os.environ["TELEMETRY_SUMMARY_ARTIFACT"] = str(summary_path)

print("\nCell 12 summary:")
print(json.dumps(summary, indent=2))
print(f"\nCell 12 complete. Artifact: {summary_path}")
print("Next: Run Cell 13.")

In [ ]:
# Cell 13 — KPI Computation

import os
import json
import csv
import re
from pathlib import Path
from datetime import datetime, timezone
from statistics import mean


def _env(k, d=""):
    return os.environ.get(k, d)


def _now_utc():
    return datetime.now(timezone.utc).isoformat().replace("+00:00", "Z")


def _parse_ts(ts):
    if not ts:
        return None
    s = str(ts)
    if s.endswith("Z"):
        s = s[:-1] + "+00:00"
    try:
        return datetime.fromisoformat(s)
    except Exception:
        return None


def _first_nonempty(*vals):
    for v in vals:
        if v is None:
            continue
        s = str(v).strip()
        if s:
            return s
    return ""


def _candidate_output_roots():
    roots = []
    out_env = _env("OUTPUT_DIR", "").strip()
    if out_env:
        roots.append(Path(os.path.expanduser(out_env)).resolve())

    cwd = Path.cwd().resolve()
    roots.append((cwd / "OCIK8sClusterAutoscalerVsKarpenter" / "artifacts").resolve())
    roots.append((cwd / "artifacts").resolve())

    uniq = []
    seen = set()
    for r in roots:
        k = str(r)
        if k not in seen:
            seen.add(k)
            uniq.append(r)
    return uniq


def _is_valid_run_dir(run_dir: Path):
    if not run_dir.is_dir():
        return False
    must_have = [
        run_dir / "scenario_execution_burst_summary.json",
        run_dir / "scenario_execution_steady_summary.json",
        run_dir / "telemetry" / "telemetry_collection_summary.json",
        run_dir / "telemetry" / "oci_metrics.csv",
    ]
    return all(p.exists() for p in must_have)


def _discover_run_context():
    run_id_env = _env("RUN_ID", "").strip()
    out_env = _env("OUTPUT_DIR", "").strip()
    roots = _candidate_output_roots()

    if run_id_env and out_env:
        out = Path(os.path.expanduser(out_env)).resolve()
        ad = out / run_id_env
        if _is_valid_run_dir(ad):
            return out, run_id_env, ad

    if run_id_env:
        for r in roots:
            ad = r / run_id_env
            if _is_valid_run_dir(ad):
                return r, run_id_env, ad

    if out_env:
        out = Path(os.path.expanduser(out_env)).resolve()
        if out.exists():
            runs = [p for p in out.glob("ca-vs-kpo-*") if _is_valid_run_dir(p)]
            if runs:
                runs.sort(key=lambda p: p.stat().st_mtime, reverse=True)
                ad = runs[0]
                return out, ad.name, ad

    candidates = []
    for r in roots:
        if not r.exists():
            continue
        for p in r.glob("ca-vs-kpo-*"):
            if _is_valid_run_dir(p):
                candidates.append((p.stat().st_mtime, r, p))

    if candidates:
        candidates.sort(key=lambda x: x[0], reverse=True)
        _, out, ad = candidates[0]
        return out, ad.name, ad

    raise RuntimeError(
        "Could not resolve run context for Cell 13. "
        "Run Cell 2 in this kernel, then rerun Cell 13."
    )


def _percentile(values, p):
    if not values:
        return None
    vals = sorted(values)
    if len(vals) == 1:
        return float(vals[0])
    idx = (len(vals) - 1) * p
    lo = int(idx)
    hi = min(lo + 1, len(vals) - 1)
    frac = idx - lo
    return float(vals[lo] * (1 - frac) + vals[hi] * frac)


def _summary_stats(values):
    if not values:
        return {
            "count": 0,
            "mean": None,
            "p50": None,
            "p90": None,
            "min": None,
            "max": None,
        }
    return {
        "count": len(values),
        "mean": float(mean(values)),
        "p50": _percentile(values, 0.50),
        "p90": _percentile(values, 0.90),
        "min": float(min(values)),
        "max": float(max(values)),
    }


def _read_json(path: Path):
    return json.loads(path.read_text(encoding="utf-8"))


def _read_jsonl(path: Path):
    rows = []
    if not path.exists():
        return rows
    for line in path.read_text(encoding="utf-8").splitlines():
        s = line.strip()
        if not s:
            continue
        rows.append(json.loads(s))
    return rows


def _build_event_index(events):
    idx = {}
    for e in events:
        k = (
            e.get("scenario"),
            e.get("target"),
            int(e.get("repeat", 0)),
            e.get("event"),
        )
        idx[k] = e
    return idx


def _to_seconds(a, b):
    da = _parse_ts(a)
    db = _parse_ts(b)
    if not da or not db:
        return None
    return float((db - da).total_seconds())


def _controller_counts(log_path: Path):
    warning = 0
    error = 0
    info = 0
    patt_glog = re.compile(r"\s([IWEF])\d{4}\s")

    with log_path.open("r", encoding="utf-8", errors="ignore") as f:
        for line in f:
            # JSON logs (Karpenter)
            if '"level":"ERROR"' in line:
                error += 1
                continue
            if '"level":"WARN"' in line or '"level":"WARNING"' in line:
                warning += 1
                continue
            if '"level":"INFO"' in line:
                info += 1
                continue

            # glog style (Cluster Autoscaler / addon logs)
            m = patt_glog.search(line)
            if m:
                lvl = m.group(1)
                if lvl == "W":
                    warning += 1
                elif lvl in {"E", "F"}:
                    error += 1
                elif lvl == "I":
                    info += 1

    return {"info": info, "warning": warning, "error": error}


# Resolve run context robustly
output_dir, run_id, artifact_dir = _discover_run_context()
os.environ["OUTPUT_DIR"] = str(output_dir)
os.environ["RUN_ID"] = run_id

print("=== Cell 13 KPI Computation ===")
print(f"run_id: {run_id}")
print(f"artifact_dir: {artifact_dir}")

# Inputs
burst_summary = _read_json(artifact_dir / "scenario_execution_burst_summary.json")
steady_summary = _read_json(artifact_dir / "scenario_execution_steady_summary.json")
burst_events = _read_jsonl(artifact_dir / "scenario_events_burst.jsonl")
steady_events = _read_jsonl(artifact_dir / "scenario_events_steady.jsonl")
event_index = _build_event_index(burst_events + steady_events)

# 1) Per-run KPI extraction
run_records = []
for summary in [burst_summary, steady_summary]:
    profile = summary.get("profile")
    for r in summary.get("runs", []):
        target = r.get("target")
        repeat = int(r.get("repeat", 0))

        e_pending = event_index.get((profile, target, repeat, "first_pod_pending"))
        e_cap = event_index.get((profile, target, repeat, "first_capacity_ready"))
        e_all = event_index.get((profile, target, repeat, "all_running"))
        e_down = event_index.get((profile, target, repeat, "scale_down_complete"))

        pending_to_first_capacity = None
        pending_to_all_running = None
        scale_down_completion = None

        if e_pending and e_cap:
            pending_to_first_capacity = _to_seconds(
                e_pending.get("time_utc"), e_cap.get("time_utc")
            )
        if e_pending and e_all:
            pending_to_all_running = _to_seconds(
                e_pending.get("time_utc"), e_all.get("time_utc")
            )
        if e_down:
            try:
                scale_down_completion = float(
                    (e_down.get("details") or {}).get("elapsed_seconds")
                )
            except Exception:
                scale_down_completion = None

        run_records.append(
            {
                "run_id": run_id,
                "profile": profile,
                "target": target,
                "repeat": repeat,
                "status": r.get("status"),
                "pending_to_first_capacity_seconds": pending_to_first_capacity,
                "pending_to_all_running_seconds": pending_to_all_running,
                "scale_down_completion_seconds": scale_down_completion,
            }
        )

# 2) Aggregate scenario KPIs by target/profile
agg_by_target_profile = {}
for rec in run_records:
    key = (rec["target"], rec["profile"])
    agg_by_target_profile.setdefault(
        key,
        {
            "pending_to_first_capacity_seconds": [],
            "pending_to_all_running_seconds": [],
            "scale_down_completion_seconds": [],
            "statuses": [],
        },
    )
    s = agg_by_target_profile[key]
    s["statuses"].append(rec.get("status"))
    for k in [
        "pending_to_first_capacity_seconds",
        "pending_to_all_running_seconds",
        "scale_down_completion_seconds",
    ]:
        v = rec.get(k)
        if v is not None:
            s[k].append(float(v))

agg_list = []
for (target, profile), vals in sorted(agg_by_target_profile.items()):
    agg_list.append(
        {
            "target": target,
            "profile": profile,
            "run_count": len(vals["statuses"]),
            "success_count": sum(1 for x in vals["statuses"] if x == "success"),
            "pending_to_first_capacity_seconds": _summary_stats(
                vals["pending_to_first_capacity_seconds"]
            ),
            "pending_to_all_running_seconds": _summary_stats(
                vals["pending_to_all_running_seconds"]
            ),
            "scale_down_completion_seconds": _summary_stats(
                vals["scale_down_completion_seconds"]
            ),
        }
    )

# 3) Unschedulable KPIs from oci_metrics.csv
metrics_csv = artifact_dir / "telemetry" / "oci_metrics.csv"
unsched_by_target = {}
metrics_row_count = 0

with metrics_csv.open("r", encoding="utf-8", newline="") as f:
    r = csv.DictReader(f)
    for row in r:
        metrics_row_count += 1
        if row.get("metric_name") != "UnschedulablePods":
            continue

        target = row.get("target")
        try:
            value = float(row.get("value") or 0.0)
        except Exception:
            value = 0.0
        ts = row.get("timestamp")

        b = unsched_by_target.setdefault(
            target,
            {
                "datapoints_total": 0,
                "positive_datapoints": 0,
                "max_value": 0.0,
                "positive_timestamps": set(),
            },
        )
        b["datapoints_total"] += 1
        if value > 0:
            b["positive_datapoints"] += 1
            b["positive_timestamps"].add(ts)
            if value > b["max_value"]:
                b["max_value"] = value

unsched_summary = []
for target, data in sorted(unsched_by_target.items()):
    unsched_summary.append(
        {
            "target": target,
            "datapoints_total": data["datapoints_total"],
            "unschedulable_datapoints": data["positive_datapoints"],
            "unschedulable_duration_minutes": len(data["positive_timestamps"]),
            "unschedulable_max_pods": data["max_value"],
        }
    )

# 4) Controller warning/error counts from collected logs
controller_dir = artifact_dir / "telemetry" / "controller_logs"
controller_by_target = {}
controller_by_file = []

for log_path in sorted(controller_dir.glob("*.log")):
    counts = _controller_counts(log_path)
    fname = log_path.name
    target = fname.split("_", 1)[0] if "_" in fname else "unknown"

    rec = {
        "file": str(log_path),
        "target": target,
        **counts,
    }
    controller_by_file.append(rec)

    agg = controller_by_target.setdefault(
        target, {"info": 0, "warning": 0, "error": 0, "files": 0}
    )
    agg["info"] += counts["info"]
    agg["warning"] += counts["warning"]
    agg["error"] += counts["error"]
    agg["files"] += 1

# 5) Kubernetes warning/error event counts (optional, may be zero)
k8s_event_counts = {}
for target in ["ca", "kpo"]:
    p = artifact_dir / "telemetry" / f"k8s_events_{target}.json"
    if not p.exists():
        continue
    obj = _read_json(p)
    items = obj.get("items", []) or []
    warn = 0
    normal = 0
    for it in items:
        t = (it.get("type") or "").strip().lower()
        if t == "warning":
            warn += 1
        elif t == "normal":
            normal += 1
    k8s_event_counts[target] = {
        "total": len(items),
        "warning": warn,
        "normal": normal,
    }

# Outputs
kpi_dir = artifact_dir / "kpi"
kpi_dir.mkdir(parents=True, exist_ok=True)

per_run_csv = kpi_dir / "kpi_per_run.csv"
with per_run_csv.open("w", encoding="utf-8", newline="") as f:
    fields = [
        "run_id",
        "profile",
        "target",
        "repeat",
        "status",
        "pending_to_first_capacity_seconds",
        "pending_to_all_running_seconds",
        "scale_down_completion_seconds",
    ]
    w = csv.DictWriter(f, fieldnames=fields)
    w.writeheader()
    for r in run_records:
        w.writerow(r)

kpi_summary_path = kpi_dir / "kpi_summary.json"
kpi_summary = {
    "generated_at_utc": _now_utc(),
    "run_id": run_id,
    "inputs": {
        "artifact_dir": str(artifact_dir),
        "burst_summary": str(artifact_dir / "scenario_execution_burst_summary.json"),
        "steady_summary": str(artifact_dir / "scenario_execution_steady_summary.json"),
        "burst_events": str(artifact_dir / "scenario_events_burst.jsonl"),
        "steady_events": str(artifact_dir / "scenario_events_steady.jsonl"),
        "telemetry_metrics_csv": str(metrics_csv),
        "controller_logs_dir": str(controller_dir),
    },
    "per_run_kpis": run_records,
    "aggregates": {
        "by_target_profile": agg_list,
        "unschedulable": unsched_summary,
        "controller_log_severity_by_target": controller_by_target,
        "controller_log_severity_by_file": controller_by_file,
        "k8s_event_counts": k8s_event_counts,
    },
    "data_quality": {
        "metrics_rows": metrics_row_count,
        "k8s_events_available": any(
            (v.get("total", 0) > 0 for v in k8s_event_counts.values())
        ),
        "notes": [
            "pending-to-* latencies are computed from event timestamps",
            "scale-down completion latency is taken from scale_down_complete.details.elapsed_seconds",
            "unschedulable duration is computed as count of 1-minute datapoints where UnschedulablePods > 0",
        ],
    },
    "artifacts": {
        "kpi_per_run_csv": str(per_run_csv),
        "kpi_summary_json": str(kpi_summary_path),
    },
}
kpi_summary_path.write_text(
    json.dumps(kpi_summary, indent=2, sort_keys=True) + "\n", encoding="utf-8"
)

os.environ["KPI_DIR"] = str(kpi_dir)
os.environ["KPI_PER_RUN_ARTIFACT"] = str(per_run_csv)
os.environ["KPI_SUMMARY_ARTIFACT"] = str(kpi_summary_path)

print("\nCell 13 summary:")
print(
    json.dumps(
        {
            "run_id": run_id,
            "per_run_records": len(run_records),
            "aggregate_groups": len(agg_list),
            "kpi_summary": str(kpi_summary_path),
            "kpi_per_run_csv": str(per_run_csv),
        },
        indent=2,
    )
)
print(f"\nCell 13 complete. Artifact: {kpi_summary_path}")
print("Next: Run Cell 14.")

In [ ]:
# Cell 14 — Reporting (Detailed)

import os
import json
import csv
from pathlib import Path
from datetime import datetime, timezone
from collections import defaultdict


def _env(k, d=""):
    return os.environ.get(k, d)


def _now_utc():
    return datetime.now(timezone.utc).isoformat().replace("+00:00", "Z")


def _first_nonempty(*vals):
    for v in vals:
        if v is None:
            continue
        s = str(v).strip()
        if s:
            return s
    return ""


def _parse_ts(ts):
    if not ts:
        return None
    s = str(ts)
    if s.endswith("Z"):
        s = s[:-1] + "+00:00"
    try:
        return datetime.fromisoformat(s)
    except Exception:
        return None


def _fmt_ts(dt):
    if not dt:
        return ""
    return dt.isoformat().replace("+00:00", "Z")


def _delta_seconds(t1, t2):
    if not t1 or not t2:
        return None
    return round((t2 - t1).total_seconds(), 6)


def _candidate_output_roots():
    roots = []
    out_env = _env("OUTPUT_DIR", "").strip()
    if out_env:
        roots.append(Path(os.path.expanduser(out_env)).resolve())

    cwd = Path.cwd().resolve()
    roots.append((cwd / "OCIK8sClusterAutoscalerVsKarpenter" / "artifacts").resolve())
    roots.append((cwd / "artifacts").resolve())

    uniq = []
    seen = set()
    for r in roots:
        k = str(r)
        if k not in seen:
            seen.add(k)
            uniq.append(r)
    return uniq


def _is_valid_run_dir(run_dir: Path):
    if not run_dir.is_dir():
        return False
    required = [
        run_dir / "scenario_execution_burst_summary.json",
        run_dir / "scenario_execution_steady_summary.json",
        run_dir / "telemetry" / "oci_metrics.csv",
        run_dir / "kpi" / "kpi_summary.json",
    ]
    return all(p.exists() for p in required)


def _discover_run_context():
    run_id_env = _env("RUN_ID", "").strip()
    out_env = _env("OUTPUT_DIR", "").strip()
    roots = _candidate_output_roots()

    if run_id_env and out_env:
        out = Path(os.path.expanduser(out_env)).resolve()
        ad = out / run_id_env
        if _is_valid_run_dir(ad):
            return out, run_id_env, ad

    if run_id_env:
        for r in roots:
            ad = r / run_id_env
            if _is_valid_run_dir(ad):
                return r, run_id_env, ad

    if out_env:
        out = Path(os.path.expanduser(out_env)).resolve()
        if out.exists():
            runs = [p for p in out.glob("ca-vs-kpo-*") if _is_valid_run_dir(p)]
            if runs:
                runs.sort(key=lambda p: p.stat().st_mtime, reverse=True)
                ad = runs[0]
                return out, ad.name, ad

    candidates = []
    for r in roots:
        if not r.exists():
            continue
        for p in r.glob("ca-vs-kpo-*"):
            if _is_valid_run_dir(p):
                candidates.append((p.stat().st_mtime, r, p))

    if candidates:
        candidates.sort(key=lambda x: x[0], reverse=True)
        _, out, ad = candidates[0]
        return out, ad.name, ad

    raise RuntimeError("Could not resolve run context for Cell 14.")


def _json(path: Path, default=None):
    if not path.exists():
        return default
    return json.loads(path.read_text(encoding="utf-8"))


def _jsonl(path: Path):
    if not path.exists():
        return []
    rows = []
    for line in path.read_text(encoding="utf-8").splitlines():
        line = line.strip()
        if not line:
            continue
        try:
            rows.append(json.loads(line))
        except Exception:
            continue
    return rows


def _mean(values):
    vals = [float(v) for v in values if v is not None]
    if not vals:
        return None
    return sum(vals) / len(vals)


def _fmt_sec(v):
    if v is None:
        return "n/a"
    try:
        return f"{float(v):.1f}s"
    except Exception:
        return "n/a"


def _to_num(v):
    try:
        if v is None or v == "":
            return None
        return float(v)
    except Exception:
        return None


output_dir, run_id, artifact_dir = _discover_run_context()
os.environ["OUTPUT_DIR"] = str(output_dir)
os.environ["RUN_ID"] = run_id

print("=== Cell 14 Reporting (Detailed) ===")
print(f"run_id: {run_id}")
print(f"artifact_dir: {artifact_dir}")

provisioning = _json(artifact_dir / "provisioning.json", {})
post_prov = _json(artifact_dir / "post_provisioning.json", {})
generation_summary = _json(Path("tf") / "generation-summary.json", {})
telemetry_summary = _json(
    artifact_dir / "telemetry" / "telemetry_collection_summary.json", {}
)
kpi_summary = _json(artifact_dir / "kpi" / "kpi_summary.json", {})

burst_summary = _json(artifact_dir / "scenario_execution_burst_summary.json", {})
steady_summary = _json(artifact_dir / "scenario_execution_steady_summary.json", {})

burst_events = _jsonl(artifact_dir / "scenario_events_burst.jsonl")
steady_events = _jsonl(artifact_dir / "scenario_events_steady.jsonl")
all_events = burst_events + steady_events

selected_targets = provisioning.get("selected_targets") or ["ca", "kpo"]

run_events = defaultdict(dict)
for rec in all_events:
    k = (
        str(rec.get("scenario", "")),
        str(rec.get("target", "")),
        int(rec.get("repeat", 0) or 0),
    )
    run_events[k][str(rec.get("event", ""))] = rec

scenario_runs = []
for profile, summary in (("burst", burst_summary), ("steady", steady_summary)):
    for r in summary.get("runs", []) or []:
        rr = dict(r)
        rr["profile"] = profile
        scenario_runs.append(rr)

timeline_rows = []
for r in scenario_runs:
    profile = str(r.get("profile", ""))
    target = str(r.get("target", ""))
    repeat = int(r.get("repeat", 0) or 0)
    key = (profile, target, repeat)
    ev = run_events.get(key, {})

    ts_start = _parse_ts(
        r.get("started_at_utc") or (ev.get("scenario_start") or {}).get("time_utc")
    )
    ts_pending = _parse_ts((ev.get("first_pod_pending") or {}).get("time_utc"))
    ts_capacity = _parse_ts((ev.get("first_capacity_ready") or {}).get("time_utc"))
    ts_all_running = _parse_ts((ev.get("all_running") or {}).get("time_utc"))
    ts_scale_down = _parse_ts((ev.get("scale_down_complete") or {}).get("time_utc"))
    ts_end = _parse_ts(
        r.get("ended_at_utc") or (ev.get("scenario_end") or {}).get("time_utc")
    )

    pending_details = (ev.get("first_pod_pending") or {}).get("details") or {}
    capacity_details = (ev.get("first_capacity_ready") or {}).get("details") or {}
    all_running_details = (ev.get("all_running") or {}).get("details") or {}
    scale_down_details = (ev.get("scale_down_complete") or {}).get("details") or {}

    row = {
        "run_id": run_id,
        "profile": profile,
        "target": target,
        "repeat": repeat,
        "status": r.get("status"),
        "up_replicas": r.get("up_replicas"),
        "down_replicas": r.get("down_replicas"),
        "hold_seconds": r.get("hold_seconds"),
        "started_at_utc": _fmt_ts(ts_start),
        "first_pod_pending_utc": _fmt_ts(ts_pending),
        "first_capacity_ready_utc": _fmt_ts(ts_capacity),
        "all_running_utc": _fmt_ts(ts_all_running),
        "scale_down_complete_utc": _fmt_ts(ts_scale_down),
        "ended_at_utc": _fmt_ts(ts_end),
        "pending_to_first_capacity_seconds": _delta_seconds(ts_pending, ts_capacity),
        "pending_to_all_running_seconds": _delta_seconds(ts_pending, ts_all_running),
        "scenario_start_to_all_running_seconds": _delta_seconds(
            ts_start, ts_all_running
        ),
        "all_running_to_scale_down_complete_seconds": _delta_seconds(
            ts_all_running, ts_scale_down
        ),
        "scenario_total_seconds": _delta_seconds(ts_start, ts_end),
        "reported_scale_down_elapsed_seconds": _to_num(
            scale_down_details.get("elapsed_seconds")
        ),
        "first_pending_pods_total": ((pending_details.get("pods") or {}).get("total")),
        "first_pending_pods_pending": (
            (pending_details.get("pods") or {}).get("Pending")
        ),
        "first_capacity_nodes_ready": (
            (capacity_details.get("benchmark_nodes") or {}).get("ready")
        ),
        "first_capacity_nodeclaims": capacity_details.get("nodeclaims"),
        "all_running_pods_running": (
            (all_running_details.get("pods") or {}).get("Running")
        ),
        "all_running_nodes_ready": (
            (all_running_details.get("benchmark_nodes") or {}).get("ready")
        ),
    }
    timeline_rows.append(row)

execution_matrix = []
for profile, summary in (("burst", burst_summary), ("steady", steady_summary)):
    cfg = summary.get("config", {}) or {}
    for target in ["ca", "kpo"]:
        runs = [r for r in (summary.get("runs", []) or []) if r.get("target") == target]
        if not runs:
            continue
        execution_matrix.append(
            {
                "profile": profile,
                "target": target,
                "repeats": len(runs),
                "up_replicas": cfg.get("up_replicas"),
                "down_replicas": cfg.get("down_replicas"),
                "hold_seconds": cfg.get("hold_seconds"),
                "pending_timeout_seconds": cfg.get("pending_timeout_seconds"),
                "ready_timeout_seconds": cfg.get("ready_timeout_seconds"),
                "scale_down_timeout_seconds": cfg.get("scale_down_timeout_seconds"),
            }
        )

pods_requested_total = sum(int(r.get("up_replicas") or 0) for r in timeline_rows)
pods_requested_by_group = defaultdict(int)
for r in timeline_rows:
    pods_requested_by_group[(r["profile"], r["target"])] += int(
        r.get("up_replicas") or 0
    )

scenario_durations = [
    r.get("scenario_total_seconds")
    for r in timeline_rows
    if r.get("scenario_total_seconds") is not None
]

all_starts = [
    _parse_ts(r.get("started_at_utc")) for r in timeline_rows if r.get("started_at_utc")
]
all_ends = [
    _parse_ts(r.get("ended_at_utc")) for r in timeline_rows if r.get("ended_at_utc")
]
window = {
    "start_utc": _fmt_ts(min(all_starts)) if all_starts else None,
    "end_utc": _fmt_ts(max(all_ends)) if all_ends else None,
}

compartment_id = ""
targets_obj = provisioning.get("targets", {}) or {}
for t in ["ca", "kpo"]:
    tp = targets_obj.get(t, {}) or {}
    pools = tp.get("worker_pools", {}) or {}
    for _, p in pools.items():
        cid = _first_nonempty(p.get("compartment_id"), p.get("compartmentId"))
        if cid:
            compartment_id = cid
            break
    if compartment_id:
        break

region_context = {
    "region": _first_nonempty(
        provisioning.get("region"),
        generation_summary.get("region"),
        telemetry_summary.get("region"),
    ),
    "benchmark_compartment_id": _first_nonempty(
        compartment_id, _env("BENCHMARK_COMPARTMENT_OCID", "")
    ),
    "target_cluster": _first_nonempty(
        provisioning.get("target_cluster"), generation_summary.get("target_cluster")
    ),
    "selected_targets": selected_targets,
}

capacity_selection = provisioning.get("capacity_selection", {}) or {}
selected_capacity = {}
for t, tcfg in capacity_selection.items():
    selected_capacity[t] = {
        "cluster_name": tcfg.get("cluster_name"),
        "kubernetes_version": tcfg.get("kubernetes_version"),
        "worker_pools": tcfg.get("worker_pools", {}),
    }

ca_addons = ((post_prov.get("health") or {}).get("ca") or {}).get(
    "detected_cluster_addons", []
)
kpo_health = (post_prov.get("health") or {}).get("kpo") or {}
versions = {
    "kubernetes": {
        "ca": (capacity_selection.get("ca") or {}).get("kubernetes_version"),
        "kpo": (capacity_selection.get("kpo") or {}).get("kubernetes_version"),
    },
    "addons": {
        "ca_detected_addons": ca_addons,
        "ca_cluster_autoscaler_available_replicas": (
            (post_prov.get("health") or {}).get("ca") or {}
        ).get("cluster_autoscaler_available_replicas"),
        "kpo_vcn_native_cni_version": kpo_health.get("vcn_native_cni_version"),
        "metrics_server_enabled_at_generation": generation_summary.get(
            "enable_metrics_server"
        ),
    },
    "controllers": {
        "ca_namespace": _env("CA_NAMESPACE", "kube-system"),
        "kpo_namespace": _first_nonempty(
            kpo_health.get("controller_namespace"), _env("KPO_NAMESPACE", "karpenter")
        ),
    },
}

scenario_parameters = {
    "burst": {
        "config": burst_summary.get("config", {}),
        "repeats": burst_summary.get("repeats"),
    },
    "steady": {
        "config": steady_summary.get("config", {}),
        "repeats": steady_summary.get("repeats"),
    },
    "window": window,
}

cluster_identifiers = {}
for t in ["ca", "kpo"]:
    tp = targets_obj.get(t, {}) or {}
    cluster_identifiers[t] = {
        "cluster_id": tp.get("cluster_id"),
        "vcn_id": tp.get("vcn_id"),
        "apiserver_public_endpoint": tp.get("apiserver_public_endpoint"),
        "apiserver_private_endpoint": tp.get("apiserver_private_endpoint"),
        "kubeconfig_path": tp.get("kubeconfig_path"),
        "worker_pool_ids": tp.get("worker_pool_ids", {}),
    }

reporting_dir = artifact_dir / "reporting"
reporting_dir.mkdir(parents=True, exist_ok=True)

artifacts = {
    "run_manifest_json": str(artifact_dir / "run_manifest.json"),
    "scenario_events_burst_jsonl": str(artifact_dir / "scenario_events_burst.jsonl"),
    "scenario_events_steady_jsonl": str(artifact_dir / "scenario_events_steady.jsonl"),
    "oci_metrics_csv": str(artifact_dir / "telemetry" / "oci_metrics.csv"),
    "k8s_events_jsonl": str(artifact_dir / "telemetry" / "k8s_events.jsonl"),
    "controller_logs_dir": str(artifact_dir / "telemetry" / "controller_logs"),
    "kpi_summary_json": str(artifact_dir / "kpi" / "kpi_summary.json"),
    "comparison_report_md": str(artifact_dir / "comparison_report.md"),
    "execution_timeline_csv": str(reporting_dir / "scenario_timeline.csv"),
    "execution_matrix_csv": str(reporting_dir / "execution_matrix.csv"),
    "timeline_summary_json": str(reporting_dir / "timeline_summary.json"),
}

run_manifest = {
    "generated_at_utc": _now_utc(),
    "run_id": run_id,
    "region_context": region_context,
    "selected_shapes_sizes_images_flex": selected_capacity,
    "resolved_images": {
        "image_mode": _first_nonempty(
            provisioning.get("image_mode"), generation_summary.get("image_mode")
        ),
        "image_selection_artifact": provisioning.get("image_selection_artifact"),
    },
    "versions": versions,
    "scenario_parameters_and_timestamps": scenario_parameters,
    "cluster_and_nodepool_identifiers": cluster_identifiers,
    "execution_overview": {
        "planned_run_count": len(timeline_rows),
        "successful_run_count": len(
            [r for r in timeline_rows if str(r.get("status")) == "success"]
        ),
        "pods_requested_total": pods_requested_total,
        "window_start_utc": window.get("start_utc"),
        "window_end_utc": window.get("end_utc"),
    },
    "artifacts": artifacts,
}
(artifact_dir / "run_manifest.json").write_text(
    json.dumps(run_manifest, indent=2, sort_keys=True) + "\n", encoding="utf-8"
)

if timeline_rows:
    with (reporting_dir / "scenario_timeline.csv").open(
        "w", encoding="utf-8", newline=""
    ) as f:
        w = csv.DictWriter(f, fieldnames=list(timeline_rows[0].keys()))
        w.writeheader()
        w.writerows(timeline_rows)

if execution_matrix:
    with (reporting_dir / "execution_matrix.csv").open(
        "w", encoding="utf-8", newline=""
    ) as f:
        w = csv.DictWriter(f, fieldnames=list(execution_matrix[0].keys()))
        w.writeheader()
        w.writerows(execution_matrix)

k8s_events_available = (kpi_summary.get("data_quality") or {}).get(
    "k8s_events_available"
)
controller = (
    (kpi_summary.get("aggregates") or {}).get("controller_log_severity_by_target")
) or {}
agg = ((kpi_summary.get("aggregates") or {}).get("by_target_profile")) or []
agg_map = {(g.get("target"), g.get("profile")): g for g in agg}


def _kpi_mean(target, profile, key):
    g = agg_map.get((target, profile), {})
    return (g.get(key) or {}).get("mean")


timeline_summary = {
    "generated_at_utc": _now_utc(),
    "run_id": run_id,
    "execution_matrix": execution_matrix,
    "timeline_rows": timeline_rows,
    "totals": {
        "runs": len(timeline_rows),
        "success_runs": len(
            [r for r in timeline_rows if str(r.get("status")) == "success"]
        ),
        "pods_requested_total": pods_requested_total,
        "pods_requested_by_profile_target": [
            {"profile": k[0], "target": k[1], "pods_requested": v}
            for k, v in sorted(pods_requested_by_group.items())
        ],
        "scenario_total_seconds_mean": _mean(scenario_durations),
    },
    "data_quality": {
        "k8s_events_available": k8s_events_available,
        "telemetry_metrics_rows": (
            (telemetry_summary.get("counts") or {}).get("metrics_rows")
        ),
        "controller_log_files": (
            (telemetry_summary.get("counts") or {}).get("controller_log_files")
        ),
        "notes": [
            "KPO pending->first-capacity in this harness reflects NodeClaim growth signal.",
            "k8s events were empty in this run; event-derived pod lifecycle metrics are not available post-run.",
        ],
    },
}
(reporting_dir / "timeline_summary.json").write_text(
    json.dumps(timeline_summary, indent=2, sort_keys=True) + "\n", encoding="utf-8"
)

lines = []
lines.append("# CA vs KPO Comparison Report")
lines.append("")
lines.append(f"Run ID: `{run_id}`")
lines.append(f"Generated: `{_now_utc()}`")
lines.append("")
lines.append("## Scope")
lines.append(f"- Targets: `{', '.join(selected_targets)}`")
lines.append(f"- Region: `{region_context['region']}`")
lines.append(
    f"- Scenario window: `{window.get('start_utc')}` to `{window.get('end_utc')}`"
)
lines.append("")

lines.append("## What Was Executed")
lines.append(
    "| Profile | Target | Repeats | Up Replicas | Down Replicas | Hold (s) | Pending Timeout (s) | Ready Timeout (s) | ScaleDown Timeout (s) |"
)
lines.append("|---|---|---:|---:|---:|---:|---:|---:|---:|")
for m in execution_matrix:
    lines.append(
        f"| {m['profile']} | {m['target']} | {m['repeats']} | {m['up_replicas']} | {m['down_replicas']} | {m['hold_seconds']} | {m['pending_timeout_seconds']} | {m['ready_timeout_seconds']} | {m['scale_down_timeout_seconds']} |"
    )
lines.append("")

lines.append("## Workload Volume")
lines.append(f"- Total scenario runs: `{len(timeline_rows)}`")
lines.append(
    f"- Successful runs: `{len([r for r in timeline_rows if str(r.get('status')) == 'success'])}`"
)
lines.append(f"- Total requested pods across all runs: `{pods_requested_total}`")
for k, v in sorted(pods_requested_by_group.items()):
    lines.append(f"- `{k[0]} / {k[1]}` requested pods: `{v}`")
lines.append("")

lines.append("## Per-Run Timeline")
lines.append(
    "| Profile | Target | Repeat | Scenario Start | First Pending | First Capacity | All Running | Scale Down Complete | End |"
)
lines.append("|---|---|---:|---|---|---|---|---|---|")
for r in sorted(
    timeline_rows,
    key=lambda x: (
        x.get("started_at_utc") or "",
        x.get("profile") or "",
        x.get("target") or "",
        x.get("repeat") or 0,
    ),
):
    lines.append(
        f"| {r['profile']} | {r['target']} | {r['repeat']} | {r['started_at_utc'] or 'n/a'} | {r['first_pod_pending_utc'] or 'n/a'} | {r['first_capacity_ready_utc'] or 'n/a'} | {r['all_running_utc'] or 'n/a'} | {r['scale_down_complete_utc'] or 'n/a'} | {r['ended_at_utc'] or 'n/a'} |"
    )
lines.append("")

lines.append("## Per-Run Latency Detail")
lines.append(
    "| Profile | Target | Repeat | Pods (Up) | Pending->First Capacity | Pending->All Running | Start->All Running | AllRunning->ScaleDownComplete | Scenario Total | Reported ScaleDown Elapsed |"
)
lines.append("|---|---|---:|---:|---:|---:|---:|---:|---:|---:|")
for r in sorted(
    timeline_rows,
    key=lambda x: (x.get("profile") or "", x.get("target") or "", x.get("repeat") or 0),
):
    lines.append(
        f"| {r['profile']} | {r['target']} | {r['repeat']} | {r.get('up_replicas', 'n/a')} | {_fmt_sec(r.get('pending_to_first_capacity_seconds'))} | {_fmt_sec(r.get('pending_to_all_running_seconds'))} | {_fmt_sec(r.get('scenario_start_to_all_running_seconds'))} | {_fmt_sec(r.get('all_running_to_scale_down_complete_seconds'))} | {_fmt_sec(r.get('scenario_total_seconds'))} | {_fmt_sec(r.get('reported_scale_down_elapsed_seconds'))} |"
    )
lines.append("")

lines.append("## KPI Aggregate Summary")
lines.append(
    "| Profile | CA Pending->First Capacity | KPO Pending->First Capacity | CA Pending->All Running | KPO Pending->All Running | CA Scale-Down Complete | KPO Scale-Down Complete |"
)
lines.append("|---|---:|---:|---:|---:|---:|---:|")
for profile in ["burst", "steady"]:
    lines.append(
        "| "
        + profile
        + f" | {_fmt_sec(_kpi_mean('ca', profile, 'pending_to_first_capacity_seconds'))}"
        + f" | {_fmt_sec(_kpi_mean('kpo', profile, 'pending_to_first_capacity_seconds'))}"
        + f" | {_fmt_sec(_kpi_mean('ca', profile, 'pending_to_all_running_seconds'))}"
        + f" | {_fmt_sec(_kpi_mean('kpo', profile, 'pending_to_all_running_seconds'))}"
        + f" | {_fmt_sec(_kpi_mean('ca', profile, 'scale_down_completion_seconds'))}"
        + f" | {_fmt_sec(_kpi_mean('kpo', profile, 'scale_down_completion_seconds'))} |"
    )
lines.append("")

lines.append("## Controller Log Severity")
lines.append("| Target | Info | Warning | Error | Log Files |")
lines.append("|---|---:|---:|---:|---:|")
for t in ["ca", "kpo"]:
    c = controller.get(t, {})
    lines.append(
        f"| {t} | {c.get('info', 0)} | {c.get('warning', 0)} | {c.get('error', 0)} | {c.get('files', 0)} |"
    )
lines.append("")

lines.append("## Data Quality and Caveats")
lines.append(f"- Kubernetes events available: `{k8s_events_available}`")
if not k8s_events_available:
    lines.append(
        "- `k8s_events.jsonl` is empty for this run, so event-derived pod lifecycle KPIs are unavailable post-run."
    )
lines.append(
    "- KPO pending->first-capacity uses NodeClaim growth signal in this harness."
)
lines.append(
    "- For per-pod lifecycle (Scheduled/ContainerStarted/Ready), collect live pod watch data during scenario execution (next run enhancement)."
)
lines.append("")

lines.append("## Artifact Index")
for k, v in artifacts.items():
    lines.append(f"- `{k}`: `{v}`")

report_path = artifact_dir / "comparison_report.md"
report_path.write_text("\n".join(lines) + "\n", encoding="utf-8")

report_summary = {
    "generated_at_utc": _now_utc(),
    "run_id": run_id,
    "run_manifest": str(artifact_dir / "run_manifest.json"),
    "comparison_report": str(report_path),
    "reporting_dir": str(reporting_dir),
    "artifacts": artifacts,
}
(artifact_dir / "reporting_summary.json").write_text(
    json.dumps(report_summary, indent=2, sort_keys=True) + "\n", encoding="utf-8"
)

os.environ["RUN_MANIFEST_ARTIFACT"] = str(artifact_dir / "run_manifest.json")
os.environ["COMPARISON_REPORT_ARTIFACT"] = str(report_path)
os.environ["REPORTING_SUMMARY_ARTIFACT"] = str(artifact_dir / "reporting_summary.json")

print("\nCell 14 summary:")
print(json.dumps(report_summary, indent=2))
print(f"\nCell 14 complete. Artifact: {report_path}")
print("Next: Run Cell 15.")

In [ ]:
# Cell 15 — Cleanup

import os
import json
import re
import time
import subprocess
from pathlib import Path
from datetime import datetime, timezone


# -------------------------------
# Helpers
# -------------------------------


def _env(k, d=""):
    return os.environ.get(k, d)


def _now_utc():
    return datetime.now(timezone.utc).isoformat().replace("+00:00", "Z")


def _parse_bool(v, default=False):
    if v is None:
        return default
    s = str(v).strip().lower()
    if s in {"1", "true", "yes", "y", "on"}:
        return True
    if s in {"0", "false", "no", "n", "off"}:
        return False
    return default


def _first_nonempty(*vals):
    for v in vals:
        if v is None:
            continue
        s = str(v).strip()
        if s:
            return s
    return ""


def _json(path: Path, default=None):
    if not path.exists():
        return default
    return json.loads(path.read_text(encoding="utf-8"))


def _candidate_output_roots():
    roots = []
    out_env = _env("OUTPUT_DIR", "").strip()
    if out_env:
        roots.append(Path(os.path.expanduser(out_env)).resolve())

    cwd = Path.cwd().resolve()
    roots.append((cwd / "OCIK8sClusterAutoscalerVsKarpenter" / "artifacts").resolve())
    roots.append((cwd / "artifacts").resolve())

    uniq = []
    seen = set()
    for r in roots:
        k = str(r)
        if k not in seen:
            seen.add(k)
            uniq.append(r)
    return uniq


def _is_valid_run_dir(run_dir: Path):
    if not run_dir.is_dir():
        return False
    required = [
        run_dir / "provisioning.json",
        run_dir / "post_provisioning.json",
        run_dir / "scenario_execution_burst_summary.json",
        run_dir / "scenario_execution_steady_summary.json",
    ]
    return all(p.exists() for p in required)


def _discover_run_context():
    run_id_env = _env("RUN_ID", "").strip()
    out_env = _env("OUTPUT_DIR", "").strip()
    roots = _candidate_output_roots()

    if run_id_env and out_env:
        out = Path(os.path.expanduser(out_env)).resolve()
        ad = out / run_id_env
        if _is_valid_run_dir(ad):
            return out, run_id_env, ad

    if run_id_env:
        for r in roots:
            ad = r / run_id_env
            if _is_valid_run_dir(ad):
                return r, run_id_env, ad

    if out_env:
        out = Path(os.path.expanduser(out_env)).resolve()
        if out.exists():
            runs = [p for p in out.glob("ca-vs-kpo-*") if _is_valid_run_dir(p)]
            if runs:
                runs.sort(key=lambda p: p.stat().st_mtime, reverse=True)
                ad = runs[0]
                return out, ad.name, ad

    candidates = []
    for r in roots:
        if not r.exists():
            continue
        for p in r.glob("ca-vs-kpo-*"):
            if _is_valid_run_dir(p):
                candidates.append((p.stat().st_mtime, r, p))

    if candidates:
        candidates.sort(key=lambda x: x[0], reverse=True)
        _, out, ad = candidates[0]
        return out, ad.name, ad

    raise RuntimeError("Could not resolve run context for Cell 15.")


def _run(cmd, timeout=None):
    p = subprocess.run(cmd, capture_output=True, text=True, timeout=timeout)
    return {
        "cmd": cmd,
        "rc": p.returncode,
        "stdout": p.stdout,
        "stderr": p.stderr,
    }


def _run_json(cmd, timeout=None):
    r = _run(cmd, timeout=timeout)
    if r["rc"] != 0:
        return r, None
    txt = (r.get("stdout") or "").strip()
    if not txt:
        return r, None
    try:
        return r, json.loads(txt)
    except Exception:
        return r, None


def _extract_manifest_name(path: Path):
    if not path.exists():
        return ""
    txt = path.read_text(encoding="utf-8")
    m = re.search(r"(?m)^metadata:\s*$[\s\S]*?^\s*name:\s*([A-Za-z0-9._-]+)\s*$", txt)
    if m:
        return m.group(1)
    m2 = re.search(r"(?m)^\s*name:\s*([A-Za-z0-9._-]+)\s*$", txt)
    return m2.group(1) if m2 else ""


def _kubectl_json(kubeconfig, args):
    cmd = ["kubectl", "--kubeconfig", str(kubeconfig)] + args + ["-o", "json"]
    r, j = _run_json(cmd)
    return r, j


def _wait_until(fn, timeout_seconds, poll_seconds=10):
    started = time.time()
    history = []
    while True:
        ok, payload = fn()
        history.append({"time_utc": _now_utc(), "ok": bool(ok), "payload": payload})
        if ok:
            return True, history
        if time.time() - started >= timeout_seconds:
            return False, history
        time.sleep(poll_seconds)


def _is_not_found(stderr_or_stdout):
    s = (stderr_or_stdout or "").lower()
    needles = [
        "notauthorizedornotfound",
        "not found",
        "404",
        "entity does not exist",
        "no such",
    ]
    return any(n in s for n in needles)


# -------------------------------
# Context discovery
# -------------------------------
output_dir, run_id, artifact_dir = _discover_run_context()
os.environ["OUTPUT_DIR"] = str(output_dir)
os.environ["RUN_ID"] = run_id

provisioning = _json(artifact_dir / "provisioning.json", {})
post_prov = _json(artifact_dir / "post_provisioning.json", {})
workload_deployments = _json(artifact_dir / "workload_deployments.json", {})
generation_summary = _json(Path("tf") / "generation-summary.json", {})

selected_targets = provisioning.get("selected_targets") or ["ca", "kpo"]

region = _first_nonempty(
    _env("OCI_REGION", ""),
    provisioning.get("region"),
    generation_summary.get("region"),
    "us-ashburn-1",
)
oci_profile = _first_nonempty(_env("OCI_PROFILE", ""), "DEFAULT")
oci_config_file = str(
    Path(
        os.path.expanduser(
            _first_nonempty(_env("OCI_CONFIG_FILE", "~/.oci/config"), "~/.oci/config")
        )
    ).resolve()
)

cleanup_timeout_nodeclaims = int(
    _first_nonempty(_env("CLEANUP_NODECLAIMS_TIMEOUT_SECONDS", "1800"), "1800")
)
cleanup_timeout_nodes = int(
    _first_nonempty(_env("CLEANUP_NODES_TIMEOUT_SECONDS", "1200"), "1200")
)
poll_seconds = int(_first_nonempty(_env("CLEANUP_POLL_SECONDS", "10"), "10"))

dry_run = _parse_bool(_env("CLEANUP_DRY_RUN", "false"), default=False)
skip_terraform_destroy = _parse_bool(
    _env("CLEANUP_SKIP_TERRAFORM_DESTROY", "false"), default=False
)

print("=== Cell 15 Cleanup ===")
print(f"run_id: {run_id}")
print(f"artifact_dir: {artifact_dir}")
print(f"selected_targets: {selected_targets}")
print(f"region: {region}")
print(f"oci_profile: {oci_profile}")
print(f"oci_config_file: {oci_config_file}")
print(f"dry_run: {dry_run}")
print(f"skip_terraform_destroy: {skip_terraform_destroy}")

# -------------------------------
# Prepare per-target runtime context
# -------------------------------
kubeconfigs = {}
stack_dirs = {}
cluster_ids = {}
worker_pool_ids = {}

for t in ["ca", "kpo"]:
    tp = (provisioning.get("targets") or {}).get(t, {}) or {}
    wd = (workload_deployments.get("targets") or {}).get(t, {}) or {}

    kubeconfigs[t] = _first_nonempty(
        wd.get("kubeconfig"),
        tp.get("kubeconfig_path"),
    )

    stack_dirs[t] = _first_nonempty(
        tp.get("stack_dir"),
        generation_summary.get(f"{t}_stack_dir"),
        str((Path("tf") / t).resolve()),
    )

    cluster_ids[t] = _first_nonempty(tp.get("cluster_id"))
    worker_pool_ids[t] = list(((tp.get("worker_pool_ids") or {}).values()))


# -------------------------------
# Cleanup execution log
# -------------------------------
cleanup_actions = []
errors = []


def _record(action, data):
    cleanup_actions.append(
        {
            "time_utc": _now_utc(),
            "action": action,
            "data": data,
        }
    )


# -------------------------------
# KPO pre-destroy cleanup: NodePool -> NodeClaims -> Nodes -> OCINodeClass
# -------------------------------
kpo_cleanup = {
    "executed": False,
    "kubeconfig": kubeconfigs.get("kpo"),
    "nodepool_candidates": [],
    "nodepools_deleted": [],
    "nodeclaims_wait": {},
    "nodes_wait": {},
    "ocinodeclass_candidates": [],
    "ocinodeclasses_deleted": [],
}

if "kpo" in selected_targets:
    kpo_cleanup["executed"] = True
    kpo_kubeconfig = kubeconfigs.get("kpo")
    if not kpo_kubeconfig:
        msg = "Missing KPO kubeconfig path in artifacts/provisioning outputs."
        errors.append(msg)
        _record("kpo_cleanup_skipped", {"reason": msg})
    else:
        np_candidates = set()
        nc_candidates = set()

        # Candidate from workload selector
        kpo_node_selector = (
            (workload_deployments.get("targets") or {}).get("kpo") or {}
        ).get("node_selector") or {}
        if kpo_node_selector.get("karpenter.sh/nodepool"):
            np_candidates.add(kpo_node_selector.get("karpenter.sh/nodepool"))

        # Candidate names from rendered manifests
        applied = (post_prov.get("applied_manifests") or {}).get("kpo") or {}
        nodepool_manifest = (
            Path(applied.get("nodepool")) if applied.get("nodepool") else None
        )
        ocinodeclass_manifest = (
            Path(applied.get("ocinodeclass")) if applied.get("ocinodeclass") else None
        )
        if nodepool_manifest:
            nm = _extract_manifest_name(nodepool_manifest)
            if nm:
                np_candidates.add(nm)
        if ocinodeclass_manifest:
            cm = _extract_manifest_name(ocinodeclass_manifest)
            if cm:
                nc_candidates.add(cm)

        # Discover live NodePools and OCINodeClass refs
        r_np, j_np = _kubectl_json(kpo_kubeconfig, ["get", "nodepool", "-A"])
        if r_np["rc"] == 0 and isinstance(j_np, dict):
            for item in j_np.get("items", []) or []:
                md = item.get("metadata", {}) or {}
                name = md.get("name")
                ns = md.get("namespace") or "default"
                ann = md.get("annotations", {}) or {}
                tmpl_labels = (
                    ((item.get("spec") or {}).get("template") or {}).get("metadata")
                    or {}
                ).get("labels", {}) or {}
                ncr = (
                    ((item.get("spec") or {}).get("template") or {}).get("spec") or {}
                ).get("nodeClassRef") or {}
                ncr_name = ncr.get("name")
                if ncr_name:
                    nc_candidates.add(ncr_name)

                is_benchmark = bool(
                    (name in np_candidates)
                    or (ann.get("benchmark.oracle.com/min-size") is not None)
                    or (tmpl_labels.get("benchmark.oracle.com/autoscaler") == "kpo")
                )
                if is_benchmark and name:
                    kpo_cleanup["nodepool_candidates"].append(
                        {"namespace": ns, "name": name}
                    )
        else:
            _record("kpo_get_nodepools_failed", r_np)

        # Fallback: if not discovered but we have name candidate, assume default ns
        if not kpo_cleanup["nodepool_candidates"]:
            for n in sorted(np_candidates):
                if n:
                    kpo_cleanup["nodepool_candidates"].append(
                        {"namespace": "default", "name": n}
                    )

        kpo_cleanup["ocinodeclass_candidates"] = sorted(x for x in nc_candidates if x)

        # Delete NodePools first
        for np in kpo_cleanup["nodepool_candidates"]:
            cmd = [
                "kubectl",
                "--kubeconfig",
                str(kpo_kubeconfig),
                "delete",
                "nodepool",
                np["name"],
                "-n",
                np["namespace"],
                "--ignore-not-found=true",
            ]
            if dry_run:
                _record("dry_run_kpo_delete_nodepool", {"cmd": cmd})
                kpo_cleanup["nodepools_deleted"].append(
                    {"namespace": np["namespace"], "name": np["name"], "dry_run": True}
                )
            else:
                r = _run(cmd)
                _record("kpo_delete_nodepool", {"target": np, "result": r})
                if r["rc"] != 0:
                    errors.append(
                        f"Failed deleting NodePool {np['namespace']}/{np['name']}: {r['stderr'].strip()}"
                    )
                else:
                    kpo_cleanup["nodepools_deleted"].append(
                        {"namespace": np["namespace"], "name": np["name"]}
                    )

        # Wait for NodeClaims to clear
        def _nodeclaims_empty():
            r, j = _kubectl_json(kpo_kubeconfig, ["get", "nodeclaims", "-A"])
            if r["rc"] != 0:
                return False, {"error": r["stderr"].strip(), "rc": r["rc"]}
            items = (j or {}).get("items", []) or []
            count = len(items)
            return count == 0, {"count": count}

        if dry_run:
            kpo_cleanup["nodeclaims_wait"] = {"skipped": True, "reason": "dry_run"}
        else:
            ok, history = _wait_until(
                _nodeclaims_empty, cleanup_timeout_nodeclaims, poll_seconds=poll_seconds
            )
            kpo_cleanup["nodeclaims_wait"] = {
                "timeout_seconds": cleanup_timeout_nodeclaims,
                "poll_seconds": poll_seconds,
                "ok": ok,
                "history_tail": history[-20:],
            }
            if not ok:
                errors.append("Timed out waiting for KPO NodeClaims to clear.")

        # Wait for KPO nodes to terminate (best effort)
        nodepool_names = [
            x["name"] for x in kpo_cleanup["nodepool_candidates"] if x.get("name")
        ]

        def _nodes_gone():
            counts = {}
            total = 0
            for name in nodepool_names:
                cmd = [
                    "kubectl",
                    "--kubeconfig",
                    str(kpo_kubeconfig),
                    "get",
                    "nodes",
                    "-l",
                    f"karpenter.sh/nodepool={name}",
                    "-o",
                    "json",
                ]
                r, j = _run_json(cmd)
                if r["rc"] != 0:
                    return False, {"error": r["stderr"].strip(), "rc": r["rc"]}
                c = len((j or {}).get("items", []) or [])
                counts[name] = c
                total += c
            return total == 0, {"total": total, "by_nodepool": counts}

        if dry_run:
            kpo_cleanup["nodes_wait"] = {"skipped": True, "reason": "dry_run"}
        elif nodepool_names:
            ok, history = _wait_until(
                _nodes_gone, cleanup_timeout_nodes, poll_seconds=poll_seconds
            )
            kpo_cleanup["nodes_wait"] = {
                "timeout_seconds": cleanup_timeout_nodes,
                "poll_seconds": poll_seconds,
                "ok": ok,
                "history_tail": history[-20:],
            }
            if not ok:
                errors.append("Timed out waiting for KPO nodes to terminate.")
        else:
            kpo_cleanup["nodes_wait"] = {
                "skipped": True,
                "reason": "no_nodepool_candidates",
            }

        # Delete OCINodeClass resources after NodePool cleanup
        for nc in kpo_cleanup["ocinodeclass_candidates"]:
            cmd = [
                "kubectl",
                "--kubeconfig",
                str(kpo_kubeconfig),
                "delete",
                "ocinodeclass",
                nc,
                "--ignore-not-found=true",
            ]
            if dry_run:
                _record("dry_run_kpo_delete_ocinodeclass", {"cmd": cmd})
                kpo_cleanup["ocinodeclasses_deleted"].append(
                    {"name": nc, "dry_run": True}
                )
            else:
                r = _run(cmd)
                _record("kpo_delete_ocinodeclass", {"target": nc, "result": r})
                if r["rc"] != 0:
                    errors.append(
                        f"Failed deleting OCINodeClass {nc}: {r['stderr'].strip()}"
                    )
                else:
                    kpo_cleanup["ocinodeclasses_deleted"].append({"name": nc})


# -------------------------------
# Terraform destroy selected targets
# -------------------------------
terraform_cleanup = {
    "executed": not skip_terraform_destroy,
    "targets": {},
}

for t in selected_targets:
    d = Path(stack_dirs.get(t, "")).resolve()
    entry = {
        "stack_dir": str(d),
        "exists": d.exists(),
        "init": None,
        "destroy": None,
        "state_after": None,
    }
    terraform_cleanup["targets"][t] = entry

    if skip_terraform_destroy:
        entry["skipped"] = True
        entry["reason"] = "CLEANUP_SKIP_TERRAFORM_DESTROY=true"
        continue

    if not d.exists():
        errors.append(f"Terraform stack dir not found for {t}: {d}")
        continue

    init_cmd = [
        "terraform",
        f"-chdir={d}",
        "init",
        "-input=false",
        "-no-color",
    ]
    destroy_cmd = [
        "terraform",
        f"-chdir={d}",
        "destroy",
        "-auto-approve",
        "-input=false",
        "-no-color",
        "-lock-timeout=20m",
    ]
    state_cmd = ["terraform", f"-chdir={d}", "state", "list"]

    if dry_run:
        entry["init"] = {"dry_run": True, "cmd": init_cmd}
        entry["destroy"] = {"dry_run": True, "cmd": destroy_cmd}
        entry["state_after"] = {"dry_run": True, "cmd": state_cmd}
        _record(
            "dry_run_terraform", {"target": t, "init": init_cmd, "destroy": destroy_cmd}
        )
        continue

    r_init = _run(init_cmd, timeout=1800)
    entry["init"] = {
        "rc": r_init["rc"],
        "stderr_tail": (r_init["stderr"] or "")[-4000:],
    }
    _record("terraform_init", {"target": t, "result": entry["init"]})
    if r_init["rc"] != 0:
        errors.append(
            f"terraform init failed for {t}: {(r_init['stderr'] or '').strip()}"
        )
        continue

    r_destroy = _run(destroy_cmd, timeout=7200)
    entry["destroy"] = {
        "rc": r_destroy["rc"],
        "stderr_tail": (r_destroy["stderr"] or "")[-4000:],
    }
    _record("terraform_destroy", {"target": t, "result": entry["destroy"]})
    if r_destroy["rc"] != 0:
        errors.append(
            f"terraform destroy failed for {t}: {(r_destroy['stderr'] or '').strip()}"
        )

    r_state = _run(state_cmd, timeout=600)
    stdout_lines = [
        x.strip() for x in (r_state["stdout"] or "").splitlines() if x.strip()
    ]
    stderr_text = r_state["stderr"] or ""
    no_state_file = "No state file was found" in stderr_text
    empty = (r_state["rc"] == 0 and len(stdout_lines) == 0) or no_state_file
    entry["state_after"] = {
        "rc": r_state["rc"],
        "empty": empty,
        "resource_count": len(stdout_lines),
        "resources": stdout_lines[:200],
        "stderr_tail": stderr_text[-2000:],
    }


# -------------------------------
# Residual OCI resource checks
# -------------------------------
oci_checks = []


def _oci_cmd(args):
    base = [
        "oci",
        "--config-file",
        oci_config_file,
        "--profile",
        oci_profile,
        "--region",
        region,
    ]
    return base + args + ["--output", "json"]


def _check_oci_resource(resource_type, resource_id, args):
    cmd = _oci_cmd(args)
    if dry_run:
        return {
            "resource_type": resource_type,
            "id": resource_id,
            "exists": None,
            "dry_run": True,
            "cmd": cmd,
        }

    r = _run(cmd, timeout=300)
    if r["rc"] == 0:
        return {
            "resource_type": resource_type,
            "id": resource_id,
            "exists": True,
            "rc": r["rc"],
        }

    combined = f"{r.get('stderr', '')}\n{r.get('stdout', '')}"
    if _is_not_found(combined):
        return {
            "resource_type": resource_type,
            "id": resource_id,
            "exists": False,
            "rc": r["rc"],
        }

    return {
        "resource_type": resource_type,
        "id": resource_id,
        "exists": None,
        "rc": r["rc"],
        "error": combined[-2000:],
    }


# cluster checks
for t in selected_targets:
    cid = cluster_ids.get(t)
    if cid:
        oci_checks.append(
            _check_oci_resource(
                "cluster", cid, ["ce", "cluster", "get", "--cluster-id", cid]
            )
        )

# worker nodepool checks
for t in selected_targets:
    for npid in worker_pool_ids.get(t, []) or []:
        oci_checks.append(
            _check_oci_resource(
                "nodepool", npid, ["ce", "node-pool", "get", "--node-pool-id", npid]
            )
        )

# IAM resource checks from post-provisioning
iam_obj = post_prov.get("iam") or {}
if iam_obj:
    for key, val in iam_obj.items():
        rid = (val or {}).get("id")
        if not rid:
            continue
        if "dynamicgroup" in rid:
            oci_checks.append(
                _check_oci_resource(
                    key, rid, ["iam", "dynamic-group", "get", "--dynamic-group-id", rid]
                )
            )
        elif "policy" in rid:
            oci_checks.append(
                _check_oci_resource(
                    key, rid, ["iam", "policy", "get", "--policy-id", rid]
                )
            )


# -------------------------------
# Residual report
# -------------------------------
residuals = {
    "terraform_state_residuals": {},
    "oci_existing_resources": [],
    "oci_unknown_state_resources": [],
}

for t, info in (terraform_cleanup.get("targets") or {}).items():
    state_after = info.get("state_after") or {}
    if state_after.get("empty") is False:
        residuals["terraform_state_residuals"][t] = state_after

for chk in oci_checks:
    if chk.get("exists") is True:
        residuals["oci_existing_resources"].append(chk)
    elif chk.get("exists") is None and not chk.get("dry_run"):
        residuals["oci_unknown_state_resources"].append(chk)

summary = {
    "generated_at_utc": _now_utc(),
    "run_id": run_id,
    "artifact_dir": str(artifact_dir),
    "selected_targets": selected_targets,
    "region": region,
    "oci_profile": oci_profile,
    "oci_config_file": oci_config_file,
    "dry_run": dry_run,
    "skip_terraform_destroy": skip_terraform_destroy,
    "timeouts": {
        "nodeclaims_seconds": cleanup_timeout_nodeclaims,
        "nodes_seconds": cleanup_timeout_nodes,
        "poll_seconds": poll_seconds,
    },
    "kpo_cleanup": kpo_cleanup,
    "terraform_cleanup": terraform_cleanup,
    "oci_resource_checks": oci_checks,
    "residuals": residuals,
    "errors": errors,
    "status": "success" if not errors else "completed_with_errors",
}

cleanup_summary_path = artifact_dir / "cleanup_summary.json"
actions_path = artifact_dir / "cleanup_actions.jsonl"
cleanup_summary_path.write_text(
    json.dumps(summary, indent=2, sort_keys=True) + "\n", encoding="utf-8"
)
with actions_path.open("w", encoding="utf-8") as f:
    for a in cleanup_actions:
        f.write(json.dumps(a, sort_keys=True) + "\n")

os.environ["CLEANUP_SUMMARY_ARTIFACT"] = str(cleanup_summary_path)
os.environ["CLEANUP_ACTIONS_ARTIFACT"] = str(actions_path)

print("\nCell 15 summary:")
print(
    json.dumps(
        {
            "run_id": run_id,
            "status": summary["status"],
            "errors": len(errors),
            "residual_terraform_targets": sorted(
                list(residuals["terraform_state_residuals"].keys())
            ),
            "residual_oci_resources": len(residuals["oci_existing_resources"]),
            "unknown_oci_states": len(residuals["oci_unknown_state_resources"]),
            "cleanup_summary": str(cleanup_summary_path),
            "cleanup_actions": str(actions_path),
        },
        indent=2,
    )
)
print(f"\nCell 15 complete. Artifact: {cleanup_summary_path}")